# WHOMOLLM Project
## Geo context prompt (Version 6) 19.Mar.2026 based on the Qwen 2.5 model feature: age group and household size, gender, household income level with fine-tuning 
1. Raw data
2. stay segmentation
3. reverse geocoding
4. top 5 POIs aggregation
5. semantic stay table
6. LLM prompt (daily narrative/ activity inference/ other surplus info)

## 1. Test GPU, Qwen model 

In [1]:
import os
import json
import re
import sys
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    p = torch.cuda.get_device_properties(0)
    print("VRAM (GB):", round(p.total_memory / 1024**3, 2))

# MODEL_NAME = "Qwen/Qwen2-7B-Instruct"
# gpt-oss-20b
MODEL_NAME = "gpt-oss/gpt-oss-20b"

print("Loading gpt-oss-20b...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0}, # load on GPU 0, otherwise it will load on CPU and be very slow
    trust_remote_code=True
)

model.eval()

print("gpt-oss-20b ready.")

/home/baliu/.venvs/qwen_ft/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
GPU: NVIDIA L4
VRAM (GB): 22.06
Loading gpt-oss-20b...


OSError: gpt-oss/gpt-oss-20b is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

## 2. Generate prompts (6th version with just geo context information and behaviour info, predict household size and age group ... for all users used different model, gpt-oss)
Version control:
This is the basic prompt based on geocontext and behaviour info with fine-tuned gpt-oss-20b version 6.

In [ ]:
#   ---------------------------
# Version 6 (19 Mar 2026) prompt with geo context to predict features
#   ---------------------------

import os, re, json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import pdb

# ---------------------------
# Config
# ---------------------------
RANDOM_SEED = 42
N_USERS     = 5
MAX_EVENTS  = 300
HOUR_BIN    = 1
PREC        = int(os.getenv("COORD_PREC", "4"))


DATA_SP     = Path("/data/baliu/python_code/data/sp_all copy.csv")
PROMPTS_OUT = Path("/data/baliu/python_code/data/prompts_v6_features_19Mar2026_age_hhsize_finetunedmodel.json")

# -------------------------------------------   
# Load (strings) + base cleaning
# -------------------------------------------
sp = pd.read_csv(DATA_SP, dtype=str, engine="python", on_bad_lines="skip")
print("sp shape:", sp.shape)

# Datetimes
sp["started_at"]  = pd.to_datetime(sp["started_at"], errors="coerce", utc=True)
sp["finished_at"] = pd.to_datetime(sp.get("finished_at"), errors="coerce", utc=True)

# Drop unusable
sp = sp.dropna(subset=["user_id", "started_at", "location_id"]).copy()
sp["user_id"]     = sp["user_id"].astype(str)
sp["location_id"] = sp["location_id"].astype(str)

# sp["duration_min"] = ((sp["finished_at"] - sp["started_at"]).dt.total_seconds() / 60.0).astype(float)
print(sp["duration"].head())
print(sp["duration"].dtype)

# Force numeric
sp["act_duration"] = pd.to_numeric(
    sp["act_duration"],
    errors="coerce"
)

# change it to hours
sp["act_duration_h"] = (sp["act_duration"] / 60.0).round(1)
print(sp["act_duration_h"].head())
print(sp["act_duration_h"].dtype)

# length_km: convert to numeric, coerce errors to NaN, then divide by 1000
sp["length_km"] = pd.to_numeric(sp["length"], errors="coerce") / 1000.0
sp["length_km"] = sp["length_km"].round(1)

# Time features
sp["date"]     = sp["started_at"].dt.date
sp["dow"]      = sp["started_at"].dt.dayofweek.astype(int)        #  0=Mon
sp["hour_bin"] = (sp["started_at"].dt.hour // HOUR_BIN * HOUR_BIN).astype(int)

# ---------------------------
# Format from wkt point -> lon/lat (rounded)
# --------------------------------
WKT_POINT_RE = re.compile(
    r"POINT\s*\(\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s+([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*\)",
    re.I,
)

def extract_lonlat_from_wkt(s: str):
    m = WKT_POINT_RE.search(str(s))
    if not m: return (np.nan, np.nan)
    return float(m.group(1)), float(m.group(2))

sp["geometry"] = sp.get("geometry", pd.Series(index=sp.index)).astype(str)
lonlat = sp["geometry"].apply(extract_lonlat_from_wkt)
sp[["lon","lat"]] = pd.DataFrame(lonlat.tolist(), index=sp.index)
sp["lon"] = pd.to_numeric(sp["lon"], errors="coerce").round(PREC)
sp["lat"] = pd.to_numeric(sp["lat"], errors="coerce").round(PREC)

# Lightweight tags
sp["geometry_type"] = np.where(sp["lon"].notna() & sp["lat"].notna(), "point", "-")
if "mode" not in sp.columns: sp["mode"] = "-"

sp shape: (1152987, 10)
0    1093.0
1      39.0
2      75.0
3     164.0
4      36.0
Name: duration, dtype: str
str
0    18.2
1     1.2
2     1.4
3     5.6
4     1.2
Name: act_duration_h, dtype: float64
float64


In [3]:
print(sp["act_duration_h"].head())
print(sp["act_duration_h"].dtype)
print(sp["length_km"].head())
print(sp["length_km"].dtype)

0    18.2
1     1.2
2     1.4
3     5.6
4     1.2
Name: act_duration_h, dtype: float64
float64
0     0.0
1    11.6
2     2.1
3     4.8
4    12.6
Name: length_km, dtype: float64
float64


In [4]:
print ("sp after cleaning:", sp.shape)
print (sp.head(5))

sp after cleaning: (862871, 18)
  id user_id                       started_at  \
0  0   AAGAF 2019-10-09 11:30:34.141000+00:00   
1  1   AAGAF 2019-10-10 06:14:49.141999+00:00   
2  2   AAGAF 2019-10-10 07:03:24.426000+00:00   
3  3   AAGAF 2019-10-10 11:10:24.605999+00:00   
4  4   AAGAF 2019-10-10 14:30:45.187999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-10 05:43:17.674999+00:00           0   Car                 0.0   
1 2019-10-10 06:53:54.841000+00:00           1   Car  11615.408548376423   
2 2019-10-10 08:18:20.864000+00:00           2   Car  2104.8558581266784   
3 2019-10-10 13:54:34.799339+00:00           2  Walk   4847.706521391238   
4 2019-10-10 15:07:07.127239+00:00           3   Car  12621.909934521313   

                                       geometry duration  act_duration  \
0  POINT (7.565219245342489 47.545616372908086)   1093.0        1093.0   
1  POINT (7.5637597959179645 47.54794767256367)     39.0          71

In [ ]:
# ---------------------------
# Sample 1 week per user
# ---------------------------
def sample_one_week_per_user(sp, seed=42):
    rng = np.random.default_rng(seed)
    sp = sp.copy()
    sp["date"] = pd.to_datetime(sp["date"])

    out = []
    for uid, df_u in sp.groupby("user_id"):
        days = (
            df_u[["date"]]
            .drop_duplicates()
            .sort_values("date")
            .reset_index(drop=True)
        )

        if len(days) < 7:
            continue
        days["delta"] = days["date"].diff().dt.days.fillna(1).astype(int)
        days["block"] = (days["delta"] >1).cumsum()

        valid_blocks = (
            days.groupby("block")
            .filter(lambda x: len(x) >=7)
        )

        if valid_blocks.empty:
            continue
        candidate_starts = []
        for _, g in valid_blocks.groupby("block"):
            block_dates = g["date"].sort_values().reset_index(drop=True)
            for i in range(len(block_dates) - 6):
                candidate_starts.append(g.iloc[i]["date"])

        start_date = rng.choice(candidate_starts)
        week_dates = pd.date_range(start=start_date, periods=7, freq= 'D')

        out.append(
            df_u[df_u["date"].isin(week_dates)]
        )
       
    return pd.concat(out, ignore_index=True)

sp_sampled = sample_one_week_per_user(sp, seed=RANDOM_SEED)
print("sp_sampled shape:", sp_sampled.shape)
print(sp_sampled.head(2))

# ---------------------------
# save sampled data
# ---------------------------
SP_SAMPLED_OUT = Path("/data/baliu/python_code/data/sp_sampled_1week_19Mar2026_fine-tuned_qwen.csv")
sp_sampled.to_csv(SP_SAMPLED_OUT, index=False)
print("Saved sp_sampled_1week_19Mar2026 to", SP_SAMPLED_OUT)  

sp_sampled shape: (60738, 18)
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   

                       finished_at location_id  mode             length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus  3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram  8232.492568529495   

                                       geometry duration  act_duration  \
0  POINT (7.564359990583688 47.546459921667946)   1097.0        6456.0   
1   POINT (7.603269084785703 47.58100341208616)    116.0         275.0   

   act_duration_h  length_km       date  dow  hour_bin     lon      lat  \
0           107.6        3.8 2019-10-22    1         9  7.5644  47.5465   
1             4.6        8.2 2019-10-23    2         6  7.6033  47.5810   

  geometry_type  
0         point  
1         point  
Saved sp_sampled_1week_11Mar2026 to /data/baliu/python_code/data/sp_sampled_1week_11Mar202

#### Load location name and POIs top 3

In [ ]:
# 1. Load toponym data from shapefiles
import os, re, json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import pdb
import geopandas as gpd
import fiona
# read parquet file
import pyarrow.parquet as pq
 
CACHE_DIR   = Path("/data/baliu/python_code/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NOM_CACHE_PATH = CACHE_DIR / "nominatim_cache.parquet"
nom = pd.read_parquet(NOM_CACHE_PATH)
print("Nominatim cache loaded:", nom.shape)
print("Nominatim columns:", nom.columns.tolist())
nom.head(30)
groupby_category = nom.groupby('city').size()
print(groupby_category) 

Nominatim cache loaded: (370983, 6)
Nominatim columns: ['lon', 'lat', 'road', 'neighbourhood', 'city', 'postcode']
city
A Coruña                 2
A Illa de Arousa         2
A Pobra do Caramiñal     7
A Seara                  1
Aachen                   2
                        ..
دهستان سروستان           1
دهستان فجر               3
دهستان کرکس              1
บ้านอ่าวน้ำเมา           1
เทศบาลนครเกาะสมุย       55
Length: 7400, dtype: int64


In [ ]:
from notebook_utils.geocontext import attach_nominatim, clean_nominatim_fields

sp_sampled2 = attach_nominatim(sp_sampled2, nom)
sp_sampled2 = clean_nominatim_fields(sp_sampled2)
print("sp_sampled2 after attaching nominatim:", sp_sampled2.shape)
print("sp_sampled2 columns:", sp_sampled2.columns.tolist())
print(sp_sampled2.head(10))


sp_sampled after attaching nominatim: (60738, 22)
sp_sampled columns: ['id', 'user_id', 'started_at', 'finished_at', 'location_id', 'mode', 'length', 'geometry', 'duration', 'act_duration', 'act_duration_h', 'length_km', 'date', 'dow', 'hour_bin', 'lon', 'lat', 'geometry_type', 'road', 'neighbourhood', 'city', 'postcode']
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus   3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram   8232.492568529495   
2 2019-10-23 09:38:13.596999+00:00          17  Walk   2892.684874565825   
3 2019-10-23 16:19:19.816718+00:00          10   Car  3283.143616000176

#### Build token with geocontext info

In [8]:
from pathlib import Path
from notebook_utils.geocontext import load_poi_frame

POI_PATH = Path("/data/baliu/python_code/data/version2/data/final_pois_nob.gpkg")
poi = load_poi_frame(POI_PATH)
pois = poi
poi.head()


,id,code,name,category,geometry
0,0,3100,Temple de Saint-Jean,Civic,POINT (2498816.948 1117839.539)
1,1,3100,Kapelle Oberes Heiliges Kreuz,Civic,POINT (2691857.357 1192677.631)
2,2,3102,Kirche St. Johannes,Civic,POINT (2599659.594 1202970.041)
3,3,3102,Paroisse catholique,Civic,POINT (2501879.911 1126418.086)
4,4,3300,Mosquée du Petit-Saconnex,Civic,POINT (2498411.634 1119984.893)


In [9]:
poi = poi.copy()
print(poi["category"].value_counts())
print(f"poi crs: {poi.crs}")


category
Others            158579
Unknown           114894
Entertainment     106734
Shopping           54634
Residential        45556
Transportation     40997
Services            9525
Schools             2904
Civic                780
Name: count, dtype: int64
poi crs: EPSG:2056


In [10]:
from notebook_utils.geocontext import load_poi_frame

def load_pois_once():
    global poi
    if poi is None:
        print("Loading POIs into memory...")
        poi = load_poi_frame(POI_PATH)
    return poi

print(poi.head(2))


   id  code                           name category  \
0   0  3100           Temple de Saint-Jean    Civic   
1   1  3100  Kapelle Oberes Heiliges Kreuz    Civic   

                          geometry  
0  POINT (2498816.948 1117839.539)  
1  POINT (2691857.357 1192677.631)  


In [11]:
from notebook_utils.geocontext import build_location_gdf

loc = build_location_gdf(sp_sampled2)


##### update.  
Top 2 poi buffer  
clean text part () function    
skip poi if both name and category unknown. 
";".join(out) string return.  
debug prints after merge

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
from pathlib import Path

# =========================================================
# 1. TEXT CLEANING
# =========================================================

def clean_text(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.lower() in ["none", "nan", "-", ""]:
        return None
    return s


# =========================================================
# 2. Time feature engineering
# =========================================================

dow_names = {
    1: "Monday",
    2: "Tuesday",
    3: "Wednesday",
    4: "Thursday",
    5: "Friday",
    6: "Saturday",
    0: "Sunday"
}

hour_bin_labels = {
    0: "midnight",
    1: "working hours in the early morning",
    2: "morning",
    3: "afternoon",
    4: "evening",
    5: "night"
}


def hour_to_bin(hour):

    if 0 <= hour <= 3:
        return 0
    elif 4 <= hour <= 10:
        return 1
    elif 11 <= hour <= 13:
        return 2
    elif 14 <= hour <= 18:
        return 3
    elif 19 <= hour <= 20:
        return 4
    else:
        return 5 # 21-23


def add_time_features(df):

    df["dow_names"] = df["dow"].map(dow_names)

    df["hour"] = pd.to_datetime(df["started_at"]).dt.hour

    df["hour_bin"] = df["hour"].apply(hour_to_bin)

    df["hour_block"] = df["hour_bin"].map(hour_bin_labels)

    return df

# =========================================================
# 3. Direction computation
# =========================================================

def bearing_to_direction(dx, dy):

    angle = np.degrees(np.arctan2(dy, dx))
    angle = (angle + 360) % 360

    if 22.5 <= angle < 67.5:
        return "North-East"
    elif 67.5 <= angle < 112.5:
        return "East"
    elif 112.5 <= angle < 157.5:
        return "South-East"
    elif 157.5 <= angle < 202.5:
        return "South"
    elif 202.5 <= angle < 247.5:
        return "South-West"
    elif 247.5 <= angle < 292.5:
        return "West"
    elif 292.5 <= angle < 337.5:
        return "North-West"
    else:
        return "North"

# =========================================================
# 4. Build poi context
# =========================================================

def get_poi_context_for_prompt(loc_df, poi_df, top_k=3):

    # --- convert to GeoDataFrame
    loc = gpd.GeoDataFrame(
        loc_df.copy(),
        geometry=gpd.points_from_xy(loc_df["lon"], loc_df["lat"]),
        crs="EPSG:4326"
    ).to_crs(2056)

    poi = poi_df.copy().set_crs(epsg=2056, allow_override=True)

    # --- build KDTree
    poi_xy = np.c_[poi.geometry.x.values, poi.geometry.y.values]
    tree = cKDTree(poi_xy)

    loc_xy = np.c_[loc.geometry.x.values, loc.geometry.y.values]

    dists, idxs = tree.query(loc_xy, k=top_k * 3)

    rows = []

    for i, (ds, js) in enumerate(zip(dists, idxs)):
        for d, j in zip(ds, js):
            rows.append({
                "location_id": loc.iloc[i]["location_id"],
                "name": poi.iloc[j]["name"],
                "category": poi.iloc[j]["category"],
                "dist_m": d,
                "dx": poi.geometry.iloc[j].x - loc.geometry.iloc[i].x,
                "dy": poi.geometry.iloc[j].y - loc.geometry.iloc[i].y
            })
    joined = pd.DataFrame(rows)

    # remove bad categories
    joined = joined[
        joined["category"].notna() &
        (~joined["category"].str.lower().isin(["unknown", "others"]))
    ]

    joined["dist_km"] = (joined["dist_m"] / 1000).round(1)

    joined["direction"] = [
        bearing_to_direction(dx, dy)
        for dx, dy in zip(joined.dx, joined.dy)
    ]

    joined = (
        joined
        .sort_values("dist_m")
        .groupby("location_id", group_keys=False)
        .head(top_k)
    )
    return joined

# =========================================================
# 5. Build prompt text from POI context
# =========================================================

def build_poi_prompt_text(df):

    out = []

    for _, r in df.iterrows():

        name = clean_text(r.get("name"))
        category = clean_text(r.get("category"))

        if name is None and category is None:
            continue

        label = " ".join(filter(None, [category, name]))

        out.append(
            f"{r['dist_km']} km to the {r['direction'].lower()}: {label}"
        )

    return "; ".join(out)

# =========================================================
# 6. pipeline
# =========================================================
def build_geospatial_context(sp_sampled, poi):

    sp_sampled = add_time_features(sp_sampled)

    # ── STEP 1: check poi_context is non-empty ──────────────────────────────
    poi_context = get_poi_context_for_prompt(sp_sampled, poi)
    print(f"[DEBUG] poi_context rows: {len(poi_context)}")
    print(f"[DEBUG] poi_context cols: {poi_context.columns.tolist()}")
    print(poi_context.head(3))

    if poi_context.empty:
        print("[ERROR] poi_context is empty — check POI CRS, coordinates, or category filters")
        sp_sampled["nearby_places"] = None
        return sp_sampled

    # ── STEP 2: check location_id types match ───────────────────────────────
    print(f"[DEBUG] sp_sampled location_id dtype : {sp_sampled['location_id'].dtype}")
    print(f"[DEBUG] poi_context location_id dtype: {poi_context['location_id'].dtype}")
    print(f"[DEBUG] sp_sampled location_id sample : {sp_sampled['location_id'].unique()[:5]}")
    print(f"[DEBUG] poi_context location_id sample: {poi_context['location_id'].unique()[:5]}")

    # ── STEP 3: force same dtype before groupby ─────────────────────────────
    poi_context["location_id"] = poi_context["location_id"].astype(str)
    sp_sampled = sp_sampled.copy()
    sp_sampled["location_id"] = sp_sampled["location_id"].astype(str)

    # ── STEP 4: build poi_prompt_df robustly ────────────────────────────────
    nearby_maps = {
        loc_id : build_poi_prompt_text(grp)
        for loc_id, grp in poi_context.groupby("location_id", group_keys=False)
           }

    poi_prompt_df = pd.DataFrame(
        list(nearby_maps.items()),
        columns=["location_id", "nearby_places"]
    )
    print(f"[DEBUG] poi_prompt_df rows: {len(poi_prompt_df)}")
    print(poi_prompt_df.head(3))

    if poi_prompt_df.empty:
        print("[ERROR] poi_prompt_df is empty — build_poi_prompt_text returned nothing")
        sp_sampled["nearby_places"] = None
        return sp_sampled

    # ── step 5: check merge overlap ─────────────────────────────────────────
    overlap = set(sp_sampled["location_id"]) & set(poi_prompt_df["location_id"])
    print(f"[DEBUG] location_id overlap count: {len(overlap)} / {sp_sampled['location_id'].nunique()} unique in sp_sampled")
    sp_sampled = sp_sampled.drop(columns=["nearby_places"], errors="ignore")
    sp_sampled = sp_sampled.merge(poi_prompt_df, on="location_id", how="left")

    print(f"[DEBUG] nearby_places filled: {sp_sampled['nearby_places'].notna().sum()} / {len(sp_sampled)}")
    return sp_sampled


# =========================================================
# 7. run 
# =========================================================

sp_sampled = build_geospatial_context(sp_sampled, poi)

# diagnostics
print(sp_sampled["nearby_places"].notna().value_counts())
print(sp_sampled[["location_id", "nearby_places"]].head())

# save
OUT = Path("/data/baliu/python_code/data/sp_sampled_with_geocontext.csv")

sp_sampled.to_csv(OUT, index=False)

print("Saved to:", OUT)

[DEBUG] poi_context rows: 65502
[DEBUG] poi_context cols: ['location_id', 'name', 'category', 'dist_m', 'dx', 'dy', 'dist_km', 'direction']
       location_id             name        category    dist_m        dx  \
122355         979     Bern Bahnhof        Shopping  0.000000  0.000000   
99027        30380  Sion Poste Gare  Transportation  0.112231 -0.015315   
188586       59718     Josefsgütsch   Entertainment  0.183169  0.078158   

              dy  dist_km   direction  
122355  0.000000      0.0       North  
99027   0.111181      0.0        East  
188586 -0.165657      0.0  North-West  
[DEBUG] sp_sampled location_id dtype : str
[DEBUG] poi_context location_id dtype: str
[DEBUG] sp_sampled location_id sample : <ArrowStringArray>
['10', '16', '17', '4', '18']
Length: 5, dtype: str
[DEBUG] poi_context location_id sample: <ArrowStringArray>
['979', '30380', '59718', '56608', '37556']
Length: 5, dtype: str
[DEBUG] poi_prompt_df rows: 22983
  location_id                              

In [17]:
from notebook_utils.geocontext import print_poi_feature_summary

print_poi_feature_summary(sp_sampled2)



Final sp_sampled with POI features:
Columns: ['id', 'user_id', 'started_at', 'finished_at', 'location_id', 'mode', 'length', 'geometry', 'duration', 'act_duration', 'act_duration_h', 'length_km', 'date', 'dow', 'hour_bin', 'lon', 'lat', 'geometry_type', 'road', 'neighbourhood', 'city', 'postcode', 'dow_names', 'hour', 'hour_block', 'nearby_places']

Sample rows:
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus   3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram   8232.492568529495   
2 2019-10-23 09:38:13.596999+00:00          17  Walk   2892.684874565825   
3 2019-10-23 16:19:19.816718+

In [18]:
import pandas as pd

def clean_addr_part(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.lower() in ["nan", "none", "-", ""]:
        return None
    return s

DOW_NAMES = {
    0: "Sunday", 1: "Monday", 2: "Tuesday", 3: "Wednesday",
    4: "Thursday", 5: "Friday", 6: "Saturday"
}

HOUR_BIN_LABELS = {
    0: "midnight", 1: "early morning", 2: "morning",
    3: "afternoon", 4: "evening", 5: "night"
}

def tokens_compact_1week(
    df_u: pd.DataFrame,
    max_events: int = 50,
    prec: int = 4
) -> list[str]:
    """
    Build natural-language mobility tokens from up to one week of staypoints,
    inserting a date header at the beginning of each day.
    """
    # --------------------------------------------------
    # 1. detect duration column
    # --------------------------------------------------
    duration_col = None
    for c in df_u.columns:
        if c.lower() in ["duration", "duration_min", "act_duration", "act_duration_min", "dur_min"]:
            duration_col = c
            break

    # --------------------------------------------------
    # 2. select usable columns
    # --------------------------------------------------
    use_cols = [
        "started_at", "dow", "hour_bin", "location_id",
        "lon", "lat", "mode",
        "city", "neighbourhood", "road", "nearby_places",
        "postcode"
    ]
    if duration_col:
        use_cols.append(duration_col)

    df = df_u.loc[:, [c for c in use_cols if c in df_u.columns]].copy()

    # --------------------------------------------------
    # 3. datetime handling + ordering + safety cap
    # --------------------------------------------------
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df = df.dropna(subset=["started_at"]).sort_values("started_at").reset_index(drop=True)

    if len(df) > max_events:
        df = df.head(max_events)

    # --------------------------------------------------
    # 4. build tokens with date headers
    # --------------------------------------------------
    toks = []
    current_date = None

    for r in df.itertuples(index=False):   # one variable, one namedtuple per row
        t = r.started_at
        date_str = t.date().isoformat()

        # ---- date header when day changes ----
        if date_str != current_date:
            toks.append(f"--- Date: {date_str} ---")
            current_date = date_str

        # ---- time ----
        hhmm = t.strftime("%H:%M")

        dow_raw = getattr(r, "dow", None)
        hb_raw  = getattr(r, "hour_bin", None)
        dow_name = DOW_NAMES.get(int(dow_raw), "unknown day") if pd.notna(dow_raw) else "unknown day"
        hb_label = HOUR_BIN_LABELS.get(int(hb_raw), "") if pd.notna(hb_raw) else ""

        # ---- coordinates ----
        lat = round(float(getattr(r, "lat", 0.0)), prec)
        lon = round(float(getattr(r, "lon", 0.0)), prec)

        # ---- address ----
        addr_parts = [
            clean_addr_part(getattr(r, "road",          None)),
            clean_addr_part(getattr(r, "neighbourhood", None)),
            clean_addr_part(getattr(r, "city",          None)),
        ]
        addr_parts = [p for p in addr_parts if p is not None]
        addr = ", ".join(addr_parts) if addr_parts else "unknown address"

        postcode = clean_addr_part(getattr(r, "postcode", None))
        if postcode is not None:
            addr = f"{addr} (postcode {postcode})"

        # ---- nearby POIs ----
        nearby_raw = getattr(r, "nearby_places", None)
        if isinstance(nearby_raw, list):
            nearby_str = "; ".join(nearby_raw[:5])
        elif isinstance(nearby_raw, str) and nearby_raw.strip():
            nearby_str = nearby_raw
        else:
            nearby_str = "no nearby places listed"

        # ---- mode & duration ----
        mode = str(getattr(r, "mode", "unknown")).strip().lower()
        if mode in ["nan", "none", ""]:
            mode = "unknown"

        dur_val = getattr(r, duration_col, None) if duration_col else None
        dur_str = f"{int(round(float(dur_val)))} min" if dur_val is not None and pd.notna(dur_val) else "unknown duration"

        # ---- final sentence ----
        time_ctx = f"At {hhmm} ({dow_name}, {hb_label})" if hb_label else f"At {hhmm} ({dow_name})"
        toks.append(
            f"On {time_ctx}, the user spent {dur_str} at {addr} ({lat}, {lon}). "
            f"arriving by {mode}. Nearby places include {nearby_str}."
        )

    return toks

In [19]:
# usage with poi details
toks = tokens_compact_1week(
    df_u=sp_sampled,
    max_events=40,
    prec=4
)

print(toks[:8])

['--- Date: 2019-09-02 ---', 'On At 20:21 (Sunday, evening), the user spent 503 min at Chutzenstrasse, Bern (postcode 3007) (46.936, 7.4314). arriving by walk. Nearby places include 0.1 km to the south-east: Residential Bahnhof Weissenbühl; 0.1 km to the north-east: Shopping Nido; 0.1 km to the south: Shopping.', '--- Date: 2019-09-03 ---', 'On At 05:24 (Monday, early morning), the user spent 615 min at Vogelherdstrasse, Roamer (postcode 4513) (47.2148, 7.5201). arriving by car. Nearby places include 0.1 km to the south-west: Residential Restaurant Industrie; 0.1 km to the south-west: Residential Restaurant Industrie; 0.1 km to the south-west: Residential Restaurant Industrie.', 'On At 16:34 (Monday, afternoon), the user spent 742 min at Chutzenstrasse, Bern (postcode 3007) (46.9359, 7.4311). arriving by car. Nearby places include 0.1 km to the south-east: Residential Bahnhof Weissenbühl; 0.1 km to the north-east: Shopping Nido; 0.1 km to the south: Shopping.', 'On At 18:15 (Monday, af

### Update prompts Version 6 11 Mar 2026

In [20]:
# ---------------------------------------------------------------------------
# Build prompts for age and household size prediction.  update on 11 Mar 2026 
# ----------------------------------

from textwrap import dedent
USE_CACHED = True

if USE_CACHED:
    sp_sampled = pd.read_csv("/data/baliu/python_code/data/sp_sampled_with_geocontext.csv")
else:
    sp_sampled = build_geospatial_context(sp_sampled, poi)
    sp_sampled.to_csv("/data/baliu/python_code/data/sp_sampled_with_geocontext.csv", index=False)
    
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
sp_prompt = sp_sampled.copy()
sp_prompt["date_str"] = (
    pd.to_datetime(sp_prompt["started_at"], errors="coerce")
      .dt.tz_localize(None)
      .dt.date
      .astype(str)
)
print(sp_prompt[["started_at", "date", "date_str"]].head(5))

# build valid user-day table from existing rows only
user_day = (
    sp_prompt
    .dropna(subset=["started_at"])
    .loc[:, ["user_id", "date_str"]]
    .drop_duplicates()
)
user_day = (
    user_day
    .groupby("user_id", group_keys=False)
    #.sample(n=2, random_state=RANDOM_SEED)
    #.rename(columns={"date_str": "sample_date"})
)

#print("user_day size:", len(user_day))   # must be > 0

rows_prompts = []

user_dates = (
    sp_prompt
    .dropna(subset=["started_at"])
    .loc[:, ["user_id", "date_str"]]
    .drop_duplicates()
    .groupby("user_id")["date_str"]
    .apply(list)
)

print(f"user_dates sample: {len(user_dates)}")

for user_id, dates in user_dates.items():
    
    df_u = sp_prompt[

        (sp_prompt["user_id"] == user_id) &
        (sp_prompt["date_str"].isin(dates))
    ]
    print(f"Processing user {user_id} for dates {dates}, df_u shape: {df_u.shape}")


    toks = tokens_compact_1week(df_u, max_events=MAX_EVENTS, prec=PREC)
    if not toks:
        continue
    prompt = (
        f"User: {user_id}\n\n"
        f"Mobility record for one week in Switzerland:\n"
        "Task:\n"
        "Based only on the mobility behaviour and nearby spatial context described below,"
        "Infer the user's most likely household size, household income level per month, gender, and age group based on swiss census data and situation\n\n"
        "household size: 1, 2,3, 4, 5+, household income: 0-4000, 4001-8000, 8001-12000, 12001-16000, 16001+., gender: male, female, other, perefer not to say\n\n"
        " Guidelines:\n"
        "- Use only the information explicitly provided in the mobility record.\n"
        "- Base your reasoning on spatual features, time patterns,mobility intensity, locations visited, transport modes, and activity patterns.\n"
        "- Do not assume any unobserved personal attributes.\n"
        "- keep a neutral and factual tone.\n\n"
        "Output format:\n"
        "1. Reasoning (no more than 150 words).\n"
        "2. One final line in strict JSON format, for example:\n"
        " {\"household_size\": \"4\", \"household_income\": \"8001-12000\", \"gender\": \"other\", \"age_group\": \"45-54\"}\n\n"
        "Mobility evidance:\n"
        + "\n".join(toks)
    )
    
    # store results as a dictionary
    rows_prompts.append({
        "user_id": user_id,
        "date": dates,
        "prompt": prompt})

PROMPTS_OUT = Path("/data/baliu/python_code/data/prompts_v6_1week_age_householdsize_11Mar2026_qwen_fine-tuned_v6.json")

PROMPTS_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(PROMPTS_OUT, "w", encoding="utf-8") as f:
    for r in rows_prompts:
        f.write(r["prompt"])
        f.write("\n\n" + "="*80 + "\n\n")

print(f"Saved {len(rows_prompts)} prompts -> {PROMPTS_OUT}")

# print sample
for r in rows_prompts[:1]:
    print("\n--- Sample Prompt ---")
    print(r["prompt"][:1000] + "\n...")   

                         started_at        date    date_str
0  2019-10-22 09:39:28.773999+00:00  2019-10-22  2019-10-22
1  2019-10-23 06:35:32.236999+00:00  2019-10-23  2019-10-23
2  2019-10-23 09:02:58.773999+00:00  2019-10-23  2019-10-23
3  2019-10-23 09:53:18.151999+00:00  2019-10-23  2019-10-23
4  2019-10-23 16:44:43.608999+00:00  2019-10-23  2019-10-23
user_dates sample: 2102
Processing user AAGAF for dates ['2019-10-22', '2019-10-23', '2019-10-24', '2019-10-25', '2019-10-26', '2019-10-27', '2019-10-28'], df_u shape: (23, 27)
Processing user AAINS for dates ['2019-11-13', '2019-11-14', '2019-11-15', '2019-11-16', '2019-11-17', '2019-11-18', '2019-11-19'], df_u shape: (34, 27)
Processing user AAQME for dates ['2019-12-10', '2019-12-11', '2019-12-12', '2019-12-13', '2019-12-14', '2019-12-15', '2019-12-16'], df_u shape: (23, 27)
Processing user AARWP for dates ['2019-09-18', '2019-09-19', '2019-09-20', '2019-09-21', '2019-09-22', '2019-09-23', '2019-09-24'], df_u shape: (35, 27)
Proc

Clean the prompts 

In [21]:
import json
import re
from pathlib import Path
#prompts_v6_1week_age_householdsize_12Mar2026_qwen_fine-tuned_v6.json
IN_PATH  = Path("/data/baliu/python_code/data/prompts_v6_1week_age_householdsize_11Mar2026_qwen_fine-tuned_v6.json")
OUT_PATHh = Path("/data/baliu/python_code/data/prompts_v6_1week_age_householdsize_12Mar2026_qwen_finetuned_clean.txt")

def clean_prompt_text(text: str) -> str:
    """
    Remove markdown headers, separators, examples, and keep
    only instruction + mobility evidence.
    """
    # 1. remove separator lines ---
    text = re.sub(r"^\s*---\s*$", "", text, flags=re.MULTILINE)

    # 2. remove markdown headers like ## Task Overview
    text = re.sub(r"^\s*## .*?$", "", text, flags=re.MULTILINE)

    # 3. remove Example block
    text = re.sub(
        r'## Example:.*?---',
        '',
        text,
        flags=re.DOTALL
    )

    # 4. collapse multiple blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

cleaned_rows = []

with open(IN_PATH, "r", encoding="utf-8") as f:
    raw = f.read()

blocks = [b.strip() for b in re.split(r"\n{2,}={80,}\n{2,}", raw) if b.strip()]

# apply cleaning
cleaned_blocks = [clean_prompt_text(b) for b in blocks if clean_prompt_text(b)]

# write cleaned jsonl
with open(OUT_PATHh, "w", encoding="utf-8") as f:
    for b in cleaned_blocks:
        f.write(b)
        f.write("\n\n"+ "="*80 + "\n\n")

print(f"Cleaned prompts saved: {len(cleaned_blocks)} -> {OUT_PATHh}")

Cleaned prompts saved: 2102 -> /data/baliu/python_code/data/prompts_v6_1week_age_householdsize_12Mar2026_qwen_finetuned_clean.txt


##### check prompt length

In [1]:
from transformers import AutoTokenizer

# ── token length analysis ────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

char_lengths  = [len(b) for b in cleaned_blocks]
token_lengths = [len(tokenizer.encode(b)) for b in cleaned_blocks]

import pandas as pd
import matplotlib.pyplot as plt

stats = pd.Series(token_lengths)
print("=== Token Length Stats ===")
print(stats.describe(percentiles=[.25, .5, .75, .9, .95, .99]).round(1))
print(f"\nOver 1024 tokens : {(stats > 1024).sum()} / {len(stats)} ({(stats > 1024).mean()*100:.1f}%)")
print(f"Over 2048 tokens : {(stats > 2048).sum()} / {len(stats)} ({(stats > 2048).mean()*100:.1f}%)")

# ── histogram ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(token_lengths, bins=50, color="steelblue", edgecolor="white")
ax.axvline(1024, color="red",    linestyle="--", label="1024 tokens")
ax.axvline(2048, color="orange", linestyle="--", label="2048 tokens")
ax.set_xlabel("Token count per prompt")
ax.set_ylabel("Number of prompts")
ax.set_title("Prompt token length distribution")
ax.legend()
plt.tight_layout()
plt.savefig("/data/baliu/python_code/data/prompt_length_dist.png", dpi=150)
plt.show()

/home/baliu/.venvs/qwen_ft/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'cleaned_blocks' is not defined

## 3. Predict Sociodemographics using "Qwen" model based on the prompts we just generated 

#### Test GPU, decide Model version/ Model name without finetuing verson

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# Proper 4-bit config
bnb_config = BitsAndBytesConfig( # the parameter are actually in bitandbytescofig, not in the model loading.
    # all the parameters are for 4 bit .
    # we can try later when doing the fine tuning.
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained( # the parameters are not belong to here, that's why we need to remove them, 
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("✅ Qwen ready.")

/home/baliu/.venvs/qwen_ft/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 339/339 [05:55<00:00,  1.05s/it] 


✅ Qwen ready.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================================
# FILE PATHS
# ============================================================================
gt_path = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
prompts_path = Path("/data/baliu/python_code/data/prompts_v6_1week_age_householdsize_12Mar2026_qwen_finetuned_clean.txt")
# ============================================================================
# Load datasets
# ============================================================================
gt = pd.read_csv(gt_path)
prompts = pd.read_csv(prompts_path, header=None, names=["prompt_text"], sep="\n\n={80,}\n\n", engine="python")

print(f"Loaded ground truth ({len(gt)} rows)")
print(f"Loaded predictions  ({len(prompts)} rows)")

print("\n" + "="*70)
print("GROUND TRUTH COLUMNS")
print("="*70)
print(gt.columns.tolist())
print("\nFirst row:")
print(gt.head(1))

print("\n" + "="*70)
print("PREDICTION COLUMNS")
print("="*70)
print(prompts.columns.tolist())
print("\nFirst row:")
print(prompts.head(1))

# ============================================================================
# SELECT & RENAME COLUMNS
# ============================================================================
# Ground truth columns
gt = gt[["user_id", "age_group", "household_size_group", "income_level", "gender"]].copy()

In [24]:
import pandas as pd
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

# ============================================================
# 1. Load both files
# ============================================================
gt_path   = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
prompts_path = Path("/data/baliu/python_code/data/prompts_v3_1week_age_householdsize_04Mar2026_qwen_clean_v4.txt")

gt   = pd.read_csv(gt_path, low_memory=False)
prompts = pd.read_csv(prompts_path, header=None, names=["prompt_text"], sep="\n\n={80,}\n\n", engine="python")

print("GT shape  :", gt.shape)
print("Prompts shape:", prompts.shape)

# ============================================================
# 2. Extract user_id from prompt_text
#    Format observed: first line is "User: AAGAF"
# ============================================================
def extract_user_id(text: str) -> str | None:
    if pd.isna(text):
        return None
    first_line = str(text).strip().splitlines()[0]  # "User: AAGAF"
    if ":" in first_line:
        return first_line.split(":", 1)[1].strip()
    return first_line.strip()

prompts["user_id"] = prompts["prompt_text"].apply(extract_user_id)

print("\nSample extracted user_ids:", prompts["user_id"].head(5).tolist())
print("Unique users in prompts:", prompts["user_id"].nunique())

# ============================================================
# 3. Select only relevant ground truth columns
# ============================================================

LABEL_COLS = ["age_group", "household_size_group", "gender", "income_level"]
# add more if needed: "gender", "education", "main_employment" etc.

gt_labels = gt[["user_id"] + LABEL_COLS].copy()
gt_labels["user_id"] = gt_labels["user_id"].astype(str).str.strip()
prompts["user_id"]      = prompts["user_id"].astype(str).str.strip()

print("\nGT label sample:")
print(gt_labels.head(3))

# ============================================================
# 4. Merge on user_id
# ============================================================
merged = prompts.merge(gt_labels, on="user_id", how="inner")

print(f"\nAfter merge: {len(merged)} rows")
print(f"Unmatched prompts (no GT): {len(prompts) - len(merged)}")
print(merged[["user_id"] + LABEL_COLS].head(3))

# ============================================================
# 5. Drop rows with missing labels
# ============================================================
before = len(merged)
merged = merged.dropna(subset=LABEL_COLS)

# also drop "Prefer not to say" in income_level (uninformative)
merged = merged[~merged["income_level"].str.lower().str.contains("prefer not", na=False)]

print(f"\nAfter dropping missing/refused labels: {len(merged)} rows (dropped {before - len(merged)})")

# ============================================================
# 6. Build instruction-response pairs (Qwen2.5 chat format)
# ============================================================
SYSTEM_MSG = (
    "You are a mobility behaviour analyst. "
    "Given a one-week sequence of GPS staypoint events for a Swiss resident, "
    "predict their demographic profile: age group, household size, gender, and income level."
)

def build_label_str(row: pd.Series) -> str:
    return (
        f"Age group: {row['age_group']}\n"
        f"Household size: {row['household_size_group']}\n"
        f"Gender: {row['gender']}\n"
        f"Income level: {row['income_level']}"
    )

def build_chat_record(row: pd.Series) -> dict:
    return {
        "user_id": row["user_id"],
        "messages": [
            {"role": "system",    "content": SYSTEM_MSG},
            {"role": "user",      "content": str(row["prompt_text"]).strip()},
            {"role": "assistant", "content": build_label_str(row)},
        ]
    }

records = [build_chat_record(row) for _, row in merged.iterrows()]
print(f"\nTotal training records: {len(records)}")

# preview one record
print("\n--- Example record ---")
for msg in records[0]["messages"]:
    print(f"[{msg['role'].upper()}]\n{msg['content'][:300]}\n")

# ============================================================
# 7. 80 / 20 train-test split
# ============================================================
train_records, test_records = train_test_split(
    records, test_size=0.2, random_state=42
)
print(f"Train: {len(train_records)} | Test: {len(test_records)}")

# ============================================================
# 8. Save as JSONL (standard format for SFT fine-tuning)
# ============================================================
OUT_DIR = Path("/data/baliu/python_code/data/finetune_ready")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records → {path}")

save_jsonl(train_records, OUT_DIR / "train.jsonl")
save_jsonl(test_records,  OUT_DIR / "test.jsonl")

# also save merged CSV for inspection
merged.to_csv(OUT_DIR / "merged_prompt_gt.csv", index=False)
print("Done.")

GT shape  : (90909, 129)
Prompts shape: (109086, 1)

Sample extracted user_ids: ['AAGAF', '', '', "Based only on the mobility behaviour and nearby spatial context described below,infer the user's most likely household size and age group based on swiss census data and situation", '1, 2,3, 4, 5+, age: 0-17, 18-24, 25-34, 35-44, 45-54, 55-64, 65+. gender: male, female, other, perefer not to say']
Unique users in prompts: 63517

GT label sample:
  user_id age_group household_size_group  gender       income_level
0   AAALY     45-66                    4  Female  Prefer not to say
1   AAAPP     45-66                    4    Male          4001-8000
2   AAAVA     25-44                    2  Female             >16000

After merge: 1682 rows
Unmatched prompts (no GT): 107404
  user_id age_group household_size_group  gender income_level
0   AAGAF     25-44                    2  Female        <4000
1   AAINS     18-24                    4    Male  12001-16000
2   AAQME     18-24                   

In [25]:
import trl
print(trl.__version__)

# also see what's actually available
print([x for x in dir(trl) if "Collat" in x or "completion" in x.lower()])

0.29.0
['LogCompletionsCallback']


In [2]:
import json
import pandas as pd
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

# ============================================================
# PATHS & CONFIG
# ============================================================
GT_CSV    = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
SP_CSV    = Path("/data/baliu/python_code/data/sp_sampled_with_geocontext.csv")
OUT_DIR   = Path("/data/baliu/python_code/data/finetune_ready")
MODEL_DIR = Path("/data/baliu/python_code/models/qwen25_mobility_lora")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
LABEL_COLS = ["age_group", "household_size_group", "gender", "income_level"]
SEED       = 42
MAX_SEQ_LEN = 8091

# ============================================================
# STEP 1: Load data
# ============================================================
sp = pd.read_csv(SP_CSV, low_memory=False)
gt = pd.read_csv(GT_CSV, low_memory=False)
sp["user_id"] = sp["user_id"].astype(str).str.strip()
gt["user_id"] = gt["user_id"].astype(str).str.strip()
print(f"Staypoints: {len(sp)} rows | {sp['user_id'].nunique()} users")
print(f"GT        : {len(gt)} rows")

# ============================================================
# STEP 2: Build one prompt per user from trajectory tokens
# ============================================================
SYSTEM_MSG = (
    "You are a mobility behaviour analyst. "
    "Given a one-week sequence of GPS staypoint events for a Swiss resident, "
    "predict their demographic profile."
)

def build_user_prompt(df_u: pd.DataFrame) -> str:
    toks = tokens_compact_1week(df_u, max_events=40, prec=4)
    return "\n".join(toks)

prompt_records = []
for user_id, df_u in sp.groupby("user_id"):
    prompt_text = build_user_prompt(df_u)
    if prompt_text.strip():
        prompt_records.append({"user_id": user_id, "prompt_text": prompt_text})

prompt_df = pd.DataFrame(prompt_records)
print(f"\nBuilt prompts for {len(prompt_df)} users")

# ============================================================
# STEP 3: Merge with GT labels
# ============================================================
gt_labels = gt[["user_id"] + LABEL_COLS].drop_duplicates("user_id")
merged = prompt_df.merge(gt_labels, on="user_id", how="inner")
print(f"Matched users: {len(merged)}")

# drop missing / refused labels
merged = merged.dropna(subset=LABEL_COLS)
merged = merged[
    ~merged["income_level"].astype(str).str.lower().str.contains("prefer not", na=False)
]
print(f"After dropping bad labels: {len(merged)}")

# ============================================================
# STEP 4: Build chat format records
# ============================================================

def build_label_response(row):
    return (
        f"Age group: {row['age_group']}\n"
        f"Household size: {row['household_size_group']}\n"
        f"Gender: {row['gender']}\n"
        f"Income level: {row['income_level']}"
    )

records = [
    {
        "messages": [
            {"role": "system",    "content": SYSTEM_MSG},
            {"role": "user",      "content": row["prompt_text"].strip()},
            {"role": "assistant", "content": build_label_response(row)},
        ]
    }
    for _, row in merged.iterrows()
]
print(f"Total records: {len(records)}")

# preview
print("\n--- Example ---")
for msg in records[0]["messages"]:
    print(f"[{msg['role'].upper()}]\n{msg['content'][:300]}\n")

# ============================================================
# STEP 5: 80/20 split + save JSONL
# ============================================================
train_records, test_records = train_test_split(
    records, test_size=0.2, random_state=SEED
)
print(f"Train: {len(train_records)} | Test: {len(test_records)}")

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in data:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Saved → {path}")

save_jsonl(train_records, OUT_DIR / "train.jsonl")
save_jsonl(test_records,  OUT_DIR / "test.jsonl")

# ============================================================
# STEP 6: Tokeniser + apply chat template
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

def apply_chat_template(examples):
    return {
        "text": [
            tokenizer.apply_chat_template(
                msg, tokenize=False, add_generation_prompt=False
            )
            for msg in examples["messages"]
        ]
    }

train_ds = Dataset.from_list(train_records).map(apply_chat_template, batched=True)
test_ds  = Dataset.from_list(test_records).map(apply_chat_template,  batched=True)
print(f"\ntrain_ds: {len(train_ds)} | test_ds: {len(test_ds)}")
print("Sample text (first 400 chars):\n", train_ds[0]["text"][:400])

# ============================================================
# STEP 7: Load model (4-bit QLoRA)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=bnb_config,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

sft_config = trl.SFTConfig(
    output_dir=str(MODEL_DIR),

    # training
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    weight_decay=0.01,

    # accuracy-focused settings
    bf16=True,
    #
    # evaluation and saving
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,

    # others
    report_to="none",
    seed=SEED,
    logging_steps=20,
)

# ============================================================
# STEP 8: Train
# ============================================================
# training_args = TrainingArguments(
#     output_dir=str(MODEL_DIR),
#     num_train_epochs=3,
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,
#     gradient_accumulation_steps=8,
#     learning_rate=2e-4,
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.05,
#     weight_decay=0.01,
#     bf16=True,
#     logging_steps=20,
#     eval_strategy="steps",
#     eval_steps=100,
#     save_strategy="steps",
#     save_steps=100,
#     save_total_limit=3,
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     report_to="none",
#     seed=SEED,
# )

# formatting the dataset for trl
def formatting_func(examples):
    return examples["text"]

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset= train_ds,
    eval_dataset = test_ds,
    peft_config=lora_config,
    formatting_func=formatting_func,
    args = sft_config,
)

print("[INFO] Trainable parameters:")
trainer.model.print_trainable_parameters()

print("\n[INFO] Starting fine-tuning ...")
trainer.train()


trainer.save_model(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print(f"Saved → {MODEL_DIR / 'final'}")

metrics = trainer.evaluate()
print("Eval metrics:", metrics)

Staypoints: 60738 rows | 2102 users
GT        : 90909 rows


NameError: name 'tokens_compact_1week' is not defined

The fine-tuned qwen 2.5 for training v2, 16 Mar 2026

In [21]:
script = '''
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch, gc, json, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

# ============================================================
# PATHS
# ============================================================
GT_CSV    = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
SP_CSV    = Path("/data/baliu/python_code/data/sp_sampled_with_geocontext.csv")
OUT_DIR   = Path("/data/baliu/python_code/data/finetune_ready")
MODEL_DIR = Path("/data/baliu/python_code/models/qwen25_mobility_lora")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME  = "Qwen/Qwen2.5-3B-Instruct"
LABEL_COLS  = ["age_group", "household_size_group", "income_level", "gender","household_children"]
SEED        = 42
MAX_SEQ_LEN = 8192

# ============================================================
# Load data
# ============================================================
sp = pd.read_csv(SP_CSV, low_memory=False)
gt = pd.read_csv(GT_CSV, low_memory=False)
sp["user_id"] = sp["user_id"].astype(str).str.strip()
gt["user_id"] = gt["user_id"].astype(str).str.strip()
print(f"Staypoints: {len(sp)} rows | {sp['user_id'].nunique()} users")

# ============================================================
# LOAD TOKENIZER FIRST (needed for tokens_compact_1week)
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

# ============================================================
# BUILD PROMPTS
# ============================================================
DOW_NAMES = {0:"Sunday",1:"Monday",2:"Tuesday",3:"Wednesday",
             4:"Thursday",5:"Friday",6:"Saturday"}
HOUR_BIN_LABELS = {0:"midnight",1:"early morning",2:"morning",
                   3:"afternoon",4:"evening",5:"night"}

def clean_addr_part(s):
    if pd.isna(s): return None
    s = str(s).strip()
    return None if s.lower() in ["nan","none","-",""] else s

def tokens_compact_1week(df_u, max_events=40, prec=4):
    duration_col = next(
        (c for c in df_u.columns if c.lower() in
         ["duration", "duration_min", "act_duration", "act_duration_min", "dur_min"]),
        None
    )
    use_cols = ["started_at", "dow", "hour_bin", "location_id", "lon", "lat",
                "mode", "city", "neighbourhood", "road", "nearby_places", "postcode"]
    if duration_col:
        use_cols.append(duration_col)

    df = df_u.loc[:, [c for c in use_cols if c in df_u.columns]].copy()
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df = df.dropna(subset=["started_at"]).sort_values("started_at").reset_index(drop=True)
    if len(df) > max_events:
        df = df.head(max_events)

    toks, current_date = [], None

    for r in df.itertuples(index=False):
        t = r.started_at
        date_str = t.date().isoformat()
        if date_str != current_date:
            toks.append(f"--- Date: {date_str} ---")
            current_date = date_str

        hhmm     = t.strftime("%H:%M")
        dow_raw  = getattr(r, "dow", None)
        hb_raw   = getattr(r, "hour_bin", None)
        dow_name = DOW_NAMES.get(int(dow_raw), "unknown") if pd.notna(dow_raw) else "unknown"
        hb_label = HOUR_BIN_LABELS.get(int(hb_raw), "") if pd.notna(hb_raw) else ""

        lat = round(float(getattr(r, "lat", 0.0)), prec)
        lon = round(float(getattr(r, "lon", 0.0)), prec)

        addr_parts = [clean_addr_part(getattr(r, c, None))
                      for c in ["road", "neighbourhood", "city"]]
        addr = ", ".join(p for p in addr_parts if p) or "unknown address"
        postcode = clean_addr_part(getattr(r, "postcode", None))
        if postcode:
            addr = f"{addr} (postcode {postcode})"

        nearby_raw = getattr(r, "nearby_places", None)
        if isinstance(nearby_raw, list):
            nearby_str = "; ".join(nearby_raw[:5])
        elif isinstance(nearby_raw, str) and nearby_raw.strip():
            nearby_str = nearby_raw
        else:
            nearby_str = "no nearby places listed"

        mode    = str(getattr(r, "mode", "unknown")).strip().lower()
        mode    = "unknown" if mode in ["nan", "none", ""] else mode
        dur_val = getattr(r, duration_col, None) if duration_col else None
        dur_str = (f"{int(round(float(dur_val)))}"
                   if dur_val is not None and pd.notna(dur_val) else "unknown")

        # ── natural language format (Option B) ──────────────────────────
        time_ctx = (f"At {hhmm} ({dow_name}, {hb_label})"
                    if hb_label else f"At {hhmm} ({dow_name})")
        toks.append(
            f"{time_ctx}, the user traveled by {mode} to {addr} "
            f"(coordinates: {lat}, {lon}), staying for approximately {dur_str} minutes. "
            f"The surrounding area features {nearby_str}."
        )
    return toks   # ← function ends here


# ── print sample tokens OUTSIDE the function ────────────────────────────
# sample_user = sp["user_id"].iloc[0]
# sample_df   = sp[sp["user_id"] == sample_user]
# sample_toks = tokens_compact_1week(sample_df)
# for t in sample_toks[:5]:
#     print(t)
    
    
# The cause the model learns to respond to must be identical in training and evaluation, so we define it once here and reuse in both places.
# Update the system message to be more specific about the task and the expected output format, to guide the model's learning and inference.

SYSTEM_MSG = (
    "You are an expert in human mobility behaviour analysis and Swiss sociodemographics. "
    "Your task is to infer a resident's demographic profile solely from their one week GPS staypoint trajectory."
    "The trajectory contains sequences of visited locations with timestamps, transport modes, "
    "stay durations, addresses, and nearby points of interest. These behavioural signals "
    "reveal demographic characteristics:\\n"
    "  - Daily routine and working hours suggest age group and employment status\\n"
    "  - Transport mode (car, bike, public transit, walking) correlates with income and age\\n"
    "  - Types of visited places (schools, offices, shops, parks) indicate household structure\\n"
    "  - Spatial range and neighbourhood type reflect income level\\n"
    "  - Weekend vs weekday patterns distinguish household size and family structure\\n\\n"
    "Based on these mobility patterns, predict the user's demographic profile.\\n"
    "Reply ONLY in this exact format with no extra text or explanation:\\n"
    "Age group: <value>\\n"
    "Household size: <value>\\n"
    "Income level: <value>\\n"
    "Gender: <value>\\n\\n"
    "Household children: <value>\\n\\n"
    "Valid values only:\\n"
    "  Age group     : 0-17, 18-24, 25-44, 45-66, 67+\\n"
    "  Household size: 1, 2, 3, 4, 5+\\n"
    "  Income level  : <4000, 4001-8000, 8001-12000, 12001-16000, 16001+\\n"
    "  Gender        : male, female, other, prefer not to say\\n"
    "  Household children: 0, 1, 2, 3, 4+ \\n" 
)

# user message. the trajectory input (one per user) built automatically by tokens_compact_1week, which converts the raw staypoint data into a concise textual format that captures the key mobility patterns and contextual information relevant for demographic inference.
prompt_records = []
for user_id, df_u in sp.groupby("user_id"):
    toks = tokens_compact_1week(df_u, max_events=40, prec=4)
    prompt_text = "\\n".join(toks) # fix \\n issue when writing to CSV
    if prompt_text.strip():
        prompt_records.append({"user_id": user_id, "prompt_text": prompt_text})

prompt_df = pd.DataFrame(prompt_records)
print(f"Built prompts for {len(prompt_df)} users")

# ============================================================
# MERGE + CLEAN LABELS
# ============================================================
# household_children is derived: kk (infants) + k (children) + jug (youth)
# compute it from gt BEFORE merging
gt["household_children"] = (
    pd.to_numeric(gt["hh_size_kk"],  errors="coerce").fillna(0) +
    pd.to_numeric(gt["hh_size_k"],   errors="coerce").fillna(0) +
    pd.to_numeric(gt["hh_size_jug"], errors="coerce").fillna(0)
).astype(int)

# cap at 4+ to match valid values in SYSTEM_MSG
gt["household_children"] = gt["household_children"].apply(
    lambda x: "4+" if x >= 4 else str(x)
)

# verify
print("household_children unique values:", gt["household_children"].unique())

label_cols = ["age_group", "household_size_group", "income_level", "gender", "household_children"]
gt_labels = gt[["user_id"] + label_cols].drop_duplicates("user_id")
merged    = prompt_df.merge(gt_labels, on="user_id", how="inner")
merged    = merged.dropna(subset=label_cols)
merged    = merged[~merged["income_level"].astype(str).str.lower().str.contains("prefer not", na=False)]
print(f"Final records: {len(merged)}")

# normalise label values (e.g. income_level has some messy values like ">16000" or "16001+", age_group has "65+" etc.)
def normalise_income(val):
    val = str(val).strip()
    mapping = {
        ">16000"  : "16001+",
        "16001+"  : "16001+",
        "<4000"   : "<4000",
    }
    return mapping.get(val, val)

def normalise_age(val):
    val = str(val).strip()
    mapping = {
        "61+" : "67+",
        "65+" : "67+",
    }
    return mapping.get(val, val)

def normalise_gender(val):
    return str(val).strip().lower()

merged["income_level"]        = merged["income_level"].apply(normalise_income)
merged["age_group"]           = merged["age_group"].apply(normalise_age)
merged["gender"]              = merged["gender"].apply(normalise_gender)

# verify the unique values are clean
print("income_level values :", merged["income_level"].unique())
print("age_group values    :", merged["age_group"].unique())
print("gender values       :", merged["gender"].unique())
print("household_size values:", merged["household_size_group"].unique())

# ============================================================
# BUILD CHAT RECORDS + SPLIT
# the "effect" the model learns to produce must be identical in training and evaluation, 
# so we define it once here and reuse in both places.
# ============================================================
def build_label_response(row):
    return (f"Age group: {row[\'age_group\']}\\n"
            f"Household size: {row[\'household_size_group\']}\\n"
            f"Income level: {row[\'income_level\']}\\n"
            f"Gender: {row[\'gender\']}\\n"
            f"Household children: {row[\'household_children\']}") 

records = [
    {"messages": [
        {"role": "system",    "content": SYSTEM_MSG},  # cause
        {"role": "user",      "content": row["prompt_text"].strip()},    # input
        {"role": "assistant", "content": build_label_response(row)},    # effect
    ]}
    for _, row in merged.iterrows()
]

train_records, test_records = train_test_split(records, test_size=0.2, random_state=SEED)
print(f"Train: {len(train_records)} | Test: {len(test_records)}")

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in data:
            f.write(json.dumps(rec, ensure_ascii=False) + "\\n")

save_jsonl(train_records, OUT_DIR / "train.jsonl")
save_jsonl(test_records,  OUT_DIR / "test.jsonl")

# ============================================================
# APPLY CHAT TEMPLATE
# ============================================================
def apply_chat_template(examples):
    return {"text": [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in examples["messages"]
    ]}

train_ds = Dataset.from_list(train_records).map(apply_chat_template, batched=True)
test_ds  = Dataset.from_list(test_records).map(apply_chat_template,  batched=True)

def truncate_text(examples):
    return {"text": [
        tokenizer.decode(
            tokenizer.encode(t, max_length=MAX_SEQ_LEN, truncation=True),
            skip_special_tokens=False
        )
        for t in examples["text"]
    ]}

train_ds = train_ds.map(truncate_text, batched=True)
test_ds  = test_ds.map(truncate_text, batched=True)
print("[CHECK] Truncation to 8192 applied directly on dataset")

# ============================================================
# LOAD MODEL
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.enable_input_require_grads()
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
print(f"Model loaded. GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB / 22 GB")

# ============================================================
# TRAIN # more capacity than before to capture the complex mobility-demographics relationships, but monitor for overfitting.
# ============================================================
lora_config = LoraConfig( 
    r=16, lora_alpha=32,  # need to adjust those hyperparameters based on the model size and task complexity. 16/32 is a common starting point for 4-bit QLoRA, but we can experiment with higher values if the model struggles to learn the nuanced patterns in the mobility data.
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"], ### for qwen2.5, we need to include the additional gated MLP projections to effectively fine-tune the model's understanding of the complex mobility patterns and their demographic implications.
    lora_dropout=0.05, 
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )

sft_config = SFTConfig(
    output_dir=str(MODEL_DIR),
    num_train_epochs=5 ,  # given the complexity of the task and the model's capacity, we increase the epochs to allow more learning, but we will monitor eval metrics closely to prevent overfitting.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,  # warmup ratio deprecated,  to warmup_steps
    weight_decay=0.01,
    bf16=True,

    optim="paged_adamw_8bit",
    dataloader_pin_memory=False,
    dataloader_num_workers=0,

    dataset_text_field="text",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    seed=SEED,
    logging_steps=10,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    peft_config=lora_config,
    args=sft_config,

)

trainer.model.print_trainable_parameters()
print("\\n[INFO] Starting training ...")
trainer.train()

trainer.save_model(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print(f"Done. Saved to {MODEL_DIR / 'final'}")

metrics = trainer.evaluate()
print("Eval metrics:", metrics)
pd.Series(metrics).to_csv(MODEL_DIR / "eval_metrics.csv")
'''

# save the script to a .py file 
script_path = "/data/baliu/python_code/train_qwen25_v4.py"
with open(script_path, "w") as f:
    f.write(script)
print(f"Script saved to {script_path}")

Script saved to /data/baliu/python_code/train_qwen25_v4.py


In [ ]:
script = '''
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch, gc, json, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

# ============================================================
# PATHS
# ============================================================
GT_CSV    = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
SP_CSV    = Path("/data/baliu/python_code/data/sp_sampled_with_geocontext.csv")
OUT_DIR   = Path("/data/baliu/python_code/data/finetune_ready")
MODEL_DIR = Path("/data/baliu/python_code/models/qwen25_mobility_lora")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME  = "Qwen/Qwen2.5-3B-Instruct"
LABEL_COLS  = ["age_group", "household_size_group", "income_level", "gender"]
SEED        = 42
MAX_SEQ_LEN = 8192

# ============================================================
# LOAD DATA
# ============================================================
sp = pd.read_csv(SP_CSV, low_memory=False)
gt = pd.read_csv(GT_CSV, low_memory=False)
sp["user_id"] = sp["user_id"].astype(str).str.strip()
gt["user_id"] = gt["user_id"].astype(str).str.strip()
print(f"Staypoints: {len(sp)} rows | {sp['user_id'].nunique()} users")

# ============================================================
# LOAD TOKENIZER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

# ============================================================
# BUILD PROMPTS
# ============================================================
DOW_NAMES = {0:"Sunday",1:"Monday",2:"Tuesday",3:"Wednesday",
             4:"Thursday",5:"Friday",6:"Saturday"}
HOUR_BIN_LABELS = {0:"midnight",1:"early morning",2:"morning",
                   3:"afternoon",4:"evening",5:"night"}

def clean_addr_part(s):
    if pd.isna(s): return None
    s = str(s).strip()
    return None if s.lower() in ["nan","none","-",""] else s

def tokens_compact_1week(df_u, max_events=40, prec=4):
    duration_col = next(
        (c for c in df_u.columns if c.lower() in
         ["duration", "duration_min", "act_duration", "act_duration_min", "dur_min"]),
        None
    )
    use_cols = ["started_at", "dow", "hour_bin", "location_id", "lon", "lat",
                "mode", "city", "neighbourhood", "road", "nearby_places", "postcode"]
    if duration_col:
        use_cols.append(duration_col)

    df = df_u.loc[:, [c for c in use_cols if c in df_u.columns]].copy()
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df = df.dropna(subset=["started_at"]).sort_values("started_at").reset_index(drop=True)
    if len(df) > max_events:
        df = df.head(max_events)

    toks, current_date = [], None

    for r in df.itertuples(index=False):
        t = r.started_at
        date_str = t.date().isoformat()
        if date_str != current_date:
            toks.append(f"--- Date: {date_str} ---")
            current_date = date_str

        hhmm     = t.strftime("%H:%M")
        dow_raw  = getattr(r, "dow", None)
        hb_raw   = getattr(r, "hour_bin", None)
        dow_name = DOW_NAMES.get(int(dow_raw), "unknown") if pd.notna(dow_raw) else "unknown"
        hb_label = HOUR_BIN_LABELS.get(int(hb_raw), "") if pd.notna(hb_raw) else ""

        lat = round(float(getattr(r, "lat", 0.0)), prec)
        lon = round(float(getattr(r, "lon", 0.0)), prec)

        addr_parts = [clean_addr_part(getattr(r, c, None))
                      for c in ["road", "neighbourhood", "city"]]
        addr = ", ".join(p for p in addr_parts if p) or "unknown address"
        postcode = clean_addr_part(getattr(r, "postcode", None))
        if postcode:
            addr = f"{addr} (postcode {postcode})"

        nearby_raw = getattr(r, "nearby_places", None)
        if isinstance(nearby_raw, list):
            nearby_str = "; ".join(nearby_raw[:5])
        elif isinstance(nearby_raw, str) and nearby_raw.strip():
            nearby_str = nearby_raw
        else:
            nearby_str = "no nearby places listed"

        mode    = str(getattr(r, "mode", "unknown")).strip().lower()
        mode    = "unknown" if mode in ["nan", "none", ""] else mode
        dur_val = getattr(r, duration_col, None) if duration_col else None
        dur_str = (f"{int(round(float(dur_val)))}"
                   if dur_val is not None and pd.notna(dur_val) else "unknown")

        time_ctx = (f"At {hhmm} ({dow_name}, {hb_label})"
                    if hb_label else f"At {hhmm} ({dow_name})")
        toks.append(
            f"{time_ctx}, the user traveled by {mode} to {addr} "
            f"(coordinates: {lat}, {lon}), staying for approximately {dur_str} minutes. "
            f"The surrounding area features {nearby_str}."
        )
    return toks


# ============================================================
# SYSTEM MESSAGE — identical in training and evaluation
# ============================================================
SYSTEM_MSG = (
    "You are an expert in human mobility behaviour analysis and Swiss sociodemographics. "
    "Your task is to infer a resident\'s demographic profile solely from their one week GPS staypoint trajectory. "
    "The trajectory contains sequences of visited locations with timestamps, transport modes, "
    "stay durations, addresses, and nearby points of interest. These behavioural signals "
    "reveal demographic characteristics:\\n"
    "  - Daily routine and working hours suggest age group and employment status\\n"
    "  - Transport mode (car, bike, public transit, walking) correlates with income and age\\n"
    "  - Types of visited places (schools, offices, shops, parks) indicate household structure\\n"
    "  - Spatial range and neighbourhood type reflect income level\\n"
    "  - Weekend vs weekday patterns distinguish household size and family structure\\n\\n"
    "Based on these mobility patterns, predict the user\'s demographic profile.\\n"
    "Reply ONLY in this exact format with no extra text or explanation:\\n"
    "Age group: <value>\\n"
    "Household size: <value>\\n"
    "Income level: <value>\\n"
    "Gender: <value>\\n\\n"
    "Valid values only:\\n"
    "  Age group     : 0-17, 18-24, 25-44, 45-66, 67+\\n"
    "  Household size: 1, 2, 3, 4, 5+\\n"
    "  Income level  : <4000, 4001-8000, 8001-12000, 12001-16000, 16001+\\n"
    "  Gender        : male, female, other, prefer not to say\\n"
)

# ============================================================
# BUILD USER PROMPTS
# ============================================================
prompt_records = []
for user_id, df_u in sp.groupby("user_id"):
    toks = tokens_compact_1week(df_u, max_events=40, prec=4)
    prompt_text = "\\n".join(toks)
    if prompt_text.strip():
        prompt_records.append({"user_id": user_id, "prompt_text": prompt_text})

prompt_df = pd.DataFrame(prompt_records)
print(f"Built prompts for {len(prompt_df)} users")

# ============================================================
# MERGE + CLEAN LABELS
# ============================================================
label_cols = ["age_group", "household_size_group", "income_level", "gender"]
gt_labels = gt[["user_id"] + label_cols].drop_duplicates("user_id")
merged    = prompt_df.merge(gt_labels, on="user_id", how="inner")
merged    = merged.dropna(subset=label_cols)
merged    = merged[~merged["income_level"].astype(str).str.lower().str.contains("prefer not", na=False)]
print(f"Final records: {len(merged)}")

# normalise labels
def normalise_income(val):
    val = str(val).strip()
    return {">16000": "16001+", "16001+": "16001+", "<4000": "<4000"}.get(val, val)

def normalise_age(val):
    val = str(val).strip()
    return {"61+": "67+", "65+": "67+"}.get(val, val)

def normalise_gender(val):
    return str(val).strip().lower()

merged["income_level"] = merged["income_level"].apply(normalise_income)
merged["age_group"]    = merged["age_group"].apply(normalise_age)
merged["gender"]       = merged["gender"].apply(normalise_gender)

print("income_level values  :", merged["income_level"].unique())
print("age_group values     :", merged["age_group"].unique())
print("gender values        :", merged["gender"].unique())
print("household_size values:", merged["household_size_group"].unique())

# ============================================================
# BUILD CHAT RECORDS + SPLIT
# ============================================================
def build_label_response(row):
    return (f"Age group: {row[\'age_group\']}\\n"
            f"Household size: {row[\'household_size_group\']}\\n"
            f"Income level: {row[\'income_level\']}\\n"
            f"Gender: {row[\'gender\']}")

records = [
    {"messages": [
        {"role": "system",    "content": SYSTEM_MSG},
        {"role": "user",      "content": row["prompt_text"].strip()},
        {"role": "assistant", "content": build_label_response(row)},
    ]}
    for _, row in merged.iterrows()
]

train_records, test_records = train_test_split(records, test_size=0.2, random_state=SEED)
print(f"Train: {len(train_records)} | Test: {len(test_records)}")

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in data:
            f.write(json.dumps(rec, ensure_ascii=False) + "\\n")

save_jsonl(train_records, OUT_DIR / "train.jsonl")
save_jsonl(test_records,  OUT_DIR / "test.jsonl")

# ============================================================
# APPLY CHAT TEMPLATE + TRUNCATE
# ============================================================
def apply_chat_template(examples):
    return {"text": [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in examples["messages"]
    ]}

def truncate_text(examples):
    return {"text": [
        tokenizer.decode(
            tokenizer.encode(t, max_length=MAX_SEQ_LEN, truncation=True),
            skip_special_tokens=False
        )
        for t in examples["text"]
    ]}

train_ds = Dataset.from_list(train_records).map(apply_chat_template, batched=True)
test_ds  = Dataset.from_list(test_records).map(apply_chat_template,  batched=True)
train_ds = train_ds.map(truncate_text, batched=True)
test_ds  = test_ds.map(truncate_text,  batched=True)
print("[CHECK] Truncation to 8192 applied directly on dataset")

# ============================================================
# LOAD MODEL
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.enable_input_require_grads()
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
print(f"Model loaded. GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB / 22 GB")

# ============================================================
# LORA CONFIG
# ============================================================
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# ============================================================
# SFT CONFIG
# ============================================================
sft_config = SFTConfig(
    output_dir=str(MODEL_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    weight_decay=0.01,
    bf16=True,
    optim="paged_adamw_8bi",
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    dataset_text_field="text",
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    seed=SEED,
    logging_steps=10,
)

# ============================================================
# TRAIN
# ============================================================
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    peft_config=lora_config,
    args=sft_config,
)

trainer.model.print_trainable_parameters()
print("\\n[INFO] Starting training ...")
trainer.train()

trainer.save_model(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print(f"Done. Saved to {MODEL_DIR / \'final\'}")

metrics = trainer.evaluate()
print("Eval metrics:", metrics)
pd.Series(metrics).to_csv(MODEL_DIR / "eval_metrics.csv")
'''

script_path = "/data/baliu/python_code/train_qwen25_v4.py"
with open(script_path, "w") as f:
    f.write(script)
print(f"Script saved to {script_path}")

Script saved to /data/baliu/python_code/train_qwen25_v4.py


In [26]:
import pandas as pd
import json
from pathlib import Path

# ============================================================
# LOAD THE SAVED JSONL FILES
# ============================================================
TRAIN_JSONL = Path("/data/baliu/python_code/data/finetune_ready/train.jsonl")
TEST_JSONL  = Path("/data/baliu/python_code/data/finetune_ready/test.jsonl")

train_records = [json.loads(l) for l in open(TRAIN_JSONL)]
test_records  = [json.loads(l) for l in open(TEST_JSONL)]

print(f"Train records: {len(train_records)}")
print(f"Test records : {len(test_records)}")

# ============================================================
# CHECK ONE FULL SAMPLE
# ============================================================
sample = train_records[0]["messages"]

print("\n" + "="*60)
print("SYSTEM MESSAGE")
print("="*60)
print(sample[0]["content"])

print("\n" + "="*60)
print("USER MESSAGE (trajectory) — first 12000 chars")
print("="*60)
print(sample[1]["content"][:12000])
print("..." if len(sample[1]["content"]) > 12000 else "")

print("\n" + "="*60)
print("ASSISTANT REPLY (ground truth labels)")
print("="*60)
print(sample[2]["content"])

# ============================================================
# CHECK LABEL DISTRIBUTION
# ============================================================
print("\n" + "="*60)
print("LABEL DISTRIBUTION ACROSS ALL TRAIN RECORDS")
print("="*60)

import collections

age_counts      = collections.Counter()
household_counts = collections.Counter()
income_counts   = collections.Counter()
gender_counts   = collections.Counter()

for rec in train_records:
    gt = rec["messages"][2]["content"]
    for line in gt.strip().splitlines():
        if ":" in line:
            key, _, val = line.partition(":")
            key = key.strip().lower().replace(" ", "_")
            val = val.strip()
            if key == "age_group":       age_counts[val] += 1
            if key == "household_size":  household_counts[val] += 1
            if key == "income_level":    income_counts[val] += 1
            if key == "gender":          gender_counts[val] += 1

print("\nAge group      :", dict(sorted(age_counts.items())))
print("Household size :", dict(sorted(household_counts.items())))
print("Income level   :", dict(sorted(income_counts.items())))
print("Gender         :", dict(sorted(gender_counts.items())))

# ============================================================
# CHECK TOKEN LENGTHS
# ============================================================
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct", trust_remote_code=True
)

lengths = [
    len(tokenizer.encode(rec["messages"][1]["content"]))
    for rec in train_records
]

print("\n" + "="*60)
print("USER MESSAGE TOKEN LENGTH STATS")
print("="*60)
s = pd.Series(lengths)
print(s.describe(percentiles=[.5, .75, .9, .95, .99]).round(0))
print(f"\nOver 8192 tokens: {(s > 8192).sum()} / {len(s)}")
print(f"Over 4096 tokens: {(s > 4096).sum()} / {len(s)}")

Train records: 1221
Test records : 306

SYSTEM MESSAGE
You are an expert in human mobility behaviour analysis and Swiss sociodemographics. Your task is to infer a resident's demographic profile solely from their one week GPS staypoint trajectory. The trajectory contains sequences of visited locations with timestamps, transport modes, stay durations, addresses, and nearby points of interest. These behavioural signals reveal demographic characteristics:
  - Daily routine and working hours suggest age group and employment status
  - Transport mode (car, bike, public transit, walking) correlates with income and age
  - Types of visited places (schools, offices, shops, parks) indicate household structure
  - Spatial range and neighbourhood type reflect income level
  - Weekend vs weekday patterns distinguish household size and family structure

Based on these mobility patterns, predict the user's demographic profile.
Reply ONLY in this exact format with no extra text or explanation:
Age gro

# Udpate fine tuning model v5 at 18 Mar 2026

In [7]:
%pip install scikit-learn


  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SK

In [8]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
 
import torch, gc, json
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM  # no BitsAndBytesConfig
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig
 
# ============================================================
# PATHS
# ============================================================
GT_CSV    = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
SP_CSV    = Path("/data/baliu/python_code/data/sp_sampled_with_geocontext.csv")
OUT_DIR   = Path("/data/baliu/python_code/data/finetune_ready_gptoss")   # ← fixed: was v5
MODEL_DIR = Path("/data/baliu/python_code/models/gpt-oss-20b_mobility_lora")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
 
MODEL_NAME  = "/data/baliu/python_code/models/gpt_oss_20b_base"  # ← fixed: local path, not "gpt-oss-20b"
LABEL_COLS  = ["age_group", "household_size_group", "income_level", "gender"]
SEED        = 42
MAX_SEQ_LEN = 4096   # ← fixed: was 8192, reduced for 20B on 24GB
 
# ============================================================
# LOAD DATA
# ============================================================
sp = pd.read_csv(SP_CSV, low_memory=False)
gt = pd.read_csv(GT_CSV, low_memory=False)
sp["user_id"] = sp["user_id"].astype(str).str.strip()
gt["user_id"] = gt["user_id"].astype(str).str.strip()
print(f"Staypoints: {len(sp)} rows | {sp['user_id'].nunique()} users")
 
# ============================================================
# LOAD TOKENIZER
# gpt-oss uses the "harmony" chat template bundled in the
# tokenizer config — apply_chat_template handles it automatically
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
 
# ============================================================
# BUILD PROMPTS
# ============================================================
DOW_NAMES = {0:"Sunday",1:"Monday",2:"Tuesday",3:"Wednesday",
             4:"Thursday",5:"Friday",6:"Saturday"}
HOUR_BIN_LABELS = {0:"midnight",1:"early morning",2:"morning",
                   3:"afternoon",4:"evening",5:"night"}
 
def clean_addr_part(s):
    if pd.isna(s): return None
    s = str(s).strip()
    return None if s.lower() in ["nan","none","-",""] else s
 
def tokens_compact_1week(df_u, max_events=40, prec=4):
    duration_col = next(
        (c for c in df_u.columns if c.lower() in
         ["duration", "duration_min", "act_duration", "act_duration_min", "dur_min"]),
        None
    )
    use_cols = ["started_at", "dow", "hour_bin", "location_id", "lon", "lat",
                "mode", "city", "neighbourhood", "road", "nearby_places", "postcode"]
    if duration_col:
        use_cols.append(duration_col)
 
    df = df_u.loc[:, [c for c in use_cols if c in df_u.columns]].copy()
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df = df.dropna(subset=["started_at"]).sort_values("started_at").reset_index(drop=True)
    if len(df) > max_events:
        df = df.head(max_events)
 
    toks, current_date = [], None
    for r in df.itertuples(index=False):
        t = r.started_at
        date_str = t.date().isoformat()
        if date_str != current_date:
            toks.append(f"--- Date: {date_str} ---")
            current_date = date_str
 
        hhmm     = t.strftime("%H:%M")
        dow_raw  = getattr(r, "dow", None)
        hb_raw   = getattr(r, "hour_bin", None)
        dow_name = DOW_NAMES.get(int(dow_raw), "unknown") if pd.notna(dow_raw) else "unknown"
        hb_label = HOUR_BIN_LABELS.get(int(hb_raw), "") if pd.notna(hb_raw) else ""
 
        lat = round(float(getattr(r, "lat", 0.0)), prec)
        lon = round(float(getattr(r, "lon", 0.0)), prec)
 
        addr_parts = [clean_addr_part(getattr(r, c, None))
                      for c in ["road", "neighbourhood", "city"]]
        addr = ", ".join(p for p in addr_parts if p) or "unknown address"
        postcode = clean_addr_part(getattr(r, "postcode", None))
        if postcode:
            addr = f"{addr} (postcode {postcode})"
 
        nearby_raw = getattr(r, "nearby_places", None)
        if isinstance(nearby_raw, list):
            nearby_str = "; ".join(nearby_raw[:5])
        elif isinstance(nearby_raw, str) and nearby_raw.strip():
            nearby_str = nearby_raw
        else:
            nearby_str = "no nearby places listed"
 
        mode    = str(getattr(r, "mode", "unknown")).strip().lower()
        mode    = "unknown" if mode in ["nan", "none", ""] else mode
        dur_val = getattr(r, duration_col, None) if duration_col else None
        dur_str = (f"{int(round(float(dur_val)))}"
                   if dur_val is not None and pd.notna(dur_val) else "unknown")
 
        time_ctx = (f"At {hhmm} ({dow_name}, {hb_label})"
                    if hb_label else f"At {hhmm} ({dow_name})")
        toks.append(
            f"{time_ctx}, the user traveled by {mode} to {addr} "
            f"(coordinates: {lat}, {lon}), staying for approximately {dur_str} minutes. "
            f"The surrounding area features {nearby_str}."
        )
    return toks
 
# ============================================================
# SYSTEM MESSAGE
# ============================================================
SYSTEM_MSG = (
    "You are learning to predict the sociodemographic profile of Swiss residents "
    "from their one-week GPS mobility trajectories.\n\n"
    "During training, you are shown real Swiss residents whose age group, household size, "
    "income level, and gender are known from survey data. For each resident, you also see "
    "their full one-week staypoint trajectory.\n\n"
    "Learn the correspondence between mobility patterns and sociodemographics:\n"
    "  - Daily commute to university by train, low income, household 3-4  →  student (age 18-24)\n"
    "  - Regular weekday drives to office, high income, household 2  →  professional (age 25-66)\n"
    "  - Short local walks only, limited range, household 1  →  retired (age 67+)\n"
    "  - Weekend visits to playgrounds/schools, household 4-5+  →  family with children\n"
    "  - Wide spatial range, car use, upscale neighbourhoods  →  higher income\n\n"
    "At inference time, apply what you have learned to predict the demographic profile.\n\n"
    "OUTPUT RULES — follow exactly:\n"
    "  1. Output ONLY the four lines below. No explanation, no extra text.\n"
    "  2. Do NOT repeat the trajectory.\n"
    "  3. Replace each <value> with exactly one valid value.\n\n"
    "Age group: <value>\n"
    "Household size: <value>\n"
    "Income level: <value>\n"
    "Gender: <value>\n\n"
    "Valid values:\n"
    "  Age group     : 0-17 | 18-24 | 25-44 | 45-66 | 67+\n"
    "  Household size: 1 | 2 | 3 | 4 | 5+\n"
    "  Income level  : <4000 | 4001-8000 | 8001-12000 | 12001-16000 | 16001+\n"
    "  Gender        : male | female | other | prefer not to say\n"
)
 
# ============================================================
# BUILD USER PROMPTS
# ============================================================
prompt_records = []
for user_id, df_u in sp.groupby("user_id"):
    toks = tokens_compact_1week(df_u, max_events=40, prec=4)
    prompt_text = "\n".join(toks)
    if prompt_text.strip():
        prompt_records.append({"user_id": user_id, "prompt_text": prompt_text})
 
prompt_df = pd.DataFrame(prompt_records)
print(f"Built prompts for {len(prompt_df)} users")
 
# ============================================================
# MERGE + CLEAN LABELS
# ============================================================
label_cols = ["age_group", "household_size_group", "income_level", "gender"]
gt_labels  = gt[["user_id"] + label_cols].drop_duplicates("user_id")
merged     = prompt_df.merge(gt_labels, on="user_id", how="inner")
merged     = merged.dropna(subset=label_cols)
merged = merged[~merged["income_level"].astype(str).str.lower().str.contains("prefer not", na=False)]
merged = merged[~merged["household_size_group"].astype(str).str.lower().str.contains("unknown", na=False)]
merged = merged[~merged["gender"].astype(str).str.lower().str.contains("prefer not", na=False)]
print(f"Final records after cleaning: {len(merged)}")
 
def normalise_income(val):
    val = str(val).strip()
    return {">16000": "16001+", "16001+": "16001+", "<4000": "<4000"}.get(val, val)
 
def normalise_age(val):
    val = str(val).strip()
    return {"61+": "67+", "65+": "67+"}.get(val, val)
 
def normalise_gender(val):
    return str(val).strip().lower()
 
def normalise_household(val):
    val = str(val).strip()
    try:
        n = int(float(val))
        return "5+" if n >= 5 else str(n)
    except:
        return val
 
merged["income_level"]         = merged["income_level"].apply(normalise_income)
merged["age_group"]            = merged["age_group"].apply(normalise_age)
merged["gender"]               = merged["gender"].apply(normalise_gender)
merged["household_size_group"] = merged["household_size_group"].apply(normalise_household)
 
print("age_group values     :", sorted(merged["age_group"].unique()))
print("household_size values:", sorted(merged["household_size_group"].unique()))
print("income_level values  :", sorted(merged["income_level"].unique()))
print("gender values        :", sorted(merged["gender"].unique()))
 
# ============================================================
# BUILD CHAT RECORDS
# ============================================================
def build_label_response(row):
    return (f"Age group: {row['age_group']}\n"
            f"Household size: {row['household_size_group']}\n"
            f"Income level: {row['income_level']}\n"
            f"Gender: {row['gender']}")
 
records = [
    {"messages": [
        {"role": "system",    "content": SYSTEM_MSG},
        {"role": "user",      "content": row["prompt_text"].strip()},
        {"role": "assistant", "content": build_label_response(row)},
    ]}
    for _, row in merged.iterrows()
]
 
train_records, test_records = train_test_split(records, test_size=0.2, random_state=SEED)
print(f"Train: {len(train_records)} | Test: {len(test_records)}")
 
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in data:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
 
save_jsonl(train_records, OUT_DIR / "train.jsonl")
save_jsonl(test_records,  OUT_DIR / "test.jsonl")
 
# ============================================================
# TRUNCATE USER TURN ONLY
# ============================================================
RESERVED   = 350   # ← fixed: was 250, increased for larger system message
USER_LIMIT = MAX_SEQ_LEN - RESERVED
 
def safe_truncate(examples):
    results = []
    for msg_list in examples["messages"]:
        system_text    = msg_list[0]["content"]
        user_text      = msg_list[1]["content"]
        assistant_text = msg_list[2]["content"]
 
        user_tokens = tokenizer.encode(user_text)
        if len(user_tokens) > USER_LIMIT:
            user_tokens = user_tokens[:USER_LIMIT]
            user_text   = tokenizer.decode(user_tokens, skip_special_tokens=True)
 
        results.append(tokenizer.apply_chat_template(
            [{"role": "system",    "content": system_text},
             {"role": "user",      "content": user_text},
             {"role": "assistant", "content": assistant_text}],
            tokenize=False, add_generation_prompt=False
        ))
    return {"text": results}
 
train_ds = Dataset.from_list(train_records).map(safe_truncate, batched=True)
test_ds  = Dataset.from_list(test_records).map(safe_truncate,  batched=True)
 
# sanity check
sample_text = train_ds[0]["text"]
print("\n[CHECK] Last 200 chars of first training sample:")
print(sample_text[-200:])
assert "Age group:"    in sample_text, "MISSING: Age group!"
assert "Income level:" in sample_text, "MISSING: Income level!"
print("[CHECK] Labels confirmed — OK")
 
# ============================================================
# LOAD MODEL
# gpt-oss-20b has native MXFP4 quantization built in.
# Do NOT pass BitsAndBytesConfig — it will conflict with MXFP4.
# ============================================================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.enable_input_require_grads()
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
used = torch.cuda.memory_allocated() / 1024**3
print(f"Model loaded. GPU: {used:.2f} GB / 24 GB")
assert used < 22, f"OOM risk: {used:.1f} GB used before training!"
 
# ============================================================
# LORA CONFIG
# r=8 (not 16): saves ~2GB of adapter memory for 20B model
# attention projections only: further reduces memory footprint
# ============================================================
lora_config = LoraConfig(
    r=8,             # ← fixed: was 16, reduced for 20B memory budget
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # ← fixed: attention only
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
 
# ============================================================
# SFT CONFIG
# ============================================================
sft_config = SFTConfig(
    output_dir                  = str(MODEL_DIR / "checkpoints"),
    num_train_epochs            = 3,       # fewer epochs — 20B learns faster
    per_device_train_batch_size = 1,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = 16,
    learning_rate               = 1e-4,    # slightly lower than 3B (2e-4)
    lr_scheduler_type           = "cosine",
    warmup_steps                = 30,
    weight_decay                = 0.01,
    bf16                        = True,
    optim                       = "adamw_torch",   # ← fixed: no paged_adamw (BnB feature)
    dataloader_pin_memory       = False,
    dataloader_num_workers      = 0,
    dataset_text_field          = "text",
    eval_strategy               = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 50,
    save_total_limit            = 2,       # ← fixed: removed duplicate save_steps/save_total_limit
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    report_to                   = "none",
    seed                        = SEED,
    logging_steps               = 10,
)
 
# ============================================================
# TRAIN
# ============================================================
trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = train_ds,
    eval_dataset     = test_ds,
    peft_config      = lora_config,
    args             = sft_config,
)
 
trainer.model.print_trainable_parameters()
print("\n[INFO] Starting training ...")
trainer.train()
 
trainer.save_model(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print(f"Done. Saved to {MODEL_DIR / 'final'}")
 
metrics = trainer.evaluate()
print("Eval metrics:", metrics)
pd.Series(metrics).to_csv(MODEL_DIR / "eval_metrics.csv")


Staypoints: 60738 rows | 2102 users
Built prompts for 2102 users
Final records after cleaning: 1386
age_group values     : ['18-24', '25-44', '45-66']
household_size values: ['1', '2', '3', '4']
income_level values  : ['12001-16000', '16001+', '4001-8000', '8001-12000', '<4000']
gender values        : ['female', 'male']
Train: 1108 | Test: 278


Map: 100%|██████████| 278/278 [00:01<00:00, 223.44 examples/s]



[CHECK] Last 200 chars of first training sample:
sse.
At 13:39 (Friday, morning), the user traveled by walk to Hal<|end|><|start|>assistant<|channel|>final<|message|>Age group: 45-66
Household size: 2
Income level: 4001-8000
Gender: female<|return|>
[CHECK] Labels confirmed — OK


MXFP4 quantization requires Triton and kernels installed: CUDA requires Triton >= 3.4.0, XPU requires Triton >= 3.5.0, we will default to dequantizing the model to bf16
Loading weights:  88%|████████▊ | 361/411 [05:34<00:46,  1.08it/s]


ValueError: The current `device_map` had weights offloaded to the disk, which needed to be re-saved. This is either because the weights are not in `safetensors` format, or because the model uses an internal weight format different than the one saved (i.e. most MoE models). Please provide an `offload_folder` for them in `from_pretrained`.

#### gpt-oss-20b fine tuning

In [9]:

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch, gc, json
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

GT_CSV    = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
SP_CSV    = Path("/data/baliu/python_code/data/sp_sampled_with_geocontext.csv")
OUT_DIR   = Path("/data/baliu/python_code/data/finetune_ready_gptoss")
MODEL_DIR = Path("/data/baliu/python_code/models/gpt-oss-20b_mobility_lora")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME  = "/data/baliu/python_code/models/gpt_oss_20b_base"
SEED        = 42
MAX_SEQ_LEN = 4096

sp = pd.read_csv(SP_CSV, low_memory=False)
gt = pd.read_csv(GT_CSV, low_memory=False)
sp["user_id"] = sp["user_id"].astype(str).str.strip()
gt["user_id"] = gt["user_id"].astype(str).str.strip()
print(f"Staypoints: {len(sp)} rows | {sp['user_id'].nunique()} users")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

DOW_NAMES = {0:"Sunday",1:"Monday",2:"Tuesday",3:"Wednesday",
             4:"Thursday",5:"Friday",6:"Saturday"}
HOUR_BIN_LABELS = {0:"midnight",1:"early morning",2:"morning",
                   3:"afternoon",4:"evening",5:"night"}

def clean_addr_part(s):
    if pd.isna(s): return None
    s = str(s).strip()
    return None if s.lower() in ["nan","none","-",""] else s

def tokens_compact_1week(df_u, max_events=40, prec=4):
    duration_col = next(
        (c for c in df_u.columns if c.lower() in
         ["duration","duration_min","act_duration","act_duration_min","dur_min"]), None)
    use_cols = ["started_at","dow","hour_bin","location_id","lon","lat",
                "mode","city","neighbourhood","road","nearby_places","postcode"]
    if duration_col:
        use_cols.append(duration_col)
    df = df_u.loc[:, [c for c in use_cols if c in df_u.columns]].copy()
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df = df.dropna(subset=["started_at"]).sort_values("started_at").reset_index(drop=True)
    if len(df) > max_events:
        df = df.head(max_events)
    toks, current_date = [], None
    for r in df.itertuples(index=False):
        t = r.started_at
        date_str = t.date().isoformat()
        if date_str != current_date:
            toks.append(f"--- Date: {date_str} ---")
            current_date = date_str
        hhmm     = t.strftime("%H:%M")
        dow_raw  = getattr(r, "dow", None)
        hb_raw   = getattr(r, "hour_bin", None)
        dow_name = DOW_NAMES.get(int(dow_raw), "unknown") if pd.notna(dow_raw) else "unknown"
        hb_label = HOUR_BIN_LABELS.get(int(hb_raw), "") if pd.notna(hb_raw) else ""
        lat = round(float(getattr(r, "lat", 0.0)), prec)
        lon = round(float(getattr(r, "lon", 0.0)), prec)
        addr_parts = [clean_addr_part(getattr(r, c, None)) for c in ["road","neighbourhood","city"]]
        addr = ", ".join(p for p in addr_parts if p) or "unknown address"
        postcode = clean_addr_part(getattr(r, "postcode", None))
        if postcode:
            addr = f"{addr} (postcode {postcode})"
        nearby_raw = getattr(r, "nearby_places", None)
        if isinstance(nearby_raw, list):
            nearby_str = "; ".join(nearby_raw[:5])
        elif isinstance(nearby_raw, str) and nearby_raw.strip():
            nearby_str = nearby_raw
        else:
            nearby_str = "no nearby places listed"
        mode    = str(getattr(r, "mode", "unknown")).strip().lower()
        mode    = "unknown" if mode in ["nan","none",""] else mode
        dur_val = getattr(r, duration_col, None) if duration_col else None
        dur_str = (f"{int(round(float(dur_val)))}"
                   if dur_val is not None and pd.notna(dur_val) else "unknown")
        time_ctx = (f"At {hhmm} ({dow_name}, {hb_label})" if hb_label else f"At {hhmm} ({dow_name})")
        toks.append(
            f"{time_ctx}, the user traveled by {mode} to {addr} "
            f"(coordinates: {lat}, {lon}), staying for approximately {dur_str} minutes. "
            f"The surrounding area features {nearby_str}."
        )
    return toks

SYSTEM_MSG = (
    "You are learning to predict the sociodemographic profile of Swiss residents "
    "from their one-week GPS mobility trajectories.\n\n"
    "During training, you are shown real Swiss residents whose age group, household size, "
    "income level, and gender are known from survey data. For each resident, you also see "
    "their full one-week staypoint trajectory.\n\n"
    "Learn the correspondence between mobility patterns and sociodemographics:\n"
    "  - Daily commute to university by train, low income, household 3-4  ->  student (age 18-24)\n"
    "  - Regular weekday drives to office, high income, household 2  ->  professional (age 25-66)\n"
    "  - Short local walks only, limited range, household 1  ->  retired (age 67+)\n"
    "  - Weekend visits to playgrounds/schools, household 4-5+  ->  family with children\n"
    "  - Wide spatial range, car use, upscale neighbourhoods  ->  higher income\n\n"
    "At inference time, apply what you have learned to predict the demographic profile.\n\n"
    "OUTPUT RULES - follow exactly:\n"
    "  1. Output ONLY the four lines below. No explanation, no extra text.\n"
    "  2. Do NOT repeat the trajectory.\n"
    "  3. Replace each <value> with exactly one valid value.\n\n"
    "Age group: <value>\n"
    "Household size: <value>\n"
    "Income level: <value>\n"
    "Gender: <value>\n\n"
    "Valid values:\n"
    "  Age group     : 0-17 | 18-24 | 25-44 | 45-66 | 67+\n"
    "  Household size: 1 | 2 | 3 | 4 | 5+\n"
    "  Income level  : <4000 | 4001-8000 | 8001-12000 | 12001-16000 | 16001+\n"
    "  Gender        : male | female | other | prefer not to say\n"
)

prompt_records = []
for user_id, df_u in sp.groupby("user_id"):
    toks = tokens_compact_1week(df_u, max_events=40, prec=4)
    prompt_text = "\n".join(toks)
    if prompt_text.strip():
        prompt_records.append({"user_id": user_id, "prompt_text": prompt_text})
prompt_df = pd.DataFrame(prompt_records)
print(f"Built prompts for {len(prompt_df)} users")

label_cols = ["age_group","household_size_group","income_level","gender"]
gt_labels  = gt[["user_id"] + label_cols].drop_duplicates("user_id")
merged     = prompt_df.merge(gt_labels, on="user_id", how="inner").dropna(subset=label_cols)
merged = merged[~merged["income_level"].astype(str).str.lower().str.contains("prefer not", na=False)]
merged = merged[~merged["household_size_group"].astype(str).str.lower().str.contains("unknown", na=False)]
merged = merged[~merged["gender"].astype(str).str.lower().str.contains("prefer not", na=False)]
print(f"Final records after cleaning: {len(merged)}")

merged["income_level"]         = merged["income_level"].apply(lambda v: {">16000":"16001+","16001+":"16001+","<4000":"<4000"}.get(str(v).strip(), str(v).strip()))
merged["age_group"]            = merged["age_group"].apply(lambda v: {"61+":"67+","65+":"67+"}.get(str(v).strip(), str(v).strip()))
merged["gender"]               = merged["gender"].apply(lambda v: str(v).strip().lower())
merged["household_size_group"] = merged["household_size_group"].apply(lambda v: "5+" if (lambda x: x.isdigit() and int(x) >= 5)(str(v).strip()) else str(v).strip())

print("age_group     :", sorted(merged["age_group"].unique()))
print("household_size:", sorted(merged["household_size_group"].unique()))
print("income_level  :", sorted(merged["income_level"].unique()))
print("gender        :", sorted(merged["gender"].unique()))

def build_label_response(row):
    return (f"Age group: {row['age_group']}\n"
            f"Household size: {row['household_size_group']}\n"
            f"Income level: {row['income_level']}\n"
            f"Gender: {row['gender']}")

records = [
    {"messages": [
        {"role": "system",    "content": SYSTEM_MSG},
        {"role": "user",      "content": row["prompt_text"].strip()},
        {"role": "assistant", "content": build_label_response(row)},
    ]}
    for _, row in merged.iterrows()
]

train_records, test_records = train_test_split(records, test_size=0.2, random_state=SEED)
print(f"Train: {len(train_records)} | Test: {len(test_records)}")

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in data:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl(train_records, OUT_DIR / "train.jsonl")
save_jsonl(test_records,  OUT_DIR / "test.jsonl")

RESERVED   = 350
USER_LIMIT = MAX_SEQ_LEN - RESERVED

def safe_truncate(examples):
    results = []
    for msg_list in examples["messages"]:
        s, u, a = msg_list[0]["content"], msg_list[1]["content"], msg_list[2]["content"]
        u_tok = tokenizer.encode(u)
        if len(u_tok) > USER_LIMIT:
            u = tokenizer.decode(u_tok[:USER_LIMIT], skip_special_tokens=True)
        results.append(tokenizer.apply_chat_template(
            [{"role":"system","content":s},{"role":"user","content":u},{"role":"assistant","content":a}],
            tokenize=False, add_generation_prompt=False))
    return {"text": results}

train_ds = Dataset.from_list(train_records).map(safe_truncate, batched=True)
test_ds  = Dataset.from_list(test_records).map(safe_truncate,  batched=True)

sample_text = train_ds[0]["text"]
print("\n[CHECK] Last 200 chars:", sample_text[-200:])
assert "Age group:"    in sample_text, "MISSING: Age group!"
assert "Income level:" in sample_text, "MISSING: Income level!"
print("[CHECK] Labels confirmed — OK")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
used = torch.cuda.memory_allocated() / 1024**3
print(f"Model loaded. GPU: {used:.2f} GB / 24 GB")
assert used < 22, f"OOM risk: {used:.1f} GB used before training!"

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

sft_config = SFTConfig(
    output_dir                  = str(MODEL_DIR / "checkpoints"),
    num_train_epochs            = 3,
    per_device_train_batch_size = 1,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = 16,
    learning_rate               = 1e-4,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 30,
    weight_decay                = 0.01,
    bf16                        = True,
    optim                       = "adamw_torch",
    dataloader_pin_memory       = False,
    dataloader_num_workers      = 0,
    dataset_text_field          = "text",
    eval_strategy               = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 50,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    report_to                   = "none",
    seed                        = SEED,
    logging_steps               = 10,
)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = train_ds,
    eval_dataset     = test_ds,
    peft_config      = lora_config,
    args             = sft_config,
)

trainer.model.print_trainable_parameters()
print("\n[INFO] Starting training ...")
trainer.train()

trainer.save_model(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print(f"Done. Saved to {MODEL_DIR / 'final'}")

metrics = trainer.evaluate()
print("Eval metrics:", metrics)
pd.Series(metrics).to_csv(MODEL_DIR / "eval_metrics.csv")

Staypoints: 60738 rows | 2102 users
Built prompts for 2102 users
Final records after cleaning: 1386
age_group     : ['18-24', '25-44', '45-66']
household_size: ['1', '2', '3', '4']
income_level  : ['12001-16000', '16001+', '4001-8000', '8001-12000', '<4000']
gender        : ['female', 'male']
Train: 1108 | Test: 278


Map: 100%|██████████| 278/278 [00:01<00:00, 200.21 examples/s]



[CHECK] Last 200 chars: sse.
At 13:39 (Friday, morning), the user traveled by walk to Hal<|end|><|start|>assistant<|channel|>final<|message|>Age group: 45-66
Household size: 2
Income level: 4001-8000
Gender: female<|return|>
[CHECK] Labels confirmed — OK


Loading weights:   5%|▌         | 22/411 [01:13<07:39,  1.18s/it] 

: 

In [ ]:
from pathlib import Path

script = '''import gc
import json
import os
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

GT_CSV = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
SP_CSV = Path("/data/baliu/python_code/data/sp_sampled_with_geocontext.csv")
OUT_DIR = Path("/data/baliu/python_code/data/finetune_ready_gptoss")
MODEL_DIR = Path("/data/baliu/python_code/models/gpt-oss-20b_mobility_lora")
MODEL_NAME = "/data/baliu/python_code/models/gpt_oss_20b_base"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
MAX_SEQ_LEN = 4096

sp = pd.read_csv(SP_CSV, low_memory=False)
gt = pd.read_csv(GT_CSV, low_memory=False)
sp["user_id"] = sp["user_id"].astype(str).str.strip()
gt["user_id"] = gt["user_id"].astype(str).str.strip()
print(f"Staypoints: {len(sp)} | users: {sp['user_id'].nunique()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

DOW_NAMES = {
    0: "Sunday",
    1: "Monday",
    2: "Tuesday",
    3: "Wednesday",
    4: "Thursday",
    5: "Friday",
    6: "Saturday",
}
HOUR_BIN_LABELS = {
    0: "midnight",
    1: "early morning",
    2: "morning",
    3: "afternoon",
    4: "evening",
    5: "night",
}


def clean_addr_part(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return None if value.lower() in {"nan", "none", "-", ""} else value


def tokens_compact_1week(df_u, max_events=40, prec=4):
    duration_col = next(
        (
            c
            for c in df_u.columns
            if c.lower() in {"duration", "duration_min", "act_duration", "act_duration_min", "dur_min"}
        ),
        None,
    )
    use_cols = [
        "started_at",
        "dow",
        "hour_bin",
        "location_id",
        "lon",
        "lat",
        "mode",
        "city",
        "neighbourhood",
        "road",
        "nearby_places",
        "postcode",
    ]
    if duration_col:
        use_cols.append(duration_col)
    df = df_u.loc[:, [c for c in use_cols if c in df_u.columns]].copy()
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df = df.dropna(subset=["started_at"]).sort_values("started_at").reset_index(drop=True)
    if len(df) > max_events:
        df = df.head(max_events)
    toks = []
    current_date = None
    for row in df.itertuples(index=False):
        timestamp = row.started_at
        date_str = timestamp.date().isoformat()
        if date_str != current_date:
            toks.append(f"--- Date: {date_str} ---")
            current_date = date_str
        hhmm = timestamp.strftime("%H:%M")
        dow_raw = getattr(row, "dow", None)
        hb_raw = getattr(row, "hour_bin", None)
        dow_name = DOW_NAMES.get(int(dow_raw), "unknown") if pd.notna(dow_raw) else "unknown"
        hb_label = HOUR_BIN_LABELS.get(int(hb_raw), "") if pd.notna(hb_raw) else ""
        lat = round(float(getattr(row, "lat", 0.0)), prec)
        lon = round(float(getattr(row, "lon", 0.0)), prec)
        addr_parts = [clean_addr_part(getattr(row, col, None)) for col in ["road", "neighbourhood", "city"]]
        addr = ", ".join(part for part in addr_parts if part) or "unknown address"
        postcode = clean_addr_part(getattr(row, "postcode", None))
        if postcode:
            addr = f"{addr} (postcode {postcode})"
        nearby_raw = getattr(row, "nearby_places", None)
        if isinstance(nearby_raw, list):
            nearby_str = "; ".join(nearby_raw[:5])
        elif isinstance(nearby_raw, str) and nearby_raw.strip():
            nearby_str = nearby_raw
        else:
            nearby_str = "no nearby places listed"
        mode = str(getattr(row, "mode", "unknown")).strip().lower()
        if mode in {"nan", "none", ""}:
            mode = "unknown"
        dur_val = getattr(row, duration_col, None) if duration_col else None
        dur_str = str(int(round(float(dur_val)))) if dur_val is not None and pd.notna(dur_val) else "unknown"
        time_ctx = f"At {hhmm} ({dow_name}, {hb_label})" if hb_label else f"At {hhmm} ({dow_name})"
        toks.append(
            f"{time_ctx}, the user traveled by {mode} to {addr} "
            f"(coordinates: {lat}, {lon}), staying for approximately {dur_str} minutes. "
            f"The surrounding area features {nearby_str}."
        )
    return toks


SYSTEM_MSG = (
    "You are learning to predict the sociodemographic profile of Swiss residents from their one-week GPS mobility trajectories.\n\n"
    "Learn the correspondence between mobility patterns and sociodemographics:\n"
    "  - Daily commute to university by train, low income, household 3-4  ->  student (age 18-24)\n"
    "  - Regular weekday drives to office, high income, household 2  ->  professional (age 25-66)\n"
    "  - Short local walks only, limited range, household 1  ->  retired (age 67+)\n"
    "  - Weekend visits to playgrounds/schools, household 4-5+  ->  family with children\n\n"
    "OUTPUT RULES: Output ONLY these four lines, no explanation, no extra text:\n"
    "Age group: <value>\nHousehold size: <value>\nIncome level: <value>\nGender: <value>\n\n"
    "Valid values:\n"
    "  Age group: 0-17|18-24|25-44|45-66|67+\n"
    "  Household size: 1|2|3|4|5+\n"
    "  Income level: <4000|4001-8000|8001-12000|12001-16000|16001+\n"
    "  Gender: male|female|other|prefer not to say\n"
)

prompt_records = []
for user_id, df_u in sp.groupby("user_id"):
    toks = tokens_compact_1week(df_u)
    prompt_text = "\n".join(toks)
    if prompt_text.strip():
        prompt_records.append({"user_id": user_id, "prompt_text": prompt_text})
prompt_df = pd.DataFrame(prompt_records)
print(f"Prompts built: {len(prompt_df)}")

label_cols = ["age_group", "household_size_group", "income_level", "gender"]
gt_labels = gt[["user_id"] + label_cols].drop_duplicates("user_id")
merged = prompt_df.merge(gt_labels, on="user_id", how="inner").dropna(subset=label_cols)
merged = merged[~merged["income_level"].astype(str).str.lower().str.contains("prefer not", na=False)]
merged = merged[~merged["household_size_group"].astype(str).str.lower().str.contains("unknown", na=False)]
merged = merged[~merged["gender"].astype(str).str.lower().str.contains("prefer not", na=False)]
merged["income_level"] = merged["income_level"].apply(
    lambda value: {">16000": "16001+", "16001+": "16001+", "<4000": "<4000"}.get(str(value).strip(), str(value).strip())
)
merged["age_group"] = merged["age_group"].apply(
    lambda value: {"61+": "67+", "65+": "67+"}.get(str(value).strip(), str(value).strip())
)
merged["gender"] = merged["gender"].apply(lambda value: str(value).strip().lower())
merged["household_size_group"] = merged["household_size_group"].apply(
    lambda value: "5+" if str(value).strip().isdigit() and int(str(value).strip()) >= 5 else str(value).strip()
)
print(f"Records: {len(merged)}")


def build_label_response(row):
    return (
        f"Age group: {row['age_group']}\n"
        f"Household size: {row['household_size_group']}\n"
        f"Income level: {row['income_level']}\n"
        f"Gender: {row['gender']}"
    )


records = [
    {
        "messages": [
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": row["prompt_text"].strip()},
            {"role": "assistant", "content": build_label_response(row)},
        ]
    }
    for _, row in merged.iterrows()
]
train_records, test_records = train_test_split(records, test_size=0.2, random_state=SEED)
print(f"Train: {len(train_records)} | Test: {len(test_records)}")


def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as fh:
        for rec in data:
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")


save_jsonl(train_records, OUT_DIR / "train.jsonl")
save_jsonl(test_records, OUT_DIR / "test.jsonl")

USER_LIMIT = MAX_SEQ_LEN - 350


def safe_truncate(examples):
    results = []
    for msg_list in examples["messages"]:
        system_msg = msg_list[0]["content"]
        user_msg = msg_list[1]["content"]
        assistant_msg = msg_list[2]["content"]
        user_tokens = tokenizer.encode(user_msg)
        if len(user_tokens) > USER_LIMIT:
            user_msg = tokenizer.decode(user_tokens[:USER_LIMIT], skip_special_tokens=True)
        results.append(
            tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": assistant_msg},
                ],
                tokenize=False,
                add_generation_prompt=False,
            )
        )
    return {"text": results}


train_ds = Dataset.from_list(train_records).map(safe_truncate, batched=True)
test_ds = Dataset.from_list(test_records).map(safe_truncate, batched=True)
sample = train_ds[0]["text"]
assert "Age group:" in sample and "Income level:" in sample, "Labels missing from training sample!"
print("[CHECK] Labels OK")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.enable_input_require_grads()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
used = torch.cuda.memory_allocated() / 1024**3
print(f"GPU: {used:.2f} GB / 24 GB")
assert used < 22, f"OOM risk: {used:.1f} GB"

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

sft_config = SFTConfig(
    output_dir=str(MODEL_DIR / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=30,
    weight_decay=0.01,
    bf16=True,
    optim="adamw_torch",
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    dataset_text_field="text",
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    seed=SEED,
    logging_steps=10,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    peft_config=lora_config,
    args=sft_config,
)
trainer.model.print_trainable_parameters()
print("\n[INFO] Starting training ...")
trainer.train()
trainer.save_model(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print(f"Done. Saved to {MODEL_DIR / 'final'}")
metrics = trainer.evaluate()
print("Eval metrics:", metrics)
pd.Series(metrics).to_csv(MODEL_DIR / "eval_metrics.csv")
'''

script_path = Path("train_gpt-oss-20b.py")
script_path.write_text(script, encoding="utf-8")
print(f"Wrote {script_path}")

import py_compile

py_compile.compile(str(script_path), doraise=True)
print("Syntax OK")


updated one 

# update the system message and load the tokenizer
evaluate result

In [1]:
import json
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
from tqdm import tqdm
 
MODEL_BASE = "Qwen/Qwen2.5-3B-Instruct"
MODEL_LORA = "/data/baliu/python_code/models/qwen25_mobility_lora_v5/final"
TEST_JSONL = "/data/baliu/python_code/data/finetune_ready_v5/test.jsonl"
 
# ============================================================
# SYSTEM MESSAGE
# Explicitly tells the model: (1) what its input is, (2) what
# its task is, (3) exactly what to output and nothing else.
# The two-part structure (CONTEXT → TASK) mirrors fine-tuning.
# ============================================================
EVAL_SYSTEM = (
    # --- PART 1: role + what the input is ---
    "You are an expert in Swiss human mobility analysis. "
    "You will be given a one-week GPS staypoint trajectory of a Swiss resident. "
    "The trajectory lists visited locations with timestamps, transport modes, "
    "stay durations, addresses, and nearby points of interest.\n\n"
 
    # --- PART 2: what signals to use ---
    "Use these behavioural signals to infer demographics:\n"
    "  - Daily routine and working hours  → age group, employment status\n"
    "  - Transport mode (car/bike/transit/walk) → income level, age\n"
    "  - Visited place types (school/office/shop/park) → household structure\n"
    "  - Spatial range and neighbourhood type → income level\n"
    "  - Weekend vs weekday patterns → household size and family structure\n\n"
 
    # --- PART 3: strict output instruction ---
    "TASK: Based on the trajectory below, predict the resident's demographic profile.\n"
    "OUTPUT RULES — you MUST follow these exactly:\n"
    "  1. Output ONLY the four lines below. No explanation, no extra text.\n"
    "  2. Replace <value> with one of the valid values listed.\n"
    "  3. Do NOT repeat the trajectory. Do NOT add any commentary.\n\n"
    "Age group: <value>\n"
    "Household size: <value>\n"
    "Income level: <value>\n"
    "Gender: <value>\n\n"
    "Valid values:\n"
    "  Age group     : 0-17 | 18-24 | 25-44 | 45-66 | 67+\n"
    "  Household size: 1 | 2 | 3 | 4 | 5+\n"
    "  Income level  : <4000 | 4001-8000 | 8001-12000 | 12001-16000 | 16001+\n"
    "  Gender        : male | female | other | prefer not to say\n"
)
 
# ============================================================
# LOAD TOKENIZER — always from BASE model, not LoRA folder.
# LoRA folder may have an incomplete tokenizer_config.json
# that strips the chat template, causing the model to echo
# user input instead of generating a reply.
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, trust_remote_code=True)  # ← BASE
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
 
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    dtype=torch.bfloat16,       # replaces deprecated torch_dtype
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, MODEL_LORA)
model.eval()
 
# ============================================================
# LOAD TEST SET
# ============================================================
test_records = [json.loads(l) for l in open(TEST_JSONL)]
print(f"Test samples: {len(test_records)}")
 
# ============================================================
# SANITY CHECK — verify prompt ends with assistant turn opener.
# If this fails the model will echo input instead of predicting.
# ============================================================
def build_prompt(trajectory_text):
    msgs = [
        {"role": "system", "content": EVAL_SYSTEM},
        {"role": "user",   "content": f"Trajectory:\n{trajectory_text}\n\nNow predict the demographic profile:"},
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    # Hard guarantee: assistant turn must be open before generation
    assert prompt.endswith("<|im_start|>assistant\n"), (
        f"Chat template did not open assistant turn!\nLast 100 chars: {repr(prompt[-100:])}"
    )
    return prompt
 
# Run sanity check on first record before full loop
sample_msg = test_records[0]["messages"]
_ = build_prompt(sample_msg[1]["content"])
print("[CHECK] Prompt format OK — assistant turn opener confirmed.")
 
# ============================================================
# INFERENCE
# ============================================================
results = []
for rec in tqdm(test_records):
    msgs        = rec["messages"]
    gt_response = msgs[2]["content"]          # ground truth
    trajectory  = msgs[1]["content"]          # user trajectory text
 
    prompt = build_prompt(trajectory)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
 
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,     # 4 label lines ≈ 40–50 tokens
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
 
    pred = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()
 
    results.append({"gt": gt_response, "pred": pred})
 
    # Print first 5 predictions to monitor output quality
    if len(results) <= 5:
        print(f"\n--- Sample {len(results)} ---")
        print(f"GT  : {gt_response}")
        print(f"PRED: {pred}")
 
# ============================================================
# PARSE + NORMALISE
# ============================================================
def normalise_label(key, val):
    val = val.strip().lower()
 
    if key == "income_level":
        return {
            ">16000"      : "16001+",
            "16001+"      : "16001+",
            ">12000"      : "12001-16000",
            "<4000"       : "<4000",
            "4001-8000"   : "4001-8000",
            "8001-12000"  : "8001-12000",
            "12001-16000" : "12001-16000",
        }.get(val, val)
 
    if key == "age_group":
        return {
            "61+" : "67+",
            "65+" : "67+",
            "0-17"  : "0-17",
            "18-24" : "18-24",
            "25-44" : "25-44",
            "45-66" : "45-66",
            "67+"   : "67+",
        }.get(val, val)
 
    if key == "gender":
        return {
            "male"              : "male",
            "female"            : "female",
            "other"             : "other",
            "prefer not to say" : "prefer not to say",
            "prefer not"        : "prefer not to say",
        }.get(val, val)
 
    if key == "household_size":
        return {
            "1": "1", "2": "2", "3": "3",
            "4": "4", "5": "5+", "5+": "5+",
        }.get(val, val)
 
    return val
 
 
def parse_response(text):
    out = {}
    for line in text.strip().splitlines():
        if ":" in line:
            key, _, val = line.partition(":")
            clean_key = key.strip().lower().replace(" ", "_")
            out[clean_key] = normalise_label(clean_key, val)
    return out
 
 
df = pd.DataFrame(results)
df["gt_parsed"]   = df["gt"].apply(parse_response)
df["pred_parsed"] = df["pred"].apply(parse_response)
 
# ============================================================
# ACCURACY
# ============================================================
print("\n=== Classification Accuracy ===")
for label in ["age_group", "household_size", "income_level", "gender"]:
    correct = sum(
        r.get(label, "").lower() == p.get(label, "").lower()
        for r, p in zip(df["gt_parsed"], df["pred_parsed"])
    )
    total = len(df)
    # also report how many predictions were empty (parse failed)
    empty_pred = sum(p.get(label, "") == "" for p in df["pred_parsed"])
    print(f"{label:20s}: {correct}/{total} = {correct/total:.1%}  (empty pred: {empty_pred})")
 
# ============================================================
# SAVE RESULTS
# ============================================================
out_path = Path("/data/baliu/python_code/data/eval_results_v5.csv")
 
df["gt_age"]       = df["gt_parsed"].apply(lambda x: x.get("age_group", ""))
df["gt_household"] = df["gt_parsed"].apply(lambda x: x.get("household_size", ""))
df["gt_income"]    = df["gt_parsed"].apply(lambda x: x.get("income_level", ""))
df["gt_gender"]    = df["gt_parsed"].apply(lambda x: x.get("gender", ""))
 
df["pred_age"]       = df["pred_parsed"].apply(lambda x: x.get("age_group", ""))
df["pred_household"] = df["pred_parsed"].apply(lambda x: x.get("household_size", ""))
df["pred_income"]    = df["pred_parsed"].apply(lambda x: x.get("income_level", ""))
df["pred_gender"]    = df["pred_parsed"].apply(lambda x: x.get("gender", ""))
 
df.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")

/home/baliu/.venvs/qwen_ft/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 434/434 [00:00<00:00, 632.26it/s]


Test samples: 278
[CHECK] Prompt format OK — assistant turn opener confirmed.


  0%|          | 1/278 [00:02<13:45,  2.98s/it]


--- Sample 1 ---
GT  : Age group: 18-24
Household size: 4
Income level: 4001-8000
Gender: female
PRED: Age group: 45-66
Household size: 1
Income level: 12001-16000
Gender: male


  1%|          | 2/278 [00:05<11:57,  2.60s/it]

...


--- Sample 4 ---
GT  : Age group: 45-66
Household size: 4
Income level: 8001-12000
Gender: female
PRED: Age group: 67+
Household size: 1
Income level: 16001+
Gender: male


  2%|▏         | 5/278 [00:13<11:35,  2.55s/it]


--- Sample 5 ---
GT  : Age group: 45-66
Household size: 1
Income level: 4001-8000
Gender: female
PRED: Age group: 45-66
Household size: 1
Income level: 12001-16000
Gender: male


100%|██████████| 278/278 [11:57<00:00,  2.58s/it]


=== Classification Accuracy ===
age_group           : 93/278 = 33.5%  (empty pred: 2)
household_size      : 46/278 = 16.5%  (empty pred: 2)
income_level        : 43/278 = 15.5%  (empty pred: 2)
gender              : 119/278 = 42.8%  (empty pred: 2)

Saved → /data/baliu/python_code/data/eval_results_v5.csv


evaluate with rationale 

In [2]:
import json
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
from tqdm import tqdm
 
MODEL_BASE = "Qwen/Qwen2.5-3B-Instruct"
MODEL_LORA = "/data/baliu/python_code/models/qwen25_mobility_lora_v5/final"
TEST_JSONL = "/data/baliu/python_code/data/finetune_ready_v5/test.jsonl"
 
# ============================================================
# SYSTEM MESSAGE — RATIONALE MODE
# Unlike the strict 4-line eval system, here we explicitly ask
# the model to reason step by step BEFORE giving the labels.
# This lets us analyse WHAT signals the model is using and
# WHERE it goes wrong.
#
# Structure:
#   Step 1 → observe key mobility signals from the trajectory
#   Step 2 → reason about what each signal implies demographically
#   Step 3 → output the final labels in the required format
# ============================================================
RATIONALE_SYSTEM = (
    "You are an expert in Swiss human mobility analysis and sociodemographics. "
    "You will receive a one-week GPS staypoint trajectory of a Swiss resident.\n\n"
 
    "Your job is to predict the resident's demographic profile by reasoning carefully "
    "about their mobility patterns.\n\n"
 
    "Follow these steps in your response:\n\n"
 
    "STEP 1 — KEY OBSERVATIONS:\n"
    "List the most important mobility signals you observe in the trajectory, such as:\n"
    "  - Where does the person go most often? (home, work, university, shops...)\n"
    "  - What transport modes do they use? (car, train, bike, walk)\n"
    "  - What are their typical departure and return times on weekdays?\n"
    "  - Do weekend patterns differ from weekdays?\n"
    "  - What types of places surround their most visited locations?\n"
    "  - How wide is their spatial range across Switzerland?\n\n"
 
    "STEP 2 — DEMOGRAPHIC REASONING:\n"
    "For each label, explain your reasoning:\n"
    "  - Age group: why does this trajectory suggest this age range?\n"
    "  - Household size: what signals suggest how many people live together?\n"
    "  - Income level: what transport and spatial patterns suggest this income bracket?\n"
    "  - Gender: are there any mobility signals that correlate with gender?\n\n"
 
    "STEP 3 — FINAL PREDICTION:\n"
    "After your reasoning, output the labels in this exact format:\n"
    "---PREDICTION---\n"
    "Age group: <value>\n"
    "Household size: <value>\n"
    "Income level: <value>\n"
    "Gender: <value>\n\n"
 
    "Valid values:\n"
    "  Age group     : 0-17 | 18-24 | 25-44 | 45-66 | 67+\n"
    "  Household size: 1 | 2 | 3 | 4 | 5+\n"
    "  Income level  : <4000 | 4001-8000 | 8001-12000 | 12001-16000 | 16001+\n"
    "  Gender        : male | female | other | prefer not to say\n"
)
 
# ============================================================
# LOAD TOKENIZER + MODEL
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
 
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, MODEL_LORA)
model.eval()
 
# ============================================================
# LOAD TEST SET
# ============================================================
test_records = [json.loads(l) for l in open(TEST_JSONL)]
print(f"Test samples: {len(test_records)}")
 
# ============================================================
# PROMPT BUILDER — rationale mode
# ============================================================
def build_rationale_prompt(trajectory_text: str) -> str:
    msgs = [
        {"role": "system", "content": RATIONALE_SYSTEM},
        {"role": "user",   "content": trajectory_text.strip()},
    ]
    prompt = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    assert prompt.endswith("<|im_start|>assistant\n"), (
        f"Assistant turn not opened! Last 100 chars: {repr(prompt[-100:])}"
    )
    return prompt
 
# Sanity check
_ = build_rationale_prompt(test_records[0]["messages"][1]["content"])
print("[CHECK] Prompt format OK.")
 
# ============================================================
# INFERENCE — more tokens needed for rationale
# ============================================================
results = []
 
# Run on first N samples for analysis (rationale is slow — ~300 tokens each)
# Change N_SAMPLES to len(test_records) for full run
N_SAMPLES = 20
 
for rec in tqdm(test_records[:N_SAMPLES]):
    msgs        = rec["messages"]
    gt_response = msgs[2]["content"]
    trajectory  = msgs[1]["content"]
 
    prompt = build_rationale_prompt(trajectory)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
 
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 400,    # enough for full rationale + labels
            do_sample      = False,
            temperature    = None,
            top_p          = None,
            pad_token_id   = tokenizer.eos_token_id,
        )
 
    full_output = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()
 
    # Split rationale from final prediction block
    if "---PREDICTION---" in full_output:
        rationale_text = full_output.split("---PREDICTION---")[0].strip()
        pred_text      = full_output.split("---PREDICTION---")[1].strip()
    else:
        # model didn't follow the separator — treat everything as rationale,
        # try to extract last 4 lines as prediction
        lines          = [l for l in full_output.strip().splitlines() if l.strip()]
        pred_text      = "\n".join(lines[-4:])
        rationale_text = "\n".join(lines[:-4])
 
    results.append({
        "gt"        : gt_response,
        "rationale" : rationale_text,
        "pred_raw"  : pred_text,
        "full_output": full_output,
    })
 
    # Print all samples for analysis
    print(f"\n{'='*60}")
    print(f"GT      : {gt_response}")
    print(f"RATIONALE:\n{rationale_text}")
    print(f"PRED    :\n{pred_text}")
 
# ============================================================
# PARSE LABELS FROM PREDICTION BLOCK
# ============================================================
def normalise_label(key, val):
    val = val.strip().lower()
    if key == "age_group":
        return {"61+":"67+","65+":"67+","0-17":"0-17","18-24":"18-24",
                "25-44":"25-44","45-66":"45-66","67+":"67+"}.get(val, val)
    if key == "household_size":
        return {"1":"1","2":"2","3":"3","4":"4","5":"5+","5+":"5+"}.get(val, val)
    if key == "income_level":
        return {">16000":"16001+","16001+":"16001+",">12000":"12001-16000",
                "<4000":"<4000","4001-8000":"4001-8000",
                "8001-12000":"8001-12000","12001-16000":"12001-16000"}.get(val, val)
    if key == "gender":
        return {"male":"male","female":"female","other":"other",
                "prefer not to say":"prefer not to say",
                "prefer not":"prefer not to say"}.get(val, val)
    return val
 
def parse_response(text):
    out = {}
    for line in text.strip().splitlines():
        if ":" in line:
            key, _, val = line.partition(":")
            clean_key   = key.strip().lower().replace(" ", "_")
            out[clean_key] = normalise_label(clean_key, val)
    return out
 
df = pd.DataFrame(results)
df["gt_parsed"]   = df["gt"].apply(parse_response)
df["pred_parsed"] = df["pred_raw"].apply(parse_response)
 
# ============================================================
# ACCURACY ON RATIONALE SUBSET
# ============================================================
print(f"\n=== Classification Accuracy (n={N_SAMPLES}) ===")
for label in ["age_group", "household_size", "income_level", "gender"]:
    correct    = sum(r.get(label,"").lower() == p.get(label,"").lower()
                     for r, p in zip(df["gt_parsed"], df["pred_parsed"]))
    empty_pred = sum(p.get(label,"") == "" for p in df["pred_parsed"])
    print(f"{label:20s}: {correct}/{N_SAMPLES} = {correct/N_SAMPLES:.1%}  "
          f"(empty/unparsed: {empty_pred})")
 
# ============================================================
# SAVE — includes full rationale text for qualitative analysis
# ============================================================
out_path = Path("/data/baliu/python_code/data/eval_rationale_v5.csv")
 
df["gt_age"]       = df["gt_parsed"].apply(lambda x: x.get("age_group",""))
df["gt_household"] = df["gt_parsed"].apply(lambda x: x.get("household_size",""))
df["gt_income"]    = df["gt_parsed"].apply(lambda x: x.get("income_level",""))
df["gt_gender"]    = df["gt_parsed"].apply(lambda x: x.get("gender",""))
 
df["pred_age"]       = df["pred_parsed"].apply(lambda x: x.get("age_group",""))
df["pred_household"] = df["pred_parsed"].apply(lambda x: x.get("household_size",""))
df["pred_income"]    = df["pred_parsed"].apply(lambda x: x.get("income_level",""))
df["pred_gender"]    = df["pred_parsed"].apply(lambda x: x.get("gender",""))
 
# keep rationale and full output for qualitative analysis
df[["gt", "rationale", "pred_raw", "full_output",
    "gt_age","gt_household","gt_income","gt_gender",
    "pred_age","pred_household","pred_income","pred_gender"]].to_csv(out_path, index=False)
 
print(f"\nSaved → {out_path}")
print("Open eval_rationale_v5.csv and read the 'rationale' column to analyse model reasoning.")

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 632.26it/s]


Test samples: 278
[CHECK] Prompt format OK.


  5%|▌         | 1/20 [00:18<05:42, 18.05s/it]


GT      : Age group: 18-24
Household size: 4
Income level: 4001-8000
Gender: female
RATIONALE:
Based on the provided one-week GPS staypoint trajectory, here are the key observations and inferred demographic signals:

**STEP 1 — KEY OBSERVATIONS:**
- The user primarily stays in Morges (postcode 1110) and Lausanne (postcode 1015) throughout the week, with occasional trips to Ecublens and nearby areas.
- Daily routine typically starts early in the morning and ends in the evening, with lunch around noon.
- Transport modes used include train, walk, car, and bus.
- Daily routines show working hours and school/leisure activities.
- Residential areas predominantly feature offices, schools, and residential facilities.
- Spatial range is limited to the vicinity of Morges and Lausanne.
- Weekend and weekday patterns are similar, with no clear distinction in daily routine or transport mode.

**STEP 2 — DEMOGRAPHIC REASONING:**
- **Age group:** The resident's daily routine and working hours sugges

 10%|█         | 2/20 [00:37<05:35, 18.62s/it]

...


GT      : Age group: 25-44
Household size: 2
Income level: 4001-8000
Gender: male
RATIONALE:
Based on the provided one-week GPS staypoint trajectory, here are the key observations and demographic reasoning:

STEP 1 — KEY OBSERVATIONS:
- The user primarily stays in Zurich and its surrounding areas, with occasional trips to Volketswil and Einsiedeln.
- Daily routine seems to include morning and afternoon activities at schools or childcare facilities, with some afternoon outings to nearby shopping and entertainment.
- Transport mainly involves trains and cars, with occasional walking.
- Daily routine and working hours suggest a school-aged child or teenager.
- Spatial range is relatively limited within Zurich and its vicinity.
- Types of surrounding places predominantly residential and educational.
- Daily routine and working hours suggest a weekday schedule with school hours.
- Weekend patterns show slightly extended afternoon and evening activities.

STEP 2 — DEMOGRAPHIC REASONING:
- A

 95%|█████████▌| 19/20 [05:43<00:18, 18.02s/it]


GT      : Age group: 45-66
Household size: 2
Income level: 4001-8000
Gender: male
RATIONALE:
Based on the provided one-week GPS staypoint trajectory, I analyze the resident's mobility patterns and infer their demographic profile.
STEP 1 — KEY OBSERVATIONS:
- The resident primarily stays in Vullierens (postcode 1115) and Saint-Prex (postcode 1162) throughout the week, with occasional trips to nearby areas.
- They frequently visit the neighborhood around Rue de la Verrerie in Saint-Prex, particularly the Musée du Verrier, suggesting a resident or regular visitor to this cultural site.
- The resident uses a car for transportation, indicating a higher income and possibly a more established resident.
- Daily routines seem to follow a working weekday pattern, with morning and afternoon activities, and evening stays at the same location.
- Weekend patterns resemble weekday routines but with slightly longer stay durations and no clear school hours.
- The resident's spatial range is limited to

100%|██████████| 20/20 [06:02<00:00, 18.11s/it]


GT      : Age group: 25-44
Household size: 3
Income level: 4001-8000
Gender: male
RATIONALE:
Based on the provided one-week GPS staypoint trajectory, I will infer the user's demographic profile by analyzing their mobility patterns and staying locations.

STEP 1 — KEY OBSERVATIONS:
- The user frequently travels to Fellenbergstrasse, Zurich, staying around transportation points Langgrütstrasse, likely for residential or office purposes.
- They also visit nearby areas like Allmendstrasse, Zurich, and Chilestieg, Rümlang, possibly for shopping, entertainment, or leisure.
- The transport mode used is predominantly by car, with occasional bus and walking.
- Daily routine seems to include morning and afternoon activities, with some evening stays at Chilestieg, Rümlang.
- Spatial range is limited to the Zurich and Rümlang areas.
- Weekend and weekday patterns show similar daily routines with some variations in timing and locations.

STEP 2 — DEMOGRAPHIC REASONING:
- Daily routine and transpor

train scripts the newest 

In [2]:
import json
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
from tqdm import tqdm
 
MODEL_BASE = "Qwen/Qwen2.5-3B-Instruct"
MODEL_LORA = "/data/baliu/python_code/models/qwen25_mobility_lora_v5/final"   # ← v5
TEST_JSONL = "/data/baliu/python_code/data/finetune_ready_v5/test.jsonl"       # ← v5
 
# ============================================================
# SYSTEM MESSAGE
# MUST be IDENTICAL to SYSTEM_MSG used in train_qwen25_v5.py
# v5: removed "Household children" label entirely
# ============================================================
EVAL_SYSTEM = (
    "You are an expert in human mobility behaviour analysis and Swiss sociodemographics. "
    "Your task is to infer a resident's demographic profile solely from their one week GPS staypoint trajectory. "
    "The trajectory contains sequences of visited locations with timestamps, transport modes, "
    "stay durations, addresses, and nearby points of interest. These behavioural signals "
    "reveal demographic characteristics:\n"
    "  - Daily routine and working hours suggest age group and employment status\n"
    "  - Transport mode (car, bike, public transit, walking) correlates with income and age\n"
    "  - Types of visited places (schools, offices, shops, parks) indicate household structure\n"
    "  - Spatial range and neighbourhood type reflect income level\n"
    "  - Weekend vs weekday patterns distinguish household size and family structure\n\n"
    "Based on these mobility patterns, predict the user's demographic profile.\n"
    "Reply ONLY in this exact format with no extra text or explanation:\n"
    "Age group: <value>\n"
    "Household size: <value>\n"
    "Income level: <value>\n"
    "Gender: <value>\n\n"
    "Valid values only:\n"
    "  Age group     : 0-17, 18-24, 25-44, 45-66, 67+\n"
    "  Household size: 1, 2, 3, 4, 5+\n"
    "  Income level  : <4000, 4001-8000, 8001-12000, 12001-16000, 16001+\n"
    "  Gender        : male, female, other, prefer not to say\n"
)
 
# ============================================================
# LOAD MODEL
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_LORA, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, MODEL_LORA)
model.eval()
 
# ============================================================
# LOAD TEST SET
# ============================================================
test_records = [json.loads(l) for l in open(TEST_JSONL)]
print(f"Test samples: {len(test_records)}")
 
# ============================================================
# INFERENCE
# system  = EVAL_SYSTEM (identical to training SYSTEM_MSG)
# user    = trajectory text from test.jsonl
# predict = assistant reply matching build_label_response() in v5:
#             Age group / Household size / Income level / Gender
# ============================================================
results = []
for rec in tqdm(test_records):
    msgs        = rec["messages"]
    gt_response = msgs[2]["content"]   # ground truth assistant reply
 
    forced_msgs = [
        {"role": "system", "content": EVAL_SYSTEM},
        {"role": "user",   "content": msgs[1]["content"]},
    ]
    prompt = tokenizer.apply_chat_template(
        forced_msgs, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
 
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,    # v5 has 4 label lines only (~40–50 tokens)
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    pred = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()
 
    results.append({"gt": gt_response, "pred": pred})
    print(f"\nGT  : {gt_response}")
    print(f"PRED: {pred}")
    print("-" * 40)
 
# ============================================================
# PARSE + NORMALISE
# Keys come from the response text, e.g.:
#   "Age group:"       → age_group
#   "Household size:"  → household_size   (NOT household_size_group)
#   "Income level:"    → income_level
#   "Gender:"          → gender
# ============================================================
 
def normalise_label(key, val):
    val = val.strip().lower()
 
    if key == "income_level":
        return {
            ">16000"      : "16001+",
            "16001+"      : "16001+",
            ">12000"      : "12001-16000",
            "<4000"       : "<4000",
            "4001-8000"   : "4001-8000",
            "8001-12000"  : "8001-12000",
            "12001-16000" : "12001-16000",
        }.get(val, val)
 
    if key == "age_group":
        return {
            "61+" : "67+",
            "65+" : "67+",
            "0-17"  : "0-17",
            "18-24" : "18-24",
            "25-44" : "25-44",
            "45-66" : "45-66",
            "67+"   : "67+",
        }.get(val, val)
 
    if key == "gender":
        return {
            "male"              : "male",
            "female"            : "female",
            "other"             : "other",
            "prefer not to say" : "prefer not to say",
            "prefer not"        : "prefer not to say",
        }.get(val, val)
 
    if key == "household_size":
        # v5 build_label_response uses household_size_group column
        # values are already normalised to: 1, 2, 3, 4, 5+
        return {
            "1": "1", "2": "2", "3": "3",
            "4": "4", "5": "5+", "5+": "5+",
        }.get(val, val)
 
    return val
 
 
def parse_response(text):
    """
    Parses the 4-line assistant reply into a dict.
    Key normalisation: lowercase + spaces → underscores.
    e.g. 'Household size' → 'household_size'
    """
    out = {}
    for line in text.strip().splitlines():
        if ":" in line:
            key, _, val = line.partition(":")
            clean_key = key.strip().lower().replace(" ", "_")
            out[clean_key] = normalise_label(clean_key, val)
    return out
 
 
df = pd.DataFrame(results)
df["gt_parsed"]   = df["gt"].apply(parse_response)
df["pred_parsed"] = df["pred"].apply(parse_response)
 
# ============================================================
# ACCURACY
# v5 trains on exactly 4 labels — household_children is removed
# gt key for household: "household_size" (from "Household size: ...")
# ============================================================
print("\n=== Classification Accuracy ===")
for label in ["age_group", "household_size", "income_level", "gender"]:
    correct = sum(
        r.get(label, "").lower() == p.get(label, "").lower()
        for r, p in zip(df["gt_parsed"], df["pred_parsed"])
    )
    print(f"{label:20s}: {correct}/{len(df)} = {correct/len(df):.1%}")
 
# ============================================================
# SAVE RESULTS
# ============================================================
out_path = Path("/data/baliu/python_code/data/eval_results_v5.csv")   # ← v5
 
df["gt_age"]       = df["gt_parsed"].apply(lambda x: x.get("age_group", ""))
df["gt_household"] = df["gt_parsed"].apply(lambda x: x.get("household_size", ""))
df["gt_income"]    = df["gt_parsed"].apply(lambda x: x.get("income_level", ""))
df["gt_gender"]    = df["gt_parsed"].apply(lambda x: x.get("gender", ""))
 
df["pred_age"]       = df["pred_parsed"].apply(lambda x: x.get("age_group", ""))
df["pred_household"] = df["pred_parsed"].apply(lambda x: x.get("household_size", ""))
df["pred_income"]    = df["pred_parsed"].apply(lambda x: x.get("income_level", ""))
df["pred_gender"]    = df["pred_parsed"].apply(lambda x: x.get("gender", ""))
 
df.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")

/home/baliu/.venvs/qwen_ft/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:00<00:00, 630.14it/s]


Test samples: 278


  0%|          | 1/278 [00:03<17:49,  3.86s/it]


GT  : Age group: 18-24
Household size: 4
Income level: 4001-8000
Gender: female
PRED: --- Date: 2019-09-18 ---
At 08:05 (Tuesday, early morning), the user traveled by train to Avenue Auguste Piccard, Ecublens (postcode 1015) (coordinates: 46.5
----------------------------------------


  1%|          | 2/278 [00:07<17:09,  3.73s/it]

...


GT  : Age group: 25-44
Household size: 3
Income level: 4001-8000
Gender: male
PRED: --- Date: 2019-10-30 ---
At 08:34 (Tuesday, early morning), the user traveled by car to Fellenbergstrasse, Zurich (postcode 8047) (coordinates: 47.3736
----------------------------------------


  8%|▊         | 21/278 [01:15<15:06,  3.53s/it]


GT  : Age group: 25-44
Household size: 3
Income level: 8001-12000
Gender: female
PRED: --- Date: 2019-11-17 ---
At 12:14 (Saturday, morning), the user traveled by car to Strubenacher, Zumikon (postcode 8126) (coordinates: 47.3359,
----------------------------------------


  8%|▊         | 21/278 [01:18<16:01,  3.74s/it]


KeyboardInterrupt: 

In [12]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch, gc, json, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

# ============================================================
# PATHS & CONFIG
# ============================================================
GT_CSV    = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
SP_CSV    = Path("/data/baliu/python_code/data/sp_sampled_with_geocontext.csv")
OUT_DIR   = Path("/data/baliu/python_code/data/finetune_ready_v2")
MODEL_DIR = Path("/data/baliu/python_code/models/qwen25_mobility_lora_v2")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME  = "Qwen/Qwen2.5-3B-Instruct"
LABEL_COLS  = ["age_group", "household_size_group", "income_level", "gender"]  
SEED        = 42
MAX_SEQ_LEN = 8192

# ============================================================
# STEP 1: Load data
# ============================================================
sp = pd.read_csv(SP_CSV, low_memory=False)
gt = pd.read_csv(GT_CSV, low_memory=False)
sp["user_id"] = sp["user_id"].astype(str).str.strip()
gt["user_id"] = gt["user_id"].astype(str).str.strip()
print(f"Staypoints : {len(sp)} rows | {sp['user_id'].nunique()} users")
print(f"GT         : {len(gt)} rows")

# ============================================================
# STEP 2: Load tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

# ============================================================
# STEP 3: Build prompts
# ============================================================
DOW_NAMES = {
    0: "Sunday", 1: "Monday", 2: "Tuesday", 3: "Wednesday",
    4: "Thursday", 5: "Friday", 6: "Saturday"
}
HOUR_BIN_LABELS = {
    0: "midnight", 1: "early morning", 2: "morning",
    3: "afternoon", 4: "evening", 5: "night"
}

def clean_addr_part(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    return None if s.lower() in ["nan", "none", "-", ""] else s

def tokens_compact_1week(df_u, max_events=40, prec=4):
    duration_col = next(
        (c for c in df_u.columns if c.lower() in
         ["duration", "duration_min", "act_duration", "act_duration_min", "dur_min"]),
        None
    )
    use_cols = ["started_at", "dow", "hour_bin", "location_id", "lon", "lat",
                "mode", "city", "neighbourhood", "road", "nearby_places", "postcode"]
    if duration_col:
        use_cols.append(duration_col)

    df = df_u.loc[:, [c for c in use_cols if c in df_u.columns]].copy()
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df = df.dropna(subset=["started_at"]).sort_values("started_at").reset_index(drop=True)
    if len(df) > max_events:
        df = df.head(max_events)

    toks, current_date = [], None
    for r in df.itertuples(index=False):
        t = r.started_at
        date_str = t.date().isoformat()
        if date_str != current_date:
            toks.append(f"--- Date: {date_str} ---")
            current_date = date_str

        hhmm     = t.strftime("%H:%M")
        dow_raw  = getattr(r, "dow", None)
        hb_raw   = getattr(r, "hour_bin", None)
        dow_name = DOW_NAMES.get(int(dow_raw), "unknown") if pd.notna(dow_raw) else "unknown"
        hb_label = HOUR_BIN_LABELS.get(int(hb_raw), "") if pd.notna(hb_raw) else ""

        lat = round(float(getattr(r, "lat", 0.0)), prec)
        lon = round(float(getattr(r, "lon", 0.0)), prec)

        addr_parts = [clean_addr_part(getattr(r, c, None))
                      for c in ["road", "neighbourhood", "city"]]
        addr = ", ".join(p for p in addr_parts if p) or "unknown address"
        postcode = clean_addr_part(getattr(r, "postcode", None))
        if postcode:
            addr = f"{addr} (postcode {postcode})"

        nearby_raw = getattr(r, "nearby_places", None)
        if isinstance(nearby_raw, list):
            nearby_str = "; ".join(nearby_raw[:5])
        elif isinstance(nearby_raw, str) and nearby_raw.strip():
            nearby_str = nearby_raw
        else:
            nearby_str = "no nearby places listed"

        mode    = str(getattr(r, "mode", "unknown")).strip().lower()
        mode    = "unknown" if mode in ["nan", "none", ""] else mode
        dur_val = getattr(r, duration_col, None) if duration_col else None
        dur_str = (f"{int(round(float(dur_val)))}"
                   if dur_val is not None and pd.notna(dur_val) else "unknown")

        time_ctx = (f"At {hhmm} ({dow_name}, {hb_label})"
                    if hb_label else f"At {hhmm} ({dow_name})")
        toks.append(
            f"{time_ctx}, the user traveled by {mode} to {addr} "
            f"(coordinates: {lat}, {lon}), staying for approximately {dur_str} minutes. "
            f"The surrounding area features {nearby_str}."
        )
    return toks

# ── strict system message — SAME in training and evaluation ──────────────
SYSTEM_MSG = (
    "You are a mobility behaviour analyst. "
    "Based only on the mobility behaviour and nearby spatial context described below, "
    "infer the user's demographic profile using Swiss census data.\n\n"
    "You MUST reply in this exact format with no extra text, explanation, or punctuation:\n"
    "Age group: <value>\n"
    "Household size: <value>\n"
    "Income level: <value>\n"
    "Gender: <value>\n\n"
    "Valid values only:\n"
    "  Age group     : 0-17, 18-24, 25-44, 45-66, 67+\n"
    "  Household size: 1, 2, 3, 4, 5+\n"
    "  Income level  : <4000, 4001-8000, 8001-12000, 12001-16000, 16001+\n"
    "  Gender        : male, female, other, prefer not to say"
)

prompt_records = []
for user_id, df_u in sp.groupby("user_id"):
    toks = tokens_compact_1week(df_u, max_events=40, prec=4)
    prompt_text = "\n".join(toks)
    if prompt_text.strip():
        prompt_records.append({"user_id": user_id, "prompt_text": prompt_text})

prompt_df = pd.DataFrame(prompt_records)
print(f"Built prompts for {len(prompt_df)} users")

# ============================================================
# STEP 4: Merge + clean labels
# ============================================================
gt_labels = gt[["user_id"] + LABEL_COLS].drop_duplicates("user_id")
merged    = prompt_df.merge(gt_labels, on="user_id", how="inner")
merged    = merged.dropna(subset=LABEL_COLS)
merged    = merged[
    ~merged["income_level"].astype(str).str.lower().str.contains("prefer not", na=False)
]
print(f"Final records: {len(merged)}")

# ============================================================
# STEP 5: Build chat records + split
# ============================================================
def build_label_response(row):
    return (
        f"Age group: {row['age_group']}\n"
        f"Household size: {row['household_size_group']}\n"
        f"Income level: {row['income_level']}\n"
        f"Gender: {row['gender']}"
    )

records = [
    {"messages": [
        {"role": "system",    "content": SYSTEM_MSG},
        {"role": "user",      "content": row["prompt_text"].strip()},
        {"role": "assistant", "content": build_label_response(row)},
    ]}
    for _, row in merged.iterrows()
]

train_records, test_records = train_test_split(records, test_size=0.2, random_state=SEED)
print(f"Train: {len(train_records)} | Test: {len(test_records)}")

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in data:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

save_jsonl(train_records, OUT_DIR / "train.jsonl")
save_jsonl(test_records,  OUT_DIR / "test.jsonl")
print("JSONL files saved.")

# ============================================================
# STEP 6: Apply chat template + truncate to MAX_SEQ_LEN
# ============================================================
def apply_chat_template(examples):
    return {"text": [
        tokenizer.apply_chat_template(
            msg, tokenize=False, add_generation_prompt=False
        )
        for msg in examples["messages"]
    ]}

def truncate_text(examples):
    return {"text": [
        tokenizer.decode(
            tokenizer.encode(t, max_length=MAX_SEQ_LEN, truncation=True),
            skip_special_tokens=False
        )
        for t in examples["text"]
    ]}

train_ds = Dataset.from_list(train_records).map(apply_chat_template, batched=True)
test_ds  = Dataset.from_list(test_records).map(apply_chat_template,  batched=True)
train_ds = train_ds.map(truncate_text, batched=True)
test_ds  = test_ds.map(truncate_text,  batched=True)
print(f"train_ds: {len(train_ds)} | test_ds: {len(test_ds)}")

# token length sanity check
sample_lengths = [len(tokenizer.encode(x)) for x in train_ds["text"][:100]]
over_limit = sum(1 for l in sample_lengths if l > MAX_SEQ_LEN)
print(f"[CHECK] Over {MAX_SEQ_LEN} tokens: {over_limit}/100 samples")

# ============================================================
# STEP 7: Load model (4-bit QLoRA) — base model only
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.enable_input_require_grads()
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
print(f"Model loaded. GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB / 22 GB")

# ============================================================
# STEP 8: LoRA config — improved vs v1
#   r: 8  → 16  (more capacity)
#   target_modules: 2 → 7 (all projection layers)
# ============================================================
lora_config = LoraConfig(
    r=16,                        # increased from 8
    lora_alpha=32,               # increased from 16
    target_modules=[             # expanded from ["q_proj", "v_proj"]
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# ============================================================
# STEP 9: SFTConfig — 5 epochs, no max_seq_length (trl 0.29)
# ============================================================
sft_config = SFTConfig(
    output_dir=str(MODEL_DIR),

    # ── training ────────────────────────────────────────────
    num_train_epochs=5,              # increased from 3
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    weight_decay=0.01,

    # ── precision & memory ──────────────────────────────────
    bf16=True,
    optim="paged_adamw_8bit",
    dataloader_pin_memory=False,
    dataloader_num_workers=0,

    # ── SFT-specific ────────────────────────────────────────
    dataset_text_field="text",

    # ── evaluation & saving ─────────────────────────────────
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # ── misc ────────────────────────────────────────────────
    report_to="none",
    seed=SEED,
    logging_steps=10,
)

# ============================================================
# STEP 10: Train
# ============================================================
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    peft_config=lora_config,
    args=sft_config,
)

trainer.model.print_trainable_parameters()
print("\n[INFO] Starting training ...")
trainer.train()

# ============================================================
# STEP 11: Save + evaluate
# ============================================================
trainer.save_model(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print(f"Saved → {MODEL_DIR / 'final'}")

metrics = trainer.evaluate()
print("Eval metrics:", metrics)
pd.Series(metrics).to_csv(MODEL_DIR / "eval_metrics.csv")

Staypoints : 60738 rows | 2102 users
GT         : 90909 rows


KeyboardInterrupt: 

In [ ]:
script = '''
import json
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch
from tqdm import tqdm

MODEL_BASE = "Qwen/Qwen2.5-3B-Instruct"
MODEL_LORA = "/data/baliu/python_code/models/qwen25_mobility_lora/final"
TEST_JSONL = "/data/baliu/python_code/data/finetune_ready/test.jsonl"

tokenizer = AutoTokenizer.from_pretrained(MODEL_LORA, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_BASE,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, MODEL_LORA)
model.eval()

test_records = [json.loads(l) for l in open(TEST_JSONL)]
print(f"Test samples: {len(test_records)}")

results = []
for rec in tqdm(test_records):          
    msgs = rec["messages"]
    gt_response = msgs[2]["content"]
    prompt = tokenizer.apply_chat_template(
        msgs[:2], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    pred = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()
    results.append({"gt": gt_response, "pred": pred})
    print(f"\\nGT  : {gt_response}")
    print(f"PRED: {pred}")
    print("-" * 40)

def parse_response(text):
    out = {}
    for line in text.strip().splitlines():
        if ":" in line:
            key, _, val = line.partition(":")
            out[key.strip().lower().replace(" ", "_")] = val.strip()
    return out

df = pd.DataFrame(results)
df["gt_parsed"]   = df["gt"].apply(parse_response)
df["pred_parsed"] = df["pred"].apply(parse_response)

for label in ["age_group", "household_size_group", "income_level"]:  # fix 2: household_size_group
    correct = sum(
        r.get(label, "").lower() == p.get(label, "").lower()
        for r, p in zip(df["gt_parsed"], df["pred_parsed"])
    )
    print(f"{label} accuracy: {correct}/{len(df)} = {correct/len(df):.1%}")

# save results
out_path = Path("/data/baliu/python_code/data/eval_results.csv")
df["gt_age"]       = df["gt_parsed"].apply(lambda x: x.get("age_group", ""))
df["gt_household"] = df["gt_parsed"].apply(lambda x: x.get("household_size_group", ""))
df["gt_income"]    = df["gt_parsed"].apply(lambda x: x.get("income_level", ""))
df["pred_age"]       = df["pred_parsed"].apply(lambda x: x.get("age_group", ""))
df["pred_household"] = df["pred_parsed"].apply(lambda x: x.get("household_size_group", ""))
df["pred_income"]    = df["pred_parsed"].apply(lambda x: x.get("income_level", ""))
df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
'''

with open("/data/baliu/python_code/evaluate_qwen25.py", "w") as f:
    f.write(script)
print("File saved successfully!")

File saved successfully!


In [7]:
import pandas as pd
df = pd.read_csv("/data/baliu/python_code/data/eval_results.csv")

# see the raw predictions
pd.set_option("display.max_colwidth", 200)
df[["gt", "pred", "gt_age", "pred_age", "gt_income", "pred_income"]].head(20)

,gt,pred,gt_age,pred_age,gt_income,pred_income
0,Age group: 45-66\nHousehold size: Unknown\nIncome level: 8001-12000,"Based on the detailed sequence of GPS locations and activities over the week, we can infer some characteristics about the user's demographic profile:\n\n1. **Geographical Mobility**: The user freq...",45-66,NaN,8001-12000,NaN
1,Age group: 45-66\nHousehold size: 2\nIncome level: 4001-8000,"Based on the one-week sequence of GPS staypoints for this Swiss resident, we can infer the following about their demographic profile:\n\n1. **Age**: The individual seems to be relatively young or ...",45-66,NaN,4001-8000,NaN
2,Age group: 45-66\nHousehold size: 2\nIncome level: 8001-12000,"Based on the detailed sequence of GPS staypoints provided, we can infer some characteristics about the user's demographic profile:\n\n1. **Location**: The user frequently visits areas around Zeini...",45-66,NaN,8001-12000,NaN
3,Age group: 25-44\nHousehold size: 4\nIncome level: 4001-8000,"Based on the sequence of GPS staypoints provided, we can infer some characteristics about the user's demographic profile:\n\n1. **Age and Mobility**: The user spends a significant amount of time i...",25-44,NaN,4001-8000,NaN
4,Age group: 25-44\nHousehold size: 3\nIncome level: 4001-8000,"Based on the detailed GPS staypoints provided over the week, we can infer some characteristics about the user's demographic profile:\n\n1. **Location**: The user frequently visits areas around Zur...",25-44,NaN,4001-8000,NaN
5,Age group: 45-66\nHousehold size: 2\nIncome level: 8001-12000,"Based on the detailed sequence of GPS locations and activities over the week, we can infer the following about the user's demographic profile:\n\n1. **Age**: The user frequently travels to differe...",45-66,NaN,8001-12000,NaN
6,Age group: 25-44\nHousehold size: Unknown\nIncome level: 4001-8000,"Based on the detailed mobility pattern observed over the week, we can infer that the user is likely a middle-aged individual with a higher socioeconomic status. Here's the reasoning:\n\n1. **Long ...",25-44,NaN,4001-8000,NaN
7,Age group: 25-44\nHousehold size: 2\nIncome level: 4001-8000,"Based on the detailed sequence of GPS staypoints provided, we can infer some characteristics about the user's demographic profile:\n\n1. **Location**: The user frequently visits areas around Lengg...",25-44,NaN,4001-8000,NaN
8,Age group: 45-66\nHousehold size: 4\nIncome level: >16000,"Based on the detailed GPS staypoints provided over the week, we can infer some characteristics about the user's demographic profile:\n\n1. **Age and Mobility**: The user spends significant time in...",45-66,NaN,>16000,NaN
9,Age group: 18-24\nHousehold size: 2\nIncome level: <4000,"Based on the detailed sequence of GPS locations and activities over the week, we can infer some characteristics about the user's demographic profile:\n\n1. **Age**: The user appears to be relative...",18-24,NaN,<4000,NaN


In [ ]:
# ---- count for the length of prompt ----
lengths = [
    len(tokenizer(rec["messages"][1]["content"])["input_ids"])  # user turn length
    for rec in records
]

import numpy as np
print(pd.Series(lengths).describe(percentiles=[.5, .75, .90, .95, .99]))
print(f"\n precentage of samples with length > 2048: {np.mean([l > 2048 for l in lengths]):.1%}")
print(f" precentage of samples with length > 4096: {np.mean([l > 4096 for l in lengths]):.1%}")
print(f" precentage of samples with length > 8192: {np.mean([l > 8192 for l in lengths]):.1%}")

In [ ]:
# check the content of one sample
from transformers import AutoTokenizer
import json

tokenizer = AutoTokenizer.from_pretrained(
    "/data/baliu/python_code/models/qwen25_mobility_lora/final",
    trust_remote_code=True
)

# check one sample from the train.jsonl
rec = json.loads(open("/data/baliu/python_code/data/finetune_ready/train.jsonl").readline())

user_tokens = len(tokenizer(rec["messages"][1]["content"])["input_ids"])
asst_tokens = len(tokenizer(rec["messages"][2]["content"])["input_ids"])

print(f"User prompt tokens : {user_tokens}")
print(f"Assistant tokens   : {asst_tokens}")
print(f"Assistant content  : {rec['messages'][2]['content']}")

In [ ]:
from trl import SFTConfig
import inspect
params = inspect.signature(SFTConfig.__init__).parameters
for name, p in params.items():
    if any(x in name.lower() for x in ['len','seq','max','token','trunc']):
        print(f'{name} = {p.default}')

In [ ]:
import json
import pandas as pd
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

# ============================================================
# PATHS & CONFIG
# ============================================================
GT_CSV    = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
SP_CSV    = Path("/data/baliu/python_code/data/sp_sampled_with_geocontext.csv")
OUT_DIR   = Path("/data/baliu/python_code/data/finetune_ready")
MODEL_DIR = Path("/data/baliu/python_code/models/qwen25_mobility_lora")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
LABEL_COLS = ["age_group", "household_size_group", "income_level"]
SEED       = 42
MAX_SEQ_LEN = 8192

# ============================================================
# STEP 1: Load data
# ============================================================
sp = pd.read_csv(SP_CSV, low_memory=False)
gt = pd.read_csv(GT_CSV, low_memory=False)
sp["user_id"] = sp["user_id"].astype(str).str.strip()
gt["user_id"] = gt["user_id"].astype(str).str.strip()
print(f"Staypoints: {len(sp)} rows | {sp['user_id'].nunique()} users")
print(f"GT        : {len(gt)} rows")

# ============================================================
# STEP 2: Build one prompt per user from trajectory tokens
# ============================================================
SYSTEM_MSG = (
    "You are a mobility behaviour analyst. "
    "Given a one-week sequence of GPS staypoint events for a Swiss resident, "
    "predict their demographic profile."
)

def build_user_prompt(df_u: pd.DataFrame) -> str:
    toks = tokens_compact_1week(df_u, max_events=40, prec=4)
    return "\n".join(toks)

prompt_records = []
for user_id, df_u in sp.groupby("user_id"):
    prompt_text = build_user_prompt(df_u)
    if prompt_text.strip():
        prompt_records.append({"user_id": user_id, "prompt_text": prompt_text})

prompt_df = pd.DataFrame(prompt_records)
print(f"\nBuilt prompts for {len(prompt_df)} users")

# ============================================================
# STEP 3: Merge with GT labels
# ============================================================
gt_labels = gt[["user_id"] + LABEL_COLS].drop_duplicates("user_id")
merged = prompt_df.merge(gt_labels, on="user_id", how="inner")
print(f"Matched users: {len(merged)}")

# drop missing / refused labels
merged = merged.dropna(subset=LABEL_COLS)
merged = merged[
    ~merged["income_level"].astype(str).str.lower().str.contains("prefer not", na=False)
]
print(f"After dropping bad labels: {len(merged)}")

# ============================================================
# STEP 4: Build chat-format records
# ============================================================
def build_label_response(row):
    return (
        f"Age group: {row['age_group']}\n"
        f"Household size: {row['household_size_group']}\n"
        f"Income level: {row['income_level']}"
    )

records = [
    {
        "messages": [
            {"role": "system",    "content": SYSTEM_MSG},
            {"role": "user",      "content": row["prompt_text"].strip()},
            {"role": "assistant", "content": build_label_response(row)},
        ]
    }
    for _, row in merged.iterrows()
]
print(f"Total records: {len(records)}")

# preview
print("\n--- Example ---")
for msg in records[0]["messages"]:
    print(f"[{msg['role'].upper()}]\n{msg['content'][:300]}\n")

# ============================================================
# STEP 5: 80/20 split + save JSONL
# ============================================================
train_records, test_records = train_test_split(
    records, test_size=0.2, random_state=SEED
)
print(f"Train: {len(train_records)} | Test: {len(test_records)}")

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for rec in data:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Saved → {path}")

save_jsonl(train_records, OUT_DIR / "train.jsonl")
save_jsonl(test_records,  OUT_DIR / "test.jsonl")

# ============================================================
# STEP 6: Tokeniser + apply chat template
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

def apply_chat_template(examples):
    return {
        "text": [
            tokenizer.apply_chat_template(
                msg, tokenize=False, add_generation_prompt=False
            )
            for msg in examples["messages"]
        ]
    }

train_ds = Dataset.from_list(train_records).map(apply_chat_template, batched=True)
test_ds  = Dataset.from_list(test_records).map(apply_chat_template,  batched=True)
print(f"\ntrain_ds: {len(train_ds)} | test_ds: {len(test_ds)}")
print("Sample text (first 400 chars):\n", train_ds[0]["text"][:400])

# ============================================================
# STEP 7: Load model (4-bit QLoRA)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================
# STEP 8: Train
# ============================================================
training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    bf16=True,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
)

print("\n[INFO] Starting fine-tuning ...")
trainer.train()

# ============================================================
# STEP 9: Save
# ============================================================
trainer.save_model(str(MODEL_DIR / "final"))
tokenizer.save_pretrained(str(MODEL_DIR / "final"))
print(f"Saved → {MODEL_DIR / 'final'}")

metrics = trainer.evaluate()
print("Eval metrics:", metrics)
pd.Series(metrics).to_csv(MODEL_DIR / "eval_metrics.csv")

prediction with rationale
v3 means a version with rationale in Qwen

In [ ]:
import json
import time
import os
import re
import sys
import signal
from pathlib import Path
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# --------------------------------------
# Config
# --------------------------------------

MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

PROMPT_TXT = Path("/data/baliu/python_code/data/prompts_v3_1week_age_householdsize_04Mar2026_qwen_clean_v4.txt")
PRED_JSON  = Path("/data/baliu/python_code/data/preds_qwen_age,householdsize_05Mar2026_v6withrationale.jsonl")
PRED_CSV   = Path("/data/baliu/python_code/data/preds_qwen_age,householdsize_05Mar2026_v6withrationale.csv")
# preds_qwen_age,householdsize_04Mar2026_v6withrationale.csv

SEP = "=" * 80

MAX_PROMPT_CHARS = 30000  # Qwen 7B can handle up to 32k tokens, which is roughly 30k chars, avoid OOM

TIMEOUT_SEC = 600
COOLDOWN_EVERY = 20
COOLDOWN_SEC = 10

# --------------------------------------
# Timeout handling
# --------------------------------------

class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("Generation timed out")

signal.signal(signal.SIGALRM, timeout_handler)

# --------------------------------------
# Load done users
# --------------------------------------
def load_done_users(path: Path) -> set:
    done = set()
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if "user_id" in obj:
                        done.add(obj["user_id"])
                except Exception:
                    continue
    return done

# --------------------------------------
# Load prompts
# --------------------------------------
with open(PROMPT_TXT, "r", encoding="utf-8") as f:
    raw = f.read()

prompts = [p.strip() for p in raw.split(SEP) if p.strip()]
print(f"📦 Loaded {len(prompts)} prompts")

done_users = load_done_users(PRED_JSON)
print(f"🔁 Found {len(done_users)} completed users")

# --------------------------------------
# Main loop
# --------------------------------------
try:
    for i, prompt in enumerate(prompts, 1):

        m = re.search(r"User:\s*(\S+)", prompt)
        if not m:
            print("⚠️ Cannot find user_id")
            continue

        user_id = m.group(1)

        if user_id in done_users:
            print(f"⏭️ Skipping {user_id}")
            continue

        print(f"\n🔮 [{i}/{len(prompts)}] Predicting {user_id}")

        if len(prompt) > MAX_PROMPT_CHARS:
            print("⚠️ Prompt too long, skipping")
            continue

        try:
            signal.alarm(TIMEOUT_SEC)

            messages =[
                {"role": "system",
                 "content": (
                     "You are a mobility behavior and socioeconomic status inference analyst."
                     "return only valid JSON with keys: age_group, gender, household_size, household_income_level."
                     "Based only on the mobility behaviour and nearby spatial context described below,"
                     # "infer the user's most likely household income level based on swiss census data and situation\n\n"
                     "Please choose exactly one of the following categories (monthly household income in CHF),age_group, gender, household_size:\n"
                     "<4000, 4001-8000, 8001-12000, 12001-16000, 16001+. age: 0-17, 18-24, 25-34, 35-44, 45-54, 55-64, 65+. gender: male, female, other, perefer not to say; household_size: 1, 2, 3, 4, 5+\n\n"
                     " Guidelines:\n"
                     " this project conducted in switzerland, please consider the swiss context when making inference. use the swiss geographic context, like the distribution of urban and rural areas, like event happened in zentrum zurich, oerlikon zurich,  public transportation networks, and local economic activities. \n"
                     "- Use only the information explicitly provided in the mobility record.\n"
                     "- Base your reasoning on mobility intensity, locations visited, transport modes, and activity patterns.\n"
                     "- keep a neutral and factual tone.\n\n"
                     "Output format:\n"
                     "1. Reasoning (no more than 150 words).\n" # need to adjust the parameters for rationale length
                     "2. One final line in strict JSON format, for example:\n"
                     " {\"household_income_level\": \"4001-8000\", \"age_group\": \"45-54\", \"gender\": \"other\", \"household_size\": 3, \"prediction_rationale\": [\"Evidence 1\", \"Evidence 2, give us the real rationale\"], \"prediction_confidence\": \"high\"}\n\n"
                     "- prediction_rationale must be a list of 3-6 short bullet-like strings, each describing a concrete mobility/semantic evidence. \n "
                     "- Provide step by step reasoning, or chain of thought"
                     "- Just provide evidence-style justification.\n"
                     "- prediction_confidence must be one of :low, medium, high.\n"
                     "no extra keys, no markdown, no commentary outside josn"

                 )
                 },
            ]

            tokenizer = AutoTokenizer.from_pretrained(
                MODEL_NAME,
                trust_remote_code=True)

            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=512, # give space for rationale
                temperature=0.5,
                do_sample=True,
                top_p=0.9,
                repetition_penalty=1.05 # to encourage more diverse outputs
            )

            signal.alarm(0)
            
            # decode only the generated part, avoid mixing prompt + answer
            gen_ids = outputs[0][inputs["input_ids"].shape[1]:]  

            decoded = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

            json_match = re.search(r"\{.*\}", decoded, re.DOTALL)
            if json_match:
                #json_str = json_match.group(0)
                #try:
                    #result = json.loads(json_str)


                # need to fix 
                result = json.loads(json_match.group(0))
            else:
                result = {"raw_output": decoded, "error": "json_parse_failed"}

            result["user_id"] = user_id

            with open(PRED_JSON, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

            done_users.add(user_id)
            print("✅ Done")

        except TimeoutError:
            print("⏰ Timeout")
            continue

        except Exception as e:
            print("❌ Error:", e)
            time.sleep(2)
            continue

        if i % COOLDOWN_EVERY == 0:
            print("🧹 Cooling down GPU...")
            time.sleep(COOLDOWN_SEC)

except KeyboardInterrupt:
    print("\n🛑 Interrupted safely.")
    sys.exit(0)

# --------------------------------------
# Save csv snapshot
# --------------------------------------

if PRED_JSON.exists():
    df = pd.read_json(PRED_JSON, lines=True)
    df.to_csv(PRED_CSV, index=False)
    print(f"\n🎉 Saved → {PRED_CSV}")
else:
    print("⚠️ No predictions generated")

read prompts from text version

In [ ]:
import json
import time
import os
import re
import sys
import signal
from pathlib import Path
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# --------------------------------------
# Config
# --------------------------------------

MODEL_NAME = "Qwen/Qwen2-7B-Instruct"

PROMPT_TXT = Path("/data/baliu/python_code/data/prompts_v3_1week_income_22Feb2026_qwen_clean_v2.txt")
PRED_JSON  = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v2withrationale.jsonl")
PRED_CSV   = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v2withrationale.csv")
# data/prompts_v3_1week_income_22Feb2026_qwen_clean_v2
# preds_qwen_age,householdsize_04Mar2026_v6withrationale

SEP = "=" * 80

MAX_PROMPT_CHARS = 25000
#MAX_PROMPT_CHARS = 30000 # Qwen 7B can handle up to 32k tokens, which is roughly 30k chars, but we set a safer limit to avoid OOM

TIMEOUT_SEC = 600
COOLDOWN_EVERY = 20
COOLDOWN_SEC = 10

# --------------------------------------
# GPU check
# --------------------------------------

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------
# Load Qwen 4bit
# --------------------------------------

print("🔄 Loading Qwen...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

print("✅ Qwen ready.")

# --------------------------------------
# Timeout handling
# --------------------------------------

class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("Generation timed out")

signal.signal(signal.SIGALRM, timeout_handler)

# --------------------------------------
# Load done users
# --------------------------------------

def load_done_users(path: Path) -> set:
    done = set()
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if "user_id" in obj:
                        done.add(obj["user_id"])
                except Exception:
                    continue
    return done

# --------------------------------------
# Load prompts
# --------------------------------------

with open(PROMPT_TXT, "r", encoding="utf-8") as f:
    raw = f.read()

prompts = [p.strip() for p in raw.split(SEP) if p.strip()]
print(f"📦 Loaded {len(prompts)} prompts")

done_users = load_done_users(PRED_JSON)
print(f"🔁 Found {len(done_users)} completed users")

# --------------------------------------
# Main loop
# --------------------------------------

try:
    for i, prompt in enumerate(prompts, 1):

        m = re.search(r"User:\s*(\S+)", prompt)
        if not m:
            print("⚠️ Cannot find user_id")
            continue

        user_id = m.group(1)

        if user_id in done_users:
            print(f"⏭️ Skipping {user_id}")
            continue

        print(f"\n🔮 [{i}/{len(prompts)}] Predicting {user_id}")

        if len(prompt) > MAX_PROMPT_CHARS:
            print("⚠️ Prompt too long, skipping")
            continue

        try:
            signal.alarm(TIMEOUT_SEC)

            messages = [
                {"role": "system",
                 "content": "Respond only in valid JSON with keys: age_group, gender, household_size."},
                {"role": "user", "content": prompt}
            ]

            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.3,
                do_sample=False
            )

            signal.alarm(0)

            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

            json_match = re.search(r"\{.*\}", decoded, re.DOTALL)
            if json_match:
                result = json.loads(json_match.group())
            else:
                result = {"raw_output": decoded, "error": "json_parse_failed"}

            result["user_id"] = user_id

            with open(PRED_JSON, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")

            done_users.add(user_id)
            print("✅ Done")

        except TimeoutError:
            print("⏰ Timeout")
            continue

        except Exception as e:
            print("❌ Error:", e)
            time.sleep(2)
            continue

        if i % COOLDOWN_EVERY == 0:
            print("🧹 Cooling down GPU...")
            time.sleep(COOLDOWN_SEC)

except KeyboardInterrupt:
    print("\n🛑 Interrupted safely.")
    sys.exit(0)

# --------------------------------------
# Save csv snapshot
# --------------------------------------

if PRED_JSON.exists():
    df = pd.read_json(PRED_JSON, lines=True)
    df.to_csv(PRED_CSV, index=False)
    print(f"\n🎉 Saved → {PRED_CSV}")
else:
    print("⚠️ No predictions generated")

## Evaluation

#### this precedure is just for income level

In [ ]:
import pandas as pd
import json
import re
from pathlib import Path

# --------------------------------------
# Load raw predictions
# --------------------------------------
raw_path = Path("/data/baliu/python_code/data/preds_qwen_age,householdsize_05Mar2026_v6withrationale.jsonl")
#preds_qwen_age,householdsize_05Mar2026_v6withrationale.jsonl

df = pd.read_json(raw_path, lines=True)
print(f"Loaded raw predictions: {len(df)} rows")

# --------------------------------------
# Income, age group, household size, gender extraction functions
# --------------------------------------
def extract_from_text(text):
    if pd.isna(text):
        return None

    try:
        data = json.loads(text)
        income_level = data.get("income_level", "")
        if income_level:
            if income_level in [">=16000", ">=16001", "16001+", "16000+", "more than 16000", "above 16000", "at least 16000", "16000+"]:
                return ">16000"
            return income_level
    except Exception:
        pass

    text = str(text).lower()

    norm = (
        text.lower()
            .replace("chf", "")
            .replace("\u202f", "")
            .replace("\xa0", "")
            .replace(",", "")
    )

    if re.search(
        r'(>=\s*16000|>=\s*16001|16001\+|16000\+|'
        r'more\s+than\s+16000|above\s+16000|at\s+least\s+16000)',
        norm
    ):
        return ">16000"

    m = re.search(r'between\s*(\d{3,5})\s*and\s*(\d{3,5})', norm)
    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    m = re.search(r'(\d{1,2}[,.]?\d{3})\s*[-–]\s*(\d{1,2}[,.]?\d{3})', norm)

    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    for g in ["<4000", "4001-8000", "8001-12000", "12001-16000", ">16000"]:
        if g in text:
            return g

    return None

def map_income_bin(low, high):
    if high <= 4000:
        return "<4000"
    elif low >= 4001 and high <= 8000:
        return "4001-8000"
    elif low >= 8001 and high <= 12000:
        return "8001-12000"
    elif low >= 12001 and high <= 16000:
        return "12001-16000"
    elif high >= 16000:
        return ">16000"
    else:
        return None

# --------------------------------------
# Apply extraction
# --------------------------------------
TEXT_COL = "household_income_level"
df["household_income_level"] = df[TEXT_COL].apply(extract_from_text)

print(df.columns.tolist())
print(df.head(3))
print(df["household_income_level"].value_counts(dropna=False))

# --------------------------------------
# Save cleaned CSV
# -----------------------------------------
df_clean = (
    df[["user_id", "household_income_level"]]
    .drop_duplicates(subset=["user_id"])
    .sort_values("user_id")
    .reset_index(drop=True)
)

out_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.jsonl")
df_clean.to_json(out_path, orient="records", lines=True)
out_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.csv")
# preds_qwen_income_22Feb2026_v2withrationale.csv
df_clean.to_csv(out_path, index=False)

print(f"Clean predictions saved to: {out_path}")
print(f"📊 Rows: {len(df_clean)}")
print(df_clean.head(10))

In [ ]:
import pandas as pd
import json
import re
from pathlib import Path

# ============================================================================
# CONFIGURATION
# ============================================================================
# Define the column containing raw model output
# If the data is in a single column (like "prediction" or "raw_output"):
TEXT_COL = "prediction"  # ← Change this to your actual column name

# If each field is already in separate columns, set TEXT_COL = None
# and define the column names:
SEPARATE_COLS = False  # Set to True if data is already in separate columns
AGE_COL = "age_group"
GENDER_COL = "gender"
INCOME_COL = "household_income_level"
HOUSEHOLD_COL = "household_size"

# ============================================================================
# LOAD RAW PREDICTIONS
# ============================================================================
raw_path = Path("/data/baliu/python_code/data/preds_qwen_age,householdsize_05Mar2026_v6withrationale.jsonl")
df = pd.read_json(raw_path, lines=True)
print(f"Loaded raw predictions: {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst row sample:")
print(df.iloc[0])

# ============================================================================
# INCOME BIN MAPPING
# ============================================================================
def map_income_bin(low, high):
    if high <= 4000:
        return "<4000"
    elif low >= 4001 and high <= 8000:
        return "4001-8000"
    elif low >= 8001 and high <= 12000:
        return "8001-12000"
    elif low >= 12001 and high <= 16000:
        return "12001-16000"
    elif high >= 16000:
        return ">16000"
    return None

# ============================================================================
# GENERIC JSON SAFE LOADER
# ============================================================================
def safe_json_load(text):
    if pd.isna(text):
        return {}
    try:
        return json.loads(text)
    except Exception:
        return {}

# ============================================================================
# GENDER CLEANING
# ============================================================================
def clean_gender(text):
    if pd.isna(text):
        return None
    text = str(text).lower()
    if "female" in text or text == "f":
        return "female"
    if "male" in text or text == "m":
        return "male"
    if "other" in text:
        return "other"
    return None

# ============================================================================
# AGE GROUP CLEANING
# ============================================================================
VALID_AGE_GROUPS = [
    "18-25", "26-35", "36-45", "46-55", "56-65", "66+"
]

def clean_age(text):
    if pd.isna(text):
        return None

    text = str(text).lower()

    for group in VALID_AGE_GROUPS:
        if group in text:
            return group

    # Catch patterns like "between 26 and 35"
    m = re.search(r'(\d{2})\s*[-–]\s*(\d{2})', text)
    if m:
        return f"{m.group(1)}-{m.group(2)}"

    return None

# ============================================================================
# HOUSEHOLD SIZE CLEANING
# ============================================================================
def clean_household_size(text):
    if pd.isna(text):
        return None

    text = str(text).lower()

    if "1" in text:
        return "1"
    if "2" in text:
        return "2"
    if "3" in text:
        return "3"
    if "4" in text:
        return "4"
    if "5" in text or "5+" in text or "more than 5" in text:
        return "5+"

    return None

# ============================================================================
# INCOME CLEANING
# ============================================================================
def clean_income(text):
    if pd.isna(text):
        return None

    # Try JSON first
    try:
        data = json.loads(text)
        income = data.get("income_level", "")
        if income:
            income = str(income)
            if "16000" in income or "16001" in income:
                return ">16000"
            return income
    except Exception:
        pass

    text = str(text).lower()
    norm = (
        text.replace("chf", "")
            .replace("\u202f", "")
            .replace("\xa0", "")
            .replace(",", "")
    )

    # High income  
    if re.search(r'(>=\s*16000|16000\+|more\s+than\s+16000|above\s+16000)', norm):
        return ">16000"

    # Between pattern
    m = re.search(r'between\s*(\d{3,5})\s*and\s*(\d{3,5})', norm)
    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    m = re.search(r'(\d{1,2}\d{3})\s*[-–]\s*(\d{1,2}\d{3})', norm)
    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    # Direct bin mention
    for g in ["<4000", "4001-8000", "8001-12000", "12001-16000", ">16000"]:
        if g in text:
            return g

    return None

# ============================================================================
# EXTRACT ALL FIELDS
# ============================================================================
def extract_all(text):
    """
    Extract all fields from a single text/JSON column.
    Returns 4 values: [age_group, gender, income, household_size]
    """
    data = safe_json_load(text)

    # Try JSON fields first
    age = data.get("age_group", None)
    gender = data.get("gender", None)
    income = data.get("income_level", None) or data.get("household_income_level", None)
    household_size = data.get("household_size", None)

    # Fallback to raw text parsing
    if age is None:
        age = clean_age(text)
    else:
        age = clean_age(age)

    if gender is None:
        gender = clean_gender(text)
    else:
        gender = clean_gender(gender)

    if income is None:
        income = clean_income(text)
    else:
        income = clean_income(income)

    if household_size is None:
        household_size = clean_household_size(text)
    else:
        household_size = clean_household_size(household_size)

    return pd.Series([age, gender, income, household_size])

# ============================================================================
# APPLY EXTRACTION
# ============================================================================
if not SEPARATE_COLS:
    # Case 1: Data is in a single column
    if TEXT_COL in df.columns:
        print(f"\n✓ Found column: {TEXT_COL}")
        print(f"Sample data:\n{df[TEXT_COL].head(2)}")
        
        # Apply extraction
        df[["age_group", "gender", "household_income_level", "household_size"]] = \
            df[TEXT_COL].apply(extract_all)
    else:
        print(f"\n Error: Column '{TEXT_COL}' not found!")
        print(f"Available columns: {df.columns.tolist()}")
        print("\nPlease update TEXT_COL to the correct column name")
        print("Example: TEXT_COL = 'prediction'")
        exit(1)
else:
    # Case 2: Data is already in separate columns
    print(f"\n✓ Using separate columns mode")
    if AGE_COL in df.columns:
        df["age_group"] = df[AGE_COL].apply(clean_age)
    if GENDER_COL in df.columns:
        df["gender"] = df[GENDER_COL].apply(clean_gender)
    if INCOME_COL in df.columns:
        df["household_income_level"] = df[INCOME_COL].apply(clean_income)
    if HOUSEHOLD_COL in df.columns:
        df["household_size"] = df[HOUSEHOLD_COL].apply(clean_household_size)

# ============================================================================
# PRINT STATISTICS
# ============================================================================
print("\n" + "="*70)
print("VALUE COUNTS")
print("="*70)

print("\nAge Group:")
print(df["age_group"].value_counts(dropna=False))

print("\nGender:")
print(df["gender"].value_counts(dropna=False))

print("\nIncome Level:")
print(df["household_income_level"].value_counts(dropna=False))

print("\nHousehold Size:")
print(df["household_size"].value_counts(dropna=False))

# Check for missing values
print("\n" + "="*70)
print("MISSING VALUES")
print("="*70)
missing = {
    "age_group": df["age_group"].isna().sum(),
    "gender": df["gender"].isna().sum(),
    "household_income_level": df["household_income_level"].isna().sum(),
    "household_size": df["household_size"].isna().sum()
}
for field, count in missing.items():
    pct = count / len(df) * 100
    print(f"{field:30s}: {count:5d} ({pct:5.2f}%)")

# ============================================================================
# SAVE CLEANED DATA
# ============================================================================
df_clean = (
    df[["user_id", "age_group", "gender", "household_income_level", "household_size"]]
    .drop_duplicates(subset=["user_id"])
    .sort_values("user_id")
    .reset_index(drop=True)
)

out_csv = Path("/data/baliu/python_code/data/preds_qwen_household_size_age_05Mar2026_v5_clean_full.csv")
df_clean.to_csv(out_csv, index=False)

out_jsonl = Path("/data/baliu/python_code/data/preds_qwen_household_size_age_05Mar2026_v5_clean_full.jsonl")
df_clean.to_json(out_jsonl, orient="records", lines=True)

print("\n" + "="*70)
print("CLEANING COMPLETED")
print("="*70)
print(f"Output CSV:   {out_csv}")
print(f"Output JSONL: {out_jsonl}")
print(f"Total rows:   {len(df_clean)}")
print("\nFirst 10 rows:")
print(df_clean.head(10))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================================
# FILE PATHS
# ============================================================================
gt_path = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
pred_path = Path("/data/baliu/python_code/data/preds_qwen_household_size_age_05Mar2026_v5_clean_full.csv")

# ============================================================================
# Load datasets
# ============================================================================
gt = pd.read_csv(gt_path)
pred = pd.read_csv(pred_path)

print(f"Loaded ground truth ({len(gt)} rows)")
print(f"Loaded predictions  ({len(pred)} rows)")

print("\n" + "="*70)
print("GROUND TRUTH COLUMNS")
print("="*70)
print(gt.columns.tolist())
print("\nFirst row:")
print(gt.head(1))

print("\n" + "="*70)
print("PREDICTION COLUMNS")
print("="*70)
print(pred.columns.tolist())
print("\nFirst row:")
print(pred.head(1))

# ============================================================================
# SELECT & RENAME COLUMNS
# ============================================================================
# Ground truth columns
gt = gt[["user_id", "age_group", "household_size_group", "income_level", "gender"]].copy()

# Prediction columns
pred = pred[["user_id", "age_group", "gender", "household_income_level", "household_size"]].copy()

# ============================================================================
# NORMALIZE HOUSEHOLD SIZE
# ============================================================================
def normalize_household_size(x):
    """Convert household size to standardized format: '1', '2', '3', '4', '5+'"""
    if pd.isna(x):
        return None
    
    x_str = str(x).strip().lower()
    
    # Handle '5+' or 'more than 5'
    if '5+' in x_str or 'more than 5' in x_str or x_str == '5+':
        return "5+"
    
    # Try to convert to number
    try:
        x_num = float(x_str)
        if x_num >= 5:
            return "5+"
        elif x_num >= 1 and x_num <= 4:
            return str(int(x_num))
        else:
            return None
    except ValueError:
        # Try to extract number
        import re
        match = re.search(r'\d+', x_str)
        if match:
            num = int(match.group())
            if num >= 5:
                return "5+"
            elif num >= 1 and num <= 4:
                return str(num)
        return None

# Apply normalization to both datasets
gt["household_size_norm"] = gt["household_size_group"].apply(normalize_household_size)
pred["household_size_norm"] = pred["household_size"].apply(normalize_household_size)

print("\n" + "="*70)
print("NORMALIZED HOUSEHOLD SIZE")
print("="*70)
print("Ground Truth:")
print(gt["household_size_norm"].value_counts(dropna=False))
print("\nPredictions:")
print(pred["household_size_norm"].value_counts(dropna=False))

# ============================================================================
# MERGE GT AND PREDICTIONS
# ============================================================================
merged = pd.merge(
    gt[["user_id", "age_group", "gender", "household_size_norm", "income_level"]],
    pred[["user_id", "age_group", "gender", "household_size_norm", "household_income_level"]],
    on="user_id",
    suffixes=("_true", "_pred"),
    how="inner"
)

print(f"\n✓ Merged {len(merged)} users")
print("\nMerged columns:")
print(merged.columns.tolist())
print("\nFirst 5 rows:")
print(merged.head(5))

# ============================================================================
# MASKS FOR VALID EVALUATION SAMPLES
# ============================================================================
# Age mask
age_mask = (
    merged["age_group_true"].notna() &
    merged["age_group_pred"].notna()
)

# Household size mask
household_mask = (
    merged["household_size_norm_true"].notna() &
    merged["household_size_norm_pred"].notna()
)

# Gender mask
gender_mask = (
    merged["gender_true"].notna() &
    merged["gender_pred"].notna()
)

# Income mask (exclude "prefer not to say")
income_mask = (
    merged["income_level"].notna() &
    merged["household_income_level"].notna() &
    (merged["income_level"].str.lower() != "prefer not to say")
)

print("\n" + "="*70)
print("EVALUATION SAMPLE SIZES")
print("="*70)
print(f"Age:           {age_mask.sum():5d} / {len(merged)} ({age_mask.sum()/len(merged)*100:.1f}%)")
print(f"Gender:        {gender_mask.sum():5d} / {len(merged)} ({gender_mask.sum()/len(merged)*100:.1f}%)")
print(f"Household:     {household_mask.sum():5d} / {len(merged)} ({household_mask.sum()/len(merged)*100:.1f}%)")
print(f"Income:        {income_mask.sum():5d} / {len(merged)} ({income_mask.sum()/len(merged)*100:.1f}%)")

# ============================================================================
# ACCURACY COMPUTATION
# ============================================================================
age_acc = (
    merged.loc[age_mask, "age_group_true"] == 
    merged.loc[age_mask, "age_group_pred"]
).mean()

gender_acc = (
    merged.loc[gender_mask, "gender_true"] == 
    merged.loc[gender_mask, "gender_pred"]
).mean()

household_acc = (
    merged.loc[household_mask, "household_size_norm_true"] == 
    merged.loc[household_mask, "household_size_norm_pred"]
).mean()

income_acc = (
    merged.loc[income_mask, "income_level"] == 
    merged.loc[income_mask, "household_income_level"]
).mean()

# ============================================================================
# MAJORITY-CLASS BASELINES
# ============================================================================
def majority_baseline(y_true):
    """Calculate accuracy if always predicting the majority class"""
    if len(y_true) == 0:
        return 0.0
    majority = y_true.value_counts().idxmax()
    return (y_true == majority).mean()

age_baseline = majority_baseline(merged.loc[age_mask, "age_group_true"])
gender_baseline = majority_baseline(merged.loc[gender_mask, "gender_true"])
household_baseline = majority_baseline(merged.loc[household_mask, "household_size_norm_true"])
income_baseline = majority_baseline(merged.loc[income_mask, "income_level"])

# ============================================================================
# OUTPUT RESULTS
# ============================================================================
print("\n" + "="*70)
print("📊 ACCURACY RESULTS")
print("="*70)
print(f"Age group:        {age_acc:.3f}  (baseline: {age_baseline:.3f})  [n={age_mask.sum()}]")
print(f"Gender:           {gender_acc:.3f}  (baseline: {gender_baseline:.3f})  [n={gender_mask.sum()}]")
print(f"Household size:   {household_acc:.3f}  (baseline: {household_baseline:.3f})  [n={household_mask.sum()}]")
print(f"Income level:     {income_acc:.3f}  (baseline: {income_baseline:.3f})  [n={income_mask.sum()}]")

# ============================================================================
# SAVE MERGED COMPARISON TABLES
# ============================================================================
output_path = Path("/data/baliu/python_code/data/merged_comparison_v6_final_qwen_v5.csv")
merged.to_csv(output_path, index=False, encoding="utf-8")

filtered_income_path = Path("/data/baliu/python_code/data/comparison_income_eval_v6_final_qwen_v5.csv")
merged.loc[income_mask].to_csv(filtered_income_path, index=False, encoding="utf-8")

print(f"\n✓ Detailed comparison saved to: {output_path}")
print(f"✓ Income-eval subset saved to: {filtered_income_path}")

# ============================================================================
# CONFUSION MATRICES
# ============================================================================
print("\n" + "="*70)
print("CONFUSION MATRICES")
print("="*70)

# Age confusion matrix
if age_mask.sum() > 0:
    age_cm_counts = pd.crosstab(
        merged.loc[age_mask, "age_group_true"],
        merged.loc[age_mask, "age_group_pred"],
        rownames=['True'],
        colnames=['Predicted']
    )
    
    age_cm_norm = pd.crosstab(
        merged.loc[age_mask, "age_group_true"],
        merged.loc[age_mask, "age_group_pred"],
        normalize="index",
        rownames=['True'],
        colnames=['Predicted']
    )
    
    age_cm_path = Path("/data/baliu/python_code/data/age_confusion_matrix_v4_final.csv")
    age_cm_norm.to_csv(age_cm_path)
    
    
    print("\nAge confusion matrix (row-normalized):")
    print(age_cm_norm.round(3))
    print(f"✓ Saved to: {age_cm_path}")

# Gender confusion matrix
if gender_mask.sum() > 0:
    gender_cm_counts = pd.crosstab(
        merged.loc[gender_mask, "gender_true"],
        merged.loc[gender_mask, "gender_pred"],
        rownames=['True'],
        colnames=['Predicted']
    )
    
    gender_cm_norm = pd.crosstab(
        merged.loc[gender_mask, "gender_true"],
        merged.loc[gender_mask, "gender_pred"],
        normalize="index",
        rownames=['True'],
        colnames=['Predicted']
    )
    
    gender_cm_path = Path("/data/baliu/python_code/data/gender_confusion_matrix_v4_final.csv")
    gender_cm_norm.to_csv(gender_cm_path)
    
    print("\nGender confusion matrix (row-normalized):")
    print(gender_cm_norm.round(3))
    print(f"✓ Saved to: {gender_cm_path}")

# Household confusion matrix
if household_mask.sum() > 0:
    household_cm_counts = pd.crosstab(
        merged.loc[household_mask, "household_size_norm_true"],
        merged.loc[household_mask, "household_size_norm_pred"],
        rownames=['True'],
        colnames=['Predicted']
    )
    
    household_cm_norm = pd.crosstab(
        merged.loc[household_mask, "household_size_norm_true"],
        merged.loc[household_mask, "household_size_norm_pred"],
        normalize="index",
        rownames=['True'],
        colnames=['Predicted']
    )
    
    household_cm_path = Path("/data/baliu/python_code/data/household_confusion_matrix_v4_final.csv")
    household_cm_norm.to_csv(household_cm_path)
    
    print("\nHousehold size confusion matrix (row-normalized):")
    print(household_cm_norm.round(3))
    print(f"✓ Saved to: {household_cm_path}")

# Income confusion matrix
if income_mask.sum() > 0:
    income_cm_counts = pd.crosstab(
        merged.loc[income_mask, "income_level"],
        merged.loc[income_mask, "household_income_level"],
        rownames=['True'],
        colnames=['Predicted']
    )
    
    income_cm_norm = pd.crosstab(
        merged.loc[income_mask, "income_level"],
        merged.loc[income_mask, "household_income_level"],
        normalize="index",
        rownames=['True'],
        colnames=['Predicted']
    )
    
    income_cm_path = Path("/data/baliu/python_code/data/income_confusion_matrix_v4_final.csv")
    income_cm_norm.to_csv(income_cm_path)
    
    print("\nIncome confusion matrix (counts):")
    print(income_cm_counts)
    
    print("\nIncome confusion matrix (row-normalized):")
    print(income_cm_norm.round(3))
    print(f"✓ Saved to: {income_cm_path}")

# ============================================================================
# summary
# ============================================================================
print("\n" + "="*70)
print("✅ EVALUATION COMPLETED")
print("="*70)
print(f"Total users evaluated: {len(merged)}")
print(f"\nAccuracy Summary:")
print(f"  Age:       {age_acc:.1%}")
print(f"  Gender:    {gender_acc:.1%}")
print(f"  Household: {household_acc:.1%}")
print(f"  Income:    {income_acc:.1%}")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# ============================================================================
# FILE PATHS
# ============================================================================
gt_path = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
pred_path = Path("/data/baliu/python_code/data/preds_qwen_household_size_age_05Mar2026_v5_clean_full.csv")

# ============================================================================
# LOAD DATA
# ============================================================================
gt = pd.read_csv(gt_path)
pred = pd.read_csv(pred_path)

print("="*70)
print("HOUSEHOLD INCOME PREDICTION EVALUATION")
print("="*70)
print(f"Ground truth: {len(gt)} rows")
print(f"Predictions:  {len(pred)} rows")

# ============================================================================
# NORMALIZATION FUNCTIONS
# ============================================================================

def normalize_gender(text):
    """Normalize gender to lowercase"""
    if pd.isna(text):
        return None
    text = str(text).strip().lower()
    if text in ['female', 'f']:
        return 'female'
    if text in ['male', 'm']:
        return 'male'
    if text in ['other', 'non-binary']:
        return 'other'
    return None


def normalize_age_group(text):
    """
    Normalize age groups to standard 10-year bins.
    
    GT format: "18-24", "25-44", "45-66", "67+"
    Pred format: "18-25", "26-35", "36-45", "46-55", "56-65", "66+"
    
    Map to unified format: "18-25", "26-35", "36-45", "46-55", "56-65", "66+"
    """
    if pd.isna(text):
        return None
    
    text = str(text).strip().lower()
    
    # Extract numbers
    import re
    numbers = re.findall(r'\d+', text)
    
    if not numbers:
        return None
    
    # Get age range
    if len(numbers) >= 2:
        low = int(numbers[0])
        high = int(numbers[1])
    elif len(numbers) == 1:
        low = int(numbers[0])
        high = low
    else:
        return None
    
    # Map to standard bins
    if low <= 25:
        return "18-25"
    elif low <= 35:
        return "26-35"
    elif low <= 45:
        return "36-45"
    elif low <= 55:
        return "46-55"
    elif low <= 65:
        return "56-65"
    else:
        return "66+"


def normalize_household_size(x):
    """Convert household size to standardized format: '1', '2', '3', '4', '5+'"""
    if pd.isna(x):
        return None
    
    x_str = str(x).strip().lower()
    
    # Handle '5+' or 'more than 5'
    if '5+' in x_str or 'more than 5' in x_str or '>5' in x_str or '>=5' in x_str:
        return "5+"
    
    # Try to convert to number
    try:
        x_num = float(x_str)
        if x_num >= 5:
            return "5+"
        elif x_num >= 1 and x_num <= 4:
            return str(int(x_num))
        else:
            return None
    except ValueError:
        # Try to extract number
        import re
        match = re.search(r'\d+', x_str)
        if match:
            num = int(match.group())
            if num >= 5:
                return "5+"
            elif num >= 1 and num <= 4:
                return str(num)
        return None


def normalize_income(text):
    """Normalize income levels to standard format"""
    if pd.isna(text):
        return None
    
    text = str(text).strip().lower()
    
    # Direct matches
    if '<4000' in text or '< 4000' in text or 'less than 4000' in text:
        return "<4000"
    if '4001-8000' in text or '4000-8000' in text:
        return "4001-8000"
    if '8001-12000' in text or '8000-12000' in text:
        return "8001-12000"
    if '12001-16000' in text or '12000-16000' in text:
        return "12001-16000"
    if '>16000' in text or '> 16000' in text or 'more than 16000' in text or '16001' in text:
        return ">16000"
    
    return None


# ============================================================================
# APPLY NORMALIZATION
# ============================================================================
print("\n" + "="*70)
print("NORMALIZING DATA")
print("="*70)

# Ground truth
gt_clean = gt[["user_id", "age_group", "household_size_group", "income_level", "gender"]].copy()
gt_clean["age_norm"] = gt_clean["age_group"].apply(normalize_age_group)
gt_clean["gender_norm"] = gt_clean["gender"].apply(normalize_gender)
gt_clean["household_norm"] = gt_clean["household_size_group"].apply(normalize_household_size)
gt_clean["income_norm"] = gt_clean["income_level"].apply(normalize_income)

# Predictions
pred_clean = pred[["user_id", "age_group", "gender", "household_income_level", "household_size"]].copy()
pred_clean["age_norm"] = pred_clean["age_group"].apply(normalize_age_group)
pred_clean["gender_norm"] = pred_clean["gender"].apply(normalize_gender)
pred_clean["household_norm"] = pred_clean["household_size"].apply(normalize_household_size)
pred_clean["income_norm"] = pred_clean["household_income_level"].apply(normalize_income)

print("✓ Normalization complete")

# Show distributions
print("\n📊 Ground Truth Distributions:")
print(f"Age:       {gt_clean['age_norm'].value_counts().to_dict()}")
print(f"Gender:    {gt_clean['gender_norm'].value_counts().to_dict()}")
print(f"Household: {gt_clean['household_norm'].value_counts().to_dict()}")
print(f"Income:    {gt_clean['income_norm'].value_counts().to_dict()}")

print("\n📊 Prediction Distributions:")
print(f"Age:       {pred_clean['age_norm'].value_counts().to_dict()}")
print(f"Gender:    {pred_clean['gender_norm'].value_counts().to_dict()}")
print(f"Household: {pred_clean['household_norm'].value_counts().to_dict()}")
print(f"Income:    {pred_clean['income_norm'].value_counts().to_dict()}")

# ============================================================================
# MERGE
# ============================================================================
print("\n" + "="*70)
print("MERGING DATA")
print("="*70)

merged = pd.merge(
    gt_clean[["user_id", "age_norm", "gender_norm", "household_norm", "income_norm"]],
    pred_clean[["user_id", "age_norm", "gender_norm", "household_norm", "income_norm"]],
    on="user_id",
    suffixes=("_true", "_pred"),
    how="inner"
)

print(f"✓ Merged {len(merged)} users")

# Sample check
print("\n🔍 Sample of merged data (first 5 rows):")
sample_cols = ["user_id", "age_norm_true", "age_norm_pred", "gender_norm_true", "gender_norm_pred"]
print(merged[sample_cols].head())

# ============================================================================
# EVALUATION MASKS
# ============================================================================
print("\n" + "="*70)
print("CREATING EVALUATION MASKS")
print("="*70)

age_mask = (
    merged["age_norm_true"].notna() &
    merged["age_norm_pred"].notna()
)

gender_mask = (
    merged["gender_norm_true"].notna() &
    merged["gender_norm_pred"].notna()
)

household_mask = (
    merged["household_norm_true"].notna() &
    merged["household_norm_pred"].notna()
)

income_mask = (
    merged["income_norm_true"].notna() &
    merged["income_norm_pred"].notna()
)

print(f"Age samples:       {age_mask.sum():5d} / {len(merged)} ({age_mask.sum()/len(merged)*100:.1f}%)")
print(f"Gender samples:    {gender_mask.sum():5d} / {len(merged)} ({gender_mask.sum()/len(merged)*100:.1f}%)")
print(f"Household samples: {household_mask.sum():5d} / {len(merged)} ({household_mask.sum()/len(merged)*100:.1f}%)")
print(f"Income samples:    {income_mask.sum():5d} / {len(merged)} ({income_mask.sum()/len(merged)*100:.1f}%)")

# ============================================================================
# COMPUTE ACCURACY
# ============================================================================
print("\n" + "="*70)
print("COMPUTING ACCURACY")
print("="*70)

def compute_accuracy(y_true, y_pred, mask):
    if mask.sum() == 0:
        return 0.0
    return (y_true[mask] == y_pred[mask]).mean()

age_acc = compute_accuracy(merged["age_norm_true"], merged["age_norm_pred"], age_mask)
gender_acc = compute_accuracy(merged["gender_norm_true"], merged["gender_norm_pred"], gender_mask)
household_acc = compute_accuracy(merged["household_norm_true"], merged["household_norm_pred"], household_mask)
income_acc = compute_accuracy(merged["income_norm_true"], merged["income_norm_pred"], income_mask)

print(f"Age accuracy:       {age_acc:.3f} ({age_acc*100:.1f}%) [n={age_mask.sum()}]")
print(f"Gender accuracy:    {gender_acc:.3f} ({gender_acc*100:.1f}%) [n={gender_mask.sum()}]")
print(f"Household accuracy: {household_acc:.3f} ({household_acc*100:.1f}%) [n={household_mask.sum()}]")
print(f"Income accuracy:    {income_acc:.3f} ({income_acc*100:.1f}%) [n={income_mask.sum()}]")

# ============================================================================
# CONFUSION MATRIX PLOTTING FUNCTION
# ============================================================================

def plot_confusion_matrix(y_true, y_pred, mask, title, save_path):
    """
    Plot confusion matrix heatmap similar to XGBoost visualization.
    """
    if mask.sum() == 0:
        print(f"⚠️  Skipping {title}: No valid samples")
        return None, None
    
    # Create confusion matrices
    cm_counts = pd.crosstab(
        y_true[mask],
        y_pred[mask],
        rownames=['True'],
        colnames=['Predicted']
    )
    
    cm_norm = pd.crosstab(
        y_true[mask],
        y_pred[mask],
        rownames=['True'],
        colnames=['Predicted'],
        normalize='index'
    )
    
    # Custom colormap (purple to yellow like your image)
    colors = ['#2d1b3d', '#3a4f8f', '#1f968b', '#73d055', '#fde724']
    cmap = LinearSegmentedColormap.from_list('custom', colors, N=100)
    
    # Create figure
    plt.figure(figsize=(10, 8))
    
    # Plot heatmap
    sns.heatmap(
        cm_counts,
        annot=True,
        fmt='d',
        cmap=cmap,
        cbar=True,
        square=True,
        linewidths=1,
        linecolor='white',
        annot_kws={'size': 11, 'weight': 'bold'}
    )
    
    # Add title
    n_test = mask.sum()
    acc = (y_true[mask] == y_pred[mask]).mean()
    plt.title(f'{title} (test n={n_test}, accuracy={acc:.3f})', 
              fontsize=14, fontweight='bold', pad=20)
    
    # Labels
    plt.xlabel('Predicted', fontsize=12, fontweight='bold')
    plt.ylabel('True', fontsize=12, fontweight='bold')
    
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    
    plt.tight_layout()
    
    # Save
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: {save_path}")
    plt.close()
    
    return cm_counts, cm_norm


# ============================================================================
# GENERATE CONFUSION MATRICES
# ============================================================================
print("\n" + "="*70)
print("GENERATING CONFUSION MATRICES")
print("="*70)

# Age
print("\n1. Age Group:")
age_cm_counts, age_cm_norm = plot_confusion_matrix(
    merged["age_norm_true"],
    merged["age_norm_pred"],
    age_mask,
    "Age Group Confusion Matrix",
    Path("/data/baliu/python_code/data/cm_age_group.png")
)
if age_cm_norm is not None:
    print("   Row-normalized confusion matrix:")
    print(age_cm_norm.round(3))
    age_cm_norm.to_csv("/data/baliu/python_code/data/cm_age_group.csv")

# Gender
print("\n2. Gender:")
gender_cm_counts, gender_cm_norm = plot_confusion_matrix(
    merged["gender_norm_true"],
    merged["gender_norm_pred"],
    gender_mask,
    "Gender Confusion Matrix",
    Path("/data/baliu/python_code/data/cm_gender.png")
)
if gender_cm_norm is not None:
    print("   Row-normalized confusion matrix:")
    print(gender_cm_norm.round(3))
    gender_cm_norm.to_csv("/data/baliu/python_code/data/cm_gender.csv")

# Household Size
print("\n3. Household Size:")
household_cm_counts, household_cm_norm = plot_confusion_matrix(
    merged["household_norm_true"],
    merged["household_norm_pred"],
    household_mask,
    "Household Size Confusion Matrix",
    Path("/data/baliu/python_code/data/cm_household_size.png")
)
if household_cm_norm is not None:
    print("   Row-normalized confusion matrix:")
    print(household_cm_norm.round(3))
    household_cm_norm.to_csv("/data/baliu/python_code/data/cm_household_size.csv")

# Income Level
print("\n4. Income Level:")
income_cm_counts, income_cm_norm = plot_confusion_matrix(
    merged["income_norm_true"],
    merged["income_norm_pred"],
    income_mask,
    "Income Level Confusion Matrix",
    Path("/data/baliu/python_code/data/cm_income_level.png")
)
if income_cm_norm is not None:
    print("   Row-normalized confusion matrix:")
    print(income_cm_norm.round(3))
    income_cm_norm.to_csv("/data/baliu/python_code/data/cm_income_level.csv")

# ============================================================================
# SAVE MERGED DATA
# ============================================================================
print("\n" + "="*70)
print("SAVING RESULTS")
print("="*70)

merged.to_csv("/data/baliu/python_code/data/merged_evaluation_results.csv", index=False)
print("✓ Merged results saved")

# Save filtered subsets
if income_mask.sum() > 0:
    merged[income_mask].to_csv("/data/baliu/python_code/data/income_evaluation_subset.csv", index=False)
    print("✓ Income subset saved")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*70)
print("📊 FINAL EVALUATION SUMMARY")
print("="*70)
print(f"\nTotal users: {len(merged)}")
print(f"\n{'Metric':<20} {'Accuracy':<12} {'Samples':<10} {'Coverage'}")
print("-" * 70)
print(f"{'Age Group':<20} {age_acc:>6.1%}        {age_mask.sum():>6d}     {age_mask.sum()/len(merged):>6.1%}")
print(f"{'Gender':<20} {gender_acc:>6.1%}        {gender_mask.sum():>6d}     {gender_mask.sum()/len(merged):>6.1%}")
print(f"{'Household Size':<20} {household_acc:>6.1%}        {household_mask.sum():>6d}     {household_mask.sum()/len(merged):>6.1%}")
print(f"{'Income Level':<20} {income_acc:>6.1%}        {income_mask.sum():>6d}     {income_mask.sum()/len(merged):>6.1%}")

print("\n" + "="*70)
print("✅ EVALUATION COMPLETED")
print("="*70)
print("\nGenerated files:")
print("  📊 cm_age_group.png & .csv")
print("  📊 cm_gender.png & .csv")
print("  📊 cm_household_size.png & .csv")
print("  📊 cm_income_level.png & .csv")
print("  📄 merged_evaluation_results.csv")
print("  📄 income_evaluation_subset.csv")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------
# File paths
# --------------------------------------
gt_path   = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
pred_path = Path("/data/baliu/python_code/data/preds_qwen_income_22Feb2026_v5withrationale_clean.csv")

# --------------------------------------
# Load data
# --------------------------------------
gt = pd.read_csv(gt_path)
pred = pd.read_csv(pred_path)

print(f"Loaded ground truth ({len(gt)} rows)")
print(f"Loaded predictions  ({len(pred)} rows)")

# --------------------------------------
# Select & align columns
# --------------------------------------
gt = gt[["user_id", "age_group", "household_size_group", "income_level"]].copy()
pred = pred[["user_id", "age_group", "gender", "household_income_level"]].copy()

# --------------------------------------
# Merge GT and predictions
# --------------------------------------
merged = pd.merge(gt, pred, on="user_id", how="inner", suffixes=("_true", "_pred"))
print(f"Merged {len(merged)} users")
print(merged.head(5))

# --------------------------------------
# Masks for valid evaluation samples
# --------------------------------------
age_mask = merged["age_group_true"].notna() & merged["age_group_pred"].notna()

# If you don't evaluate household size right now, set hh_mask to all-False safely:
hh_mask = pd.Series(False, index=merged.index)

# Income: evaluate where both exist, and GT is not "prefer not to say"
income_mask = (
    merged["income_level_true"].notna()
    & merged["household_income_level"].notna()
    & (merged["income_level_true"].astype(str).str.lower() != "prefer not to say")
)

print("\nEvaluation sample sizes:")
print(f"Age:        {age_mask.sum()}")
print(f"Household:  {hh_mask.sum()}")
print(f"Income:     {income_mask.sum()}")

# --------------------------------------
# Accuracy computation
# --------------------------------------
age_acc = (
    merged.loc[age_mask, "age_group_true"]
    == merged.loc[age_mask, "age_group_pred"]
).mean() if age_mask.any() else np.nan

income_acc = (
    merged.loc[income_mask, "income_level_true"]
    == merged.loc[income_mask, "household_income_level"]
).mean() if income_mask.any() else np.nan

# --------------------------------------
# Majority-class baselines (TRUE labels only)
# --------------------------------------
def majority_baseline(y_true: pd.Series) -> float:
    y_true = y_true.dropna()
    if len(y_true) == 0:
        return np.nan
    majority = y_true.value_counts().idxmax()
    return (y_true == majority).mean()

age_baseline = majority_baseline(merged.loc[age_mask, "age_group_true"])
income_baseline = majority_baseline(merged.loc[income_mask, "income_level_true"])

# --------------------------------------
# Output results
# --------------------------------------
print("\n📊 Accuracy Results:")
print(f"Age group accuracy:     {age_acc:.3f} (baseline {age_baseline:.3f})")
print(f"Income level accuracy:  {income_acc:.3f} (baseline {income_baseline:.3f})")

# --------------------------------------
# Save merged comparison tables
# --------------------------------------
output_path = Path("/data/baliu/python_code/data/merged_comparison_v6_final_qwen_v5.csv")
merged.to_csv(output_path, index=False, encoding="utf-8")

filtered_income_path = Path("/data/baliu/python_code/data/comparison_income_eval_v6_final_qwen_v5.csv")
merged.loc[income_mask].to_csv(filtered_income_path, index=False, encoding="utf-8")

print(f"\nDetailed comparison saved to: {output_path}")
print(f"Income-eval subset saved to: {filtered_income_path}")

# --------------------------------------
# Confusion matrices
# --------------------------------------
# Age confusion matrix (row-normalized)
if age_mask.any():
    age_cm = pd.crosstab(
        merged.loc[age_mask, "age_group_true"],
        merged.loc[age_mask, "age_group_pred"],
        normalize="index"
    )
    age_cm_path = Path("/data/baliu/python_code/data/age_confusion_matrix_v5_final.csv")
    age_cm.to_csv(age_cm_path)
    print(f"Age confusion matrix saved to: {age_cm_path}")

# Income confusion matrix (counts + row-normalized)
if income_mask.any():
    income_cm_counts = pd.crosstab(
        merged.loc[income_mask, "income_level_true"],
        merged.loc[income_mask, "household_income_level"]
    )
    print("\nIncome confusion matrix (counts):")
    print(income_cm_counts)

    income_cm_norm = pd.crosstab(
        merged.loc[income_mask, "income_level_true"],
        merged.loc[income_mask, "household_income_level"],
        normalize="index"
    )
    print("\nIncome confusion matrix (row-normalized):")
    print(income_cm_norm)

In [ ]:
"""                                                                                                            
Fine-tuning Qwen 2.5 on mobility prompt + ground truth data.

Inputs:  
  - PROMPT_TXT : one block per user, separated by "===USER_{id}===" headers
  - GT_CSV     : ground truth with columns [user_id, age, household_size, income_level, ...]
  - Participant_ID in GT_CSV should match user_id in PROMPT_TXT headers for merging

Pipeline: 
processed as instruction-response pairs for chat-style fine tuning:
  1. Parse prompts  → dict {user_id -> prompt_text}
  2. Load GT CSV    → df with user_id + labels
  3. Merge          → build instruction-response pairs
  4. 80 / 20 split
  5. QLoRA fine-tune with trl SFTTrainer
  6. Save adapter + tokenizer
  
"""

# ============================================================
# 0. Imports
# ============================================================
import re
import json
import random
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# ============================================================
# 1. Paths & config
# ============================================================
PROMPT_TXT = Path("/data/baliu/python_code/data/"
                  "prompts_v3_1week_age_householdsize_04Mar2026_qwen_clean_v4.txt")
GT_CSV     = Path("/data/baliu/python_code/data/"
                  "gt_with_age_household_income_level.csv")

MODEL_NAME  = "Qwen/Qwen2.5-7B-Instruct"   # change to 14B / 72B if VRAM allows
OUTPUT_DIR  = Path("/data/baliu/python_code/models/qwen25_mobility_lora")
LOG_DIR     = Path("/data/baliu/python_code/logs/qwen25_mobility")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

SEED        = 42
TEST_SIZE   = 0.2
MAX_SEQ_LEN = 2048      # truncate if prompt+response exceeds this

# ============================================================
# 2. Parse prompt text file
#    Expected format — each user block starts with a header:
#        ===USER_<user_id>=== participart_ID
#        <mobility token lines>
#    Adjust the regex below if your separator differs.
# ============================================================
def parse_prompt_file(path: Path) -> dict[str, str]:
    """Return {user_id (str) -> prompt_text (str)}."""
    text = path.read_text(encoding="utf-8")

    # split on header lines like  ===USER_12345===  or  USER_12345:
    blocks = re.split(r"={2,}USER_(\w+)={2,}|^USER_(\w+)\s*:", text,
                      flags=re.MULTILINE)

    prompts = {}
    # re.split with capture groups interleaves: [before, g1, g2, content, g1, g2, content, ...]
    i = 1
    while i < len(blocks) - 2:
        uid = blocks[i] or blocks[i + 1]   # one of the two capture groups matched
        content = blocks[i + 2].strip()
        if uid and content:
            prompts[str(uid)] = content
        i += 3

    if not prompts:
        # Fallback: assume one prompt per line with "user_id\t<prompt>" format
        print("[WARN] Header pattern not found — trying tab-separated fallback.")
        for line in text.splitlines():
            if "\t" in line:
                uid, _, prompt = line.partition("\t")
                prompts[uid.strip()] = prompt.strip()

    print(f"[INFO] Parsed {len(prompts)} user prompts.")
    return prompts

# ============================================================
# 3. Load ground truth
# ============================================================
def load_gt(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={"participant_ID": str})
    # normalise column name variants
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    if "participant_ID" not in df.columns:
        raise ValueError(f"Expected 'participant_ID' column in GT CSV. Found: {list(df.columns)}")
    print(f"[INFO] Ground truth loaded: {len(df)} rows, columns: {list(df.columns)}")
    return df

# ============================================================
# 4. Build instruction–response pairs
# ============================================================
SYSTEM_MSG = (
    "You are a mobility behaviour analyst. "
    "Given a one-week sequence of staypoint events for a user, "
    "predict their demographic profile."
)

def build_label_str(row: pd.Series, label_cols: list[str]) -> str:
    """Serialise GT columns into a compact answer string."""
    parts = []
    for col in label_cols:
        val = row.get(col, None)
        if pd.notna(val):
            parts.append(f"{col.replace('_', ' ').title()}: {val}")
    return "\n".join(parts) if parts else "No label available."

def build_chat_record(prompt: str, label: str) -> dict:
    """Format as Qwen2.5 chat template messages."""
    messages = [
        {"role": "system",    "content": SYSTEM_MSG},
        {"role": "user",      "content": prompt},
        {"role": "assistant", "content": label},
    ]
    return {"messages": messages}

def build_dataset(prompts: dict, gt: pd.DataFrame) -> list[dict]:
    # identify label columns (everything except user_id)
    label_cols = [c for c in gt.columns if c != "user_id"]
    print(f"[INFO] Label columns: {label_cols}")

    records = []
    missing = 0
    for uid, prompt_text in prompts.items():
        row_match = gt[gt["user_id"] == uid]
        if row_match.empty:
            missing += 1
            continue
        row = row_match.iloc[0]
        label = build_label_str(row, label_cols)
        records.append(build_chat_record(prompt_text, label))

    print(f"[INFO] Matched {len(records)} users | {missing} prompts had no GT match.")
    return records

# ============================================================
# 5. Tokenise with chat template
# ============================================================
def apply_chat_template(examples, tokenizer):
    texts = [
        tokenizer.apply_chat_template(
            msg, tokenize=False, add_generation_prompt=False
        )
        for msg in examples["messages"]
    ]
    return {"text": texts}

# ============================================================
# 6. Main fine-tune
# ============================================================
def main():
    random.seed(SEED)

    # --- load data ---
    prompts = parse_prompt_file(PROMPT_TXT)
    gt      = load_gt(GT_CSV)
    records = build_dataset(prompts, gt)

    if len(records) == 0:
        raise RuntimeError("No matched records — check user_id format in prompt file vs CSV.")

    # --- 80 / 20 split ---
    train_records, test_records = train_test_split(
        records, test_size=TEST_SIZE, random_state=SEED
    )
    print(f"[INFO] Train: {len(train_records)} | Test: {len(test_records)}")

    # save test set for later evaluation
    test_path = OUTPUT_DIR / "test_records.jsonl"
    with open(test_path, "w") as f:
        for rec in test_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"[INFO] Test records saved to {test_path}")

    # --- tokenizer ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # --- datasets (keep conversational `messages` format for assistant-only masking) ---
    train_ds = Dataset.from_list(train_records)
    test_ds  = Dataset.from_list(test_records)

    # --- 4-bit quantisation (QLoRA) ---
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.use_cache = False

    # --- LoRA config ---
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # --- SFT config ---
    sft_config = SFTConfig(
        output_dir=str(OUTPUT_DIR),
        logging_dir=str(LOG_DIR),
        num_train_epochs=3,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,       # effective batch = 16
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        fp16=False,
        bf16=True,                           # A100 / H100 on Euler
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",                    # set to "wandb" if you want W&B logging
        dataloader_num_workers=4,
        seed=SEED,
        max_length=MAX_SEQ_LEN,
        assistant_only_loss=True,
    )

    # --- SFTTrainer ---
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        args=sft_config,
    )

    # --- train ---
    print("[INFO] Starting training ...")
    trainer.train()

    # --- save final adapter + tokenizer ---
    trainer.save_model(str(OUTPUT_DIR / "final"))
    tokenizer.save_pretrained(str(OUTPUT_DIR / "final"))
    print(f"[INFO] Model saved to {OUTPUT_DIR / 'final'}")

    # --- quick eval summary ---
    metrics = trainer.evaluate()
    print("[INFO] Final eval metrics:", metrics)
    pd.Series(metrics).to_csv(OUTPUT_DIR / "eval_metrics.csv")


if __name__ == "__main__":
    main()
